# Sistema de alerta temprana para la detección de anomalías en procesos industriales multivariables

**Autora:** Angie Stephany Pineda García  
**Dataset principal:** Tennessee Eastman Process  
**TFM:** Máster Universitario en Análisis de Datos Masivos — Universidad Europea de Madrid

> **Versión para reproducibilidad en GitHub.** El código de las celdas se conserva en el mismo orden que en el cuaderno original de Google Colab. Para reducir el tamaño del repositorio se eliminaron únicamente las salidas almacenadas y los contadores de ejecución; no se modificaron las instrucciones de cálculo ni los resultados reportados en la memoria.

## Alcance del cuaderno

El cuaderno documenta el pipeline experimental completo: carga y validación de datos, partición por corridas, estandarización, PCA, Isolation Forest, Autoencoder, calibración de umbrales, persistencia temporal 3-de-5, evaluación independiente en testing, Precision–Recall, interpretabilidad, análisis de sensibilidad y prototipo final basado en PCA.

**Nota de ejecución:** varias celdas utilizan rutas de Google Drive y checkpoints generados durante el desarrollo. Para una reproducción desde cero es necesario descargar el dataset Tennessee Eastman Process multi-run y adaptar `PROJECT_DIR` a la ubicación local/Drive correspondiente.


## 1. Configuración del entorno y carga de datos

Instalación de dependencias, montaje de Google Drive, definición de rutas y carga inicial de los conjuntos del Tennessee Eastman Process.


In [ ]:
%pip install -q pyreadr
from pathlib import Path
import platform

import numpy as np
import pandas as pd
import pyreadr

SEED = 42
np.random.seed(SEED)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Pyreadr:", pyreadr.__version__)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

MY_DRIVE = Path("/content/drive/MyDrive")

print("¿Drive está montado?:", Path("/content/drive").exists())
print("¿MyDrive está disponible?:", MY_DRIVE.exists())

In [ ]:
PROJECT_DIR = MY_DRIVE / "TFM_Tennessee_Eastman"

print("¿Existe la carpeta del proyecto?:", PROJECT_DIR.exists())
print("Contenido:\n")

for elemento in PROJECT_DIR.iterdir():
    tipo = "carpeta" if elemento.is_dir() else "archivo"
    print(f"{repr(elemento.name)} | {tipo}")

In [ ]:
DATA_DIR = PROJECT_DIR / "Datos"

print("Ruta utilizada:", DATA_DIR)
print("¿Existe la carpeta Datos?:", DATA_DIR.exists())

In [ ]:
archivos_rdata = sorted(DATA_DIR.glob("*.RData"))

print(f"Archivos .RData encontrados: {len(archivos_rdata)}\n")

for archivo in archivos_rdata:
    tamaño_mb = archivo.stat().st_size / (1024 ** 2)
    print(f"{archivo.name} | {tamaño_mb:.2f} MB")

In [ ]:
from pathlib import Path
from shutil import copy2

archivo_drive = DATA_DIR / "TEP_FaultFree_Training.RData"
archivo_local = Path("/content/TEP_FaultFree_Training.RData")

print("Archivo en Drive:", archivo_drive)
print("¿Existe en Drive?:", archivo_drive.exists())

In [ ]:
if not archivo_local.exists():
    copy2(archivo_drive, archivo_local)
    print("Archivo copiado a /content")
else:
    print("El archivo ya estaba copiado en /content")

print("¿Existe la copia local?:", archivo_local.exists())
print(
    "Tamaño local:",
    round(archivo_local.stat().st_size / (1024 ** 2), 2),
    "MB"
)

In [ ]:
import pyreadr

resultado = pyreadr.read_r(str(archivo_local))

print("Objetos contenidos en el archivo:")
print(list(resultado.keys()))

In [ ]:
nombre_objeto = next(iter(resultado.keys()))
ff_train = resultado[nombre_objeto]

print("Objeto seleccionado:", nombre_objeto)
print("Tipo de objeto:", type(ff_train))
print("Dimensiones:", ff_train.shape)

In [ ]:
display(ff_train.head())

In [ ]:
print(list(resultado.keys()))
print(ff_train.shape)
print(ff_train.columns.tolist())

## 2. Validación estructural y análisis exploratorio

Revisión de identificadores, variables de proceso, completitud de corridas, nulos, estructura temporal y primeras visualizaciones.


In [ ]:
#Clasificación de las columnas

# Columnas de identificación y organización temporal
metadata_cols = ["faultNumber", "simulationRun", "sample"]

# Variables medidas del proceso
xmeas_cols = [
    col for col in ff_train.columns
    if col.startswith("xmeas_")
]

# Variables manipuladas o de control
xmv_cols = [
    col for col in ff_train.columns
    if col.startswith("xmv_")
]

# Todas las variables que utilizarán los modelos
feature_cols = xmeas_cols + xmv_cols

print("Metadatos:", metadata_cols)
print("Número de variables medidas XMEAS:", len(xmeas_cols))
print("Número de variables manipuladas XMV:", len(xmv_cols))
print("Número total de variables de proceso:", len(feature_cols))
print("Número total de columnas:", len(ff_train.columns))

In [ ]:
print(
    "Número de corridas:",
    ff_train["simulationRun"].nunique()
)

print(
    "Número de muestras diferentes:",
    ff_train["sample"].nunique()
)

print(
    "Valores de faultNumber:",
    ff_train["faultNumber"].unique()
)

In [ ]:
muestras_por_corrida = (
    ff_train
    .groupby("simulationRun")
    .size()
)

print("Mínimo de muestras por corrida:", muestras_por_corrida.min())
print("Máximo de muestras por corrida:", muestras_por_corrida.max())
print("Número de corridas analizadas:", len(muestras_por_corrida))

In [ ]:
print("Valores nulos totales:", ff_train.isna().sum().sum())
print("Filas duplicadas exactas:", ff_train.duplicated().sum())

print("\nTipos de datos:")
print(ff_train.dtypes.value_counts())

In [ ]:
resumen_ff_train = pd.DataFrame({
    "Característica": [
        "Filas",
        "Columnas totales",
        "Corridas",
        "Muestras por corrida",
        "Variables medidas",
        "Variables manipuladas",
        "Variables de proceso",
        "Valores nulos",
        "Número de fallas"
    ],
    "Valor": [
        ff_train.shape[0],
        ff_train.shape[1],
        ff_train["simulationRun"].nunique(),
        muestras_por_corrida.iloc[0],
        len(xmeas_cols),
        len(xmv_cols),
        len(feature_cols),
        ff_train.isna().sum().sum(),
        ff_train["faultNumber"].nunique()
    ]
})

display(resumen_ff_train)

In [ ]:
print("Columnas de tipo entero:")
print(ff_train.dtypes[ff_train.dtypes == "int32"])

In [ ]:
ff_train_clean = ff_train.copy()

metadata_cols = [
    "faultNumber",
    "simulationRun",
    "sample"
]

for columna in metadata_cols:
    ff_train_clean[columna] = (
        ff_train_clean[columna]
        .astype("int32")
    )

print(ff_train_clean[metadata_cols].dtypes)

In [ ]:
ff_train_clean = (
    ff_train_clean
    .sort_values(
        by=["simulationRun", "sample"]
    )
    .reset_index(drop=True)
)

display(
    ff_train_clean[
        ["faultNumber", "simulationRun", "sample"]
    ].head(10)
)

In [ ]:
#Comprobar que todas las corridas estén completas
validacion_corridas = (
    ff_train_clean
    .groupby("simulationRun")["sample"]
    .agg(
        muestra_inicial="min",
        muestra_final="max",
        numero_muestras="nunique"
    )
)

validacion_corridas["secuencia_completa"] = (
    validacion_corridas["numero_muestras"]
    ==
    validacion_corridas["muestra_final"]
    - validacion_corridas["muestra_inicial"]
    + 1
)

display(validacion_corridas.head())

print(
    "Corridas con secuencia completa:",
    validacion_corridas["secuencia_completa"].sum(),
    "de",
    len(validacion_corridas)
)

print(
    "Muestras iniciales:",
    validacion_corridas["muestra_inicial"].unique()
)

print(
    "Muestras finales:",
    validacion_corridas["muestra_final"].unique()
)

In [ ]:
corrida_1 = (
    ff_train_clean[
        ff_train_clean["simulationRun"] == 1
    ]
    .copy()
)

print("Dimensiones de la corrida 1:", corrida_1.shape)
print("Primera muestra:", corrida_1["sample"].min())
print("Última muestra:", corrida_1["sample"].max())
print(
    "¿Está ordenada temporalmente?:",
    corrida_1["sample"].is_monotonic_increasing
)

display(corrida_1.head())

In [ ]:
resumen_ff_train = pd.DataFrame({
    "Característica": [
        "Filas",
        "Columnas totales",
        "Corridas",
        "Muestras por corrida",
        "Variables medidas",
        "Variables manipuladas",
        "Variables de proceso",
        "Valores nulos",
        "Clases presentes en faultNumber"
    ],
    "Valor": [
        ff_train_clean.shape[0],
        ff_train_clean.shape[1],
        ff_train_clean["simulationRun"].nunique(),
        validacion_corridas["numero_muestras"].iloc[0],
        len(xmeas_cols),
        len(xmv_cols),
        len(feature_cols),
        ff_train_clean.isna().sum().sum(),
        ff_train_clean["faultNumber"].nunique()
    ]
})

display(resumen_ff_train)

In [ ]:
#Se agrega columna para representar el espacio temporal
corrida_1_plot = corrida_1.copy()

corrida_1_plot["tiempo_h"] = (
    corrida_1_plot["sample"] - 1
) * 3 / 60

display(
    corrida_1_plot[
        ["sample", "tiempo_h", "xmeas_7", "xmeas_8", "xmeas_9"]
    ].head()
)

In [ ]:
variables_reactor = ["xmeas_7", "xmeas_8", "xmeas_9"]

estadisticas_corrida_1 = (
    corrida_1_plot[variables_reactor]
    .describe()
    .T
    [["mean", "std", "min", "max"]]
)

estadisticas_corrida_1.columns = [
    "media",
    "desviacion_estandar",
    "minimo",
    "maximo"
]

display(estadisticas_corrida_1)

In [ ]:
#Grafica temperatura en el reactor
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))

plt.plot(
    corrida_1_plot["tiempo_h"],
    corrida_1_plot["xmeas_9"]
)

plt.xlabel("Tiempo de simulación (h)")
plt.ylabel("Temperatura del reactor (°C)")
plt.title(
    "Temperatura del reactor durante una corrida normal"
)

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
variables_reactor = ["xmeas_7", "xmeas_8", "xmeas_9"]

resumen_global_reactor = pd.DataFrame({
    "media_global": ff_train_clean[variables_reactor].mean(),
    "desviacion_global": ff_train_clean[variables_reactor].std(),
    "minimo_global": ff_train_clean[variables_reactor].min(),
    "percentil_1": ff_train_clean[variables_reactor].quantile(0.01),
    "mediana": ff_train_clean[variables_reactor].quantile(0.50),
    "percentil_99": ff_train_clean[variables_reactor].quantile(0.99),
    "maximo_global": ff_train_clean[variables_reactor].max()
})

display(resumen_global_reactor)

In [ ]:
#Comparación de temperatura de reacción entre corridas
estadisticas_temperatura_por_corrida = (
    ff_train_clean
    .groupby("simulationRun")["xmeas_9"]
    .agg(
        media="mean",
        desviacion_estandar="std",
        minimo="min",
        maximo="max"
    )
    .reset_index()
)

display(estadisticas_temperatura_por_corrida.head())

## 3. Partición por corrida y preprocesamiento

Separación de corridas normales para entrenamiento/validación, selección de las 52 variables de proceso, comprobaciones de varianza e infinitos y estandarización sin fuga de información.


In [ ]:
#Obtener identificadores de las corridas
corridas_disponibles = (
    ff_train_clean["simulationRun"]
    .drop_duplicates()
    .sort_values()
    .to_numpy()
)

print("Número total de corridas:", len(corridas_disponibles))
print("Primeras corridas:", corridas_disponibles[:10])
print("Últimas corridas:", corridas_disponibles[-10:])

In [ ]:
from sklearn.model_selection import train_test_split

corridas_train, corridas_val = train_test_split(
    corridas_disponibles,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print("Corridas de entrenamiento:", len(corridas_train))
print("Corridas de validación:", len(corridas_val))

In [ ]:
#Separara test y entrenamiento

df_train_normal = (
    ff_train_clean[
        ff_train_clean["simulationRun"].isin(corridas_train)
    ]
    .copy()
)

df_val_normal = (
    ff_train_clean[
        ff_train_clean["simulationRun"].isin(corridas_val)
    ]
    .copy()
)

print("Dimensiones del entrenamiento:", df_train_normal.shape)
print("Dimensiones de la validación:", df_val_normal.shape)

In [ ]:
#Verificar que validación y test no tienen corrdas compartidas
corridas_compartidas = (
    set(df_train_normal["simulationRun"])
    .intersection(set(df_val_normal["simulationRun"]))
)

print("Corridas compartidas:", corridas_compartidas)

In [ ]:
print("Corrida 1 en entrenamiento:", 1 in corridas_train)
print("Corrida 1 en validación:", 1 in corridas_val)

In [ ]:
#Confirmar estructura de entrenamienot y validación
print("Entrenamiento:", df_train_normal.shape)
print("Validación:", df_val_normal.shape)

print(
    "Corridas en entrenamiento:",
    df_train_normal["simulationRun"].nunique()
)

print(
    "Corridas en validación:",
    df_val_normal["simulationRun"].nunique()
)

In [ ]:
#Definir que variables entran en el modelo
metadata_cols = [
    "faultNumber",
    "simulationRun",
    "sample"
]

feature_cols = [
    col
    for col in df_train_normal.columns
    if col not in metadata_cols
]

print("Número de variables de proceso:", len(feature_cols))
print("Primeras 10 variables:", feature_cols[:10])
print("Últimas 10 variables:", feature_cols[-10:])

In [ ]:
#Revisar variables contantes
nunique_train = (
    df_train_normal[feature_cols]
    .nunique()
    .sort_values()
)

variables_constantes = nunique_train[
    nunique_train <= 1
]

print("Número de variables constantes:", len(variables_constantes))

if len(variables_constantes) > 0:
    display(variables_constantes)
else:
    print("No se encontraron variables constantes.")

In [ ]:
#Revisar si hay variables de muy baja varianza
variabilidad_train = pd.DataFrame({
    "valores_unicos": df_train_normal[feature_cols].nunique(),
    "media": df_train_normal[feature_cols].mean(),
    "desviacion_estandar": df_train_normal[feature_cols].std(),
    "minimo": df_train_normal[feature_cols].min(),
    "maximo": df_train_normal[feature_cols].max()
})

variabilidad_train["rango"] = (
    variabilidad_train["maximo"]
    - variabilidad_train["minimo"]
)

variabilidad_train = variabilidad_train.sort_values(
    "desviacion_estandar"
)

display(variabilidad_train.head(10))

In [ ]:
#Comprobar valores infinitos
import numpy as np

numero_infinitos = np.isinf(
    df_train_normal[feature_cols].to_numpy()
).sum()

print("Valores infinitos:", numero_infinitos)

In [ ]:
#Estandarización por PCA

#Matrices de las variables
X_train = df_train_normal[feature_cols].copy()
X_val = df_val_normal[feature_cols].copy()

print("Dimensiones de X_train:", X_train.shape)
print("Dimensiones de X_val:", X_val.shape)

In [ ]:
#Ajustar el escalador con entrenamiento
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled_array = scaler.fit_transform(X_train)
X_val_scaled_array = scaler.transform(X_val)

In [ ]:
#convertir de nuevo en DF
X_train_scaled = pd.DataFrame(
    X_train_scaled_array,
    columns=feature_cols,
    index=X_train.index
)

X_val_scaled = pd.DataFrame(
    X_val_scaled_array,
    columns=feature_cols,
    index=X_val.index
)

print("Entrenamiento escalado:", X_train_scaled.shape)
print("Validación escalada:", X_val_scaled.shape)

In [ ]:
#Verificación del escalado
verificacion_escalado = pd.DataFrame({
    "media_train_escalada": X_train_scaled.mean(),
    "desviacion_train_escalada": X_train_scaled.std(ddof=0),
    "media_val_escalada": X_val_scaled.mean(),
    "desviacion_val_escalada": X_val_scaled.std(ddof=0)
})

display(verificacion_escalado.head(10))

In [ ]:
print("Media absoluta máxima en entrenamiento:")
print(X_train_scaled.mean().abs().max())

print("\nDesviación estándar mínima y máxima:")
print(X_train_scaled.std(ddof=0).min())
print(X_train_scaled.std(ddof=0).max())

## 4. Modelo PCA, T²/SPE-Q y calibración inicial

Selección del número de componentes, cálculo de T² y SPE/Q, umbrales sobre validación normal y evaluación de falsas alarmas.


In [ ]:
#Ajustar un PCA exploratorio con todos los componentes
from sklearn.decomposition import PCA

pca_exploratorio = PCA(
    n_components=None,
    svd_solver="full"
)

pca_exploratorio.fit(X_train_scaled)

print(
    "Número total de componentes:",
    pca_exploratorio.n_components_
)

In [ ]:
varianza_individual = (
    pca_exploratorio.explained_variance_ratio_
)

varianza_acumulada = (
    varianza_individual.cumsum()
)

resumen_varianza = pd.DataFrame({
    "componente": range(
        1,
        len(varianza_individual) + 1
    ),
    "varianza_explicada": varianza_individual,
    "varianza_acumulada": varianza_acumulada
})

display(resumen_varianza.head(15))

In [ ]:
#Calcular cuantos componentes se necesitan
import numpy as np

umbrales_varianza = [0.90, 0.95, 0.99]

componentes_por_umbral = {}

for umbral in umbrales_varianza:
    n_componentes = (
        np.argmax(varianza_acumulada >= umbral) + 1
    )

    componentes_por_umbral[umbral] = n_componentes

    print(
        f"Componentes para explicar "
        f"{umbral * 100:.0f}%: "
        f"{n_componentes}"
    )

In [ ]:
#Gráfica de la varianza acumulada
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.plot(
    resumen_varianza["componente"],
    resumen_varianza["varianza_acumulada"],
    marker="o",
    markersize=3
)

plt.axhline(
    y=0.90,
    linestyle="--",
    label="90 %"
)

plt.axhline(
    y=0.95,
    linestyle="--",
    label="95 %"
)

plt.axhline(
    y=0.99,
    linestyle="--",
    label="99 %"
)

plt.xlabel("Número de componentes principales")
plt.ylabel("Varianza explicada acumulada")
plt.title(
    "Varianza explicada acumulada del PCA"
)

plt.ylim(0, 1.02)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#Definición provisional del número de componentes
n_componentes_pca = componentes_por_umbral[0.95]

print(
    "Número provisional de componentes PCA:",
    n_componentes_pca
)

In [ ]:
print(componentes_por_umbral)
print(n_componentes_pca)

In [ ]:
#Ajustar PCA con 36 componentes
from sklearn.decomposition import PCA

pca_model = PCA(
    n_components= n_componentes_pca,
    svd_solver="full"
)

pca_model.fit(X_train_scaled)

print("Componentes retenidos:", pca_model.n_components_)
print(
    "Varianza total explicada:",
    pca_model.explained_variance_ratio_.sum()
)

In [ ]:
scores_train = pca_model.transform(X_train_scaled)
scores_val = pca_model.transform(X_val_scaled)

print("Scores de entrenamiento:", scores_train.shape)
print("Scores de validación:", scores_val.shape)

In [ ]:
#Calcular hotelling T2
eigenvalues = pca_model.explained_variance_

T2_train = np.sum(
    (scores_train ** 2) / eigenvalues,
    axis=1
)

T2_val = np.sum(
    (scores_val ** 2) / eigenvalues,
    axis=1
)

print("T² entrenamiento:", T2_train.shape)
print("T² validación:", T2_val.shape)

In [ ]:
X_train_reconstructed = pca_model.inverse_transform(
    scores_train
)

X_val_reconstructed = pca_model.inverse_transform(
    scores_val
)

print(
    "Reconstrucción entrenamiento:",
    X_train_reconstructed.shape
)

print(
    "Reconstrucción validación:",
    X_val_reconstructed.shape
)

In [ ]:
residuals_train = (
    X_train_scaled.to_numpy()
    - X_train_reconstructed
)

residuals_val = (
    X_val_scaled.to_numpy()
    - X_val_reconstructed
)

SPE_train = np.sum(
    residuals_train ** 2,
    axis=1
)

SPE_val = np.sum(
    residuals_val ** 2,
    axis=1
)

print("SPE entrenamiento:", SPE_train.shape)
print("SPE validación:", SPE_val.shape)

In [ ]:
#Resumir distribuciones normales
resumen_estadisticos_pca = pd.DataFrame({
    "Conjunto": [
        "Entrenamiento",
        "Validación"
    ],
    "T2_media": [
        T2_train.mean(),
        T2_val.mean()
    ],
    "T2_percentil_95": [
        np.quantile(T2_train, 0.95),
        np.quantile(T2_val, 0.95)
    ],
    "T2_percentil_99": [
        np.quantile(T2_train, 0.99),
        np.quantile(T2_val, 0.99)
    ],
    "SPE_media": [
        SPE_train.mean(),
        SPE_val.mean()
    ],
    "SPE_percentil_95": [
        np.quantile(SPE_train, 0.95),
        np.quantile(SPE_val, 0.95)
    ],
    "SPE_percentil_99": [
        np.quantile(SPE_train, 0.99),
        np.quantile(SPE_val, 0.99)
    ]
})

display(resumen_estadisticos_pca)

In [ ]:
print(
    pca_model.explained_variance_ratio_.sum()
)

display(resumen_estadisticos_pca)

In [ ]:
# Nivel de confianza inicial
quantile_threshold = 0.99

T2_threshold = np.quantile(
    T2_val,
    quantile_threshold
)

SPE_threshold = np.quantile(
    SPE_val,
    quantile_threshold
)

print(f"Umbral T² al 99 %: {T2_threshold:.6f}")
print(f"Umbral SPE al 99 %: {SPE_threshold:.6f}")

In [ ]:
#Calcular falsas alarmas
# Alarmas individuales
alarm_T2_train = T2_train > T2_threshold
alarm_T2_val = T2_val > T2_threshold

alarm_SPE_train = SPE_train > SPE_threshold
alarm_SPE_val = SPE_val > SPE_threshold

# Alarma combinada: se activa si supera cualquiera de los dos límites
alarm_combined_train = alarm_T2_train | alarm_SPE_train
alarm_combined_val = alarm_T2_val | alarm_SPE_val

false_alarm_summary = pd.DataFrame({
    "Conjunto": [
        "Entrenamiento",
        "Validación"
    ],
    "FAR_T2": [
        alarm_T2_train.mean(),
        alarm_T2_val.mean()
    ],
    "FAR_SPE": [
        alarm_SPE_train.mean(),
        alarm_SPE_val.mean()
    ],
    "FAR_combinada": [
        alarm_combined_train.mean(),
        alarm_combined_val.mean()
    ]
})

display(false_alarm_summary)

In [ ]:
resultados_pca_val = (
    df_val_normal[
        ["simulationRun", "sample"]
    ]
    .copy()
    .reset_index(drop=True)
)

resultados_pca_val["T2"] = T2_val
resultados_pca_val["SPE"] = SPE_val

resultados_pca_val["alarma_T2"] = alarm_T2_val
resultados_pca_val["alarma_SPE"] = alarm_SPE_val
resultados_pca_val["alarma_combinada_puntual"] = (
    alarm_combined_val
)

resultados_pca_val = (
    resultados_pca_val
    .sort_values(
        ["simulationRun", "sample"]
    )
    .reset_index(drop=True)
)

display(resultados_pca_val.head(10))

In [ ]:
#Verificación
print("Dimensiones:", resultados_pca_val.shape)

print(
    "Corridas:",
    resultados_pca_val["simulationRun"].nunique()
)

muestras_por_corrida_val = (
    resultados_pca_val
    .groupby("simulationRun")
    .size()
)

print(
    "Mínimo de muestras por corrida:",
    muestras_por_corrida_val.min()
)

print(
    "Máximo de muestras por corrida:",
    muestras_por_corrida_val.max()
)

## 5. Persistencia temporal y checkpoints de PCA

Aplicación de la regla temporal, caracterización de episodios falsos y guardado de artefactos del detector PCA.


In [ ]:
#Regla temporal: 3 de 5 exclusivamente sobre las alarmas puntuales combinadas de validación normal

# Número de muestras de la ventana temporal
ventana = 5

# Número mínimo de alarmas puntuales requerido dentro de la ventana
minimo_alarmas = 3

resultados_pca_val["alarmas_en_ultimas_5"] = (
    resultados_pca_val
    .groupby("simulationRun")["alarma_combinada_puntual"]
    .transform(
        lambda serie: (
            serie.astype(int)
            .rolling(
                window=ventana,
                min_periods=ventana
            )
            .sum()
        )
    )
)

resultados_pca_val["alarma_persistente_3de5"] = (
    resultados_pca_val["alarmas_en_ultimas_5"]
    >= minimo_alarmas
)

display(
    resultados_pca_val[
        [
            "simulationRun",
            "sample",
            "alarma_combinada_puntual",
            "alarmas_en_ultimas_5",
            "alarma_persistente_3de5"
        ]
    ].head(10)
)

In [ ]:
#Cálculo nueva tasa de falsas alarmas

far_puntual_val = (
    resultados_pca_val["alarma_combinada_puntual"]
    .mean()
)

far_persistente_val = (
    resultados_pca_val["alarma_persistente_3de5"]
    .mean()
)

reduccion_relativa_far = (
    1 - far_persistente_val / far_puntual_val
)

resumen_persistencia = pd.DataFrame({
    "Indicador": [
        "FAR combinada puntual",
        "FAR con regla 3 de 5",
        "Reducción relativa de falsas alarmas"
    ],
    "Valor": [
        far_puntual_val,
        far_persistente_val,
        reduccion_relativa_far
    ]
})

display(resumen_persistencia)

In [ ]:
#Medir cuantas corridas normales presentan una falsa alarma
alarmas_por_corrida_val = (
    resultados_pca_val
    .groupby("simulationRun")
    .agg(
        muestras_con_alarma_puntual=(
            "alarma_combinada_puntual",
            "sum"
        ),
        muestras_con_alarma_persistente=(
            "alarma_persistente_3de5",
            "sum"
        ),
        existe_alarma_persistente=(
            "alarma_persistente_3de5",
            "any"
        )
    )
    .reset_index()
)

corridas_con_falsa_alarma = (
    alarmas_por_corrida_val["existe_alarma_persistente"]
    .sum()
)

proporcion_corridas_con_falsa_alarma = (
    alarmas_por_corrida_val["existe_alarma_persistente"]
    .mean()
)

print(
    "Corridas normales con al menos una alarma persistente:",
    corridas_con_falsa_alarma,
    "de",
    len(alarmas_por_corrida_val)
)

print(
    "Proporción de corridas afectadas:",
    proporcion_corridas_con_falsa_alarma
)

display(
    alarmas_por_corrida_val
    .sort_values(
        "muestras_con_alarma_persistente",
        ascending=False
    )
    .head(10)
)

In [ ]:
display(resumen_persistencia)

print(
    "Número total de muestras normales con alarma persistente:",
    resultados_pca_val["alarma_persistente_3de5"].sum()
)

print(
    "Total de muestras normales evaluadas:",
    len(resultados_pca_val)
)

In [ ]:
#Identificar episodios continuos de falsas alarmas
resultados_eventos_val = resultados_pca_val.copy()

# Detectar el inicio de un nuevo episodio dentro de cada corrida
inicio_evento = (
    resultados_eventos_val
    .groupby("simulationRun")["alarma_persistente_3de5"]
    .transform(
        lambda serie: (
            serie
            & ~serie.shift(fill_value=False)
        )
    )
)

# Asignar un identificador acumulativo a los episodios
resultados_eventos_val["id_evento"] = (
    inicio_evento
    .groupby(resultados_eventos_val["simulationRun"])
    .cumsum()
)

# Dejar el identificador únicamente en las muestras con alarma
resultados_eventos_val.loc[
    ~resultados_eventos_val["alarma_persistente_3de5"],
    "id_evento"
] = pd.NA

In [ ]:
#Resumir episodios
episodios_falsos = (
    resultados_eventos_val
    .dropna(subset=["id_evento"])
    .groupby(
        ["simulationRun", "id_evento"],
        as_index=False
    )
    .agg(
        muestra_inicio=("sample", "min"),
        muestra_fin=("sample", "max"),
        duracion_muestras=(
            "alarma_persistente_3de5",
            "size"
        ),
        T2_maximo=("T2", "max"),
        SPE_maximo=("SPE", "max")
    )
)

episodios_falsos["duracion_minutos"] = (
    episodios_falsos["duracion_muestras"] * 3
)

display(episodios_falsos.head(10))

In [ ]:
#Métricas operacionales

horas_normales_evaluadas = (
    len(resultados_pca_val) * 3 / 60
)

numero_episodios = len(episodios_falsos)

resumen_eventos_falsos = pd.DataFrame({
    "Indicador": [
        "Episodios falsos totales",
        "Corridas con al menos un episodio",
        "Duración media de episodio (min)",
        "Duración mediana de episodio (min)",
        "Duración máxima de episodio (min)",
        "Episodios por 100 horas normales"
    ],
    "Valor": [
        numero_episodios,
        episodios_falsos["simulationRun"].nunique(),
        episodios_falsos["duracion_minutos"].mean(),
        episodios_falsos["duracion_minutos"].median(),
        episodios_falsos["duracion_minutos"].max(),
        numero_episodios / horas_normales_evaluadas * 100
    ]
})

display(resumen_eventos_falsos)

In [ ]:
#Guardar un punto de control del modelo
from pathlib import Path
import joblib

MODELS_DIR = PROJECT_DIR / "Modelos"
RESULTS_DIR = PROJECT_DIR / "Resultados"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta de modelos:", MODELS_DIR)
print("Carpeta de resultados:", RESULTS_DIR)

In [ ]:
checkpoint_pca = {
    "scaler": scaler,
    "pca_model": pca_model,
    "feature_cols": list(feature_cols),

    "n_componentes": int(n_componentes_pca),
    "varianza_explicada": float(
        pca_model.explained_variance_ratio_.sum()
    ),

    "T2_threshold": float(T2_threshold),
    "SPE_threshold": float(SPE_threshold),
    "quantile_threshold": float(quantile_threshold),

    "ventana_persistencia": int(ventana),
    "minimo_alarmas": int(minimo_alarmas),

    "corridas_train": [
        int(x) for x in corridas_train
    ],
    "corridas_val": [
        int(x) for x in corridas_val
    ]
}

ruta_checkpoint = (
    MODELS_DIR / "pca_baseline_v1.joblib"
)

joblib.dump(
    checkpoint_pca,
    ruta_checkpoint
)

print("Modelo guardado:", ruta_checkpoint)
print("¿Existe el archivo?:", ruta_checkpoint.exists())
print(
    "Tamaño:",
    round(
        ruta_checkpoint.stat().st_size / (1024 ** 2),
        2
    ),
    "MB"
)

In [ ]:
#Guardar el escalador, PCA y configuración
checkpoint_pca = {
    "scaler": scaler,
    "pca_model": pca_model,
    "feature_cols": list(feature_cols),

    "n_componentes": int(n_componentes_pca),
    "varianza_explicada": float(
        pca_model.explained_variance_ratio_.sum()
    ),

    "T2_threshold": float(T2_threshold),
    "SPE_threshold": float(SPE_threshold),
    "quantile_threshold": float(quantile_threshold),

    "ventana_persistencia": int(ventana),
    "minimo_alarmas": int(minimo_alarmas),

    "corridas_train": [
        int(x) for x in corridas_train
    ],
    "corridas_val": [
        int(x) for x in corridas_val
    ]
}

ruta_checkpoint = (
    MODELS_DIR / "pca_baseline_v1.joblib"
)

joblib.dump(
    checkpoint_pca,
    ruta_checkpoint
)

print("Modelo guardado:", ruta_checkpoint)
print("¿Existe el archivo?:", ruta_checkpoint.exists())
print(
    "Tamaño:",
    round(
        ruta_checkpoint.stat().st_size / (1024 ** 2),
        2
    ),
    "MB"
)

In [ ]:
#Guardar resultados de validación
resumen_eventos_falsos.to_csv(
    RESULTS_DIR / "resumen_eventos_falsos_pca_v1.csv",
    index=False
)

episodios_falsos.to_csv(
    RESULTS_DIR / "episodios_falsos_pca_v1.csv",
    index=False
)

resumen_estadisticos_pca.to_csv(
    RESULTS_DIR / "estadisticos_pca_train_validacion_v1.csv",
    index=False
)

print("Resultados guardados correctamente.")

In [ ]:
#Verificar los archivos creados
print("Archivos guardados:\n")

for archivo in sorted(
    list(MODELS_DIR.iterdir())
    + list(RESULTS_DIR.iterdir())
):
    print(archivo)

In [ ]:
#Liberar memoria
import gc
import psutil

# Variables grandes que ya fueron guardadas y no necesitamos mantener en RAM
variables_a_eliminar = [
    "ff_train",
    "ff_train_clean",
    "df_train_normal",
    "df_val_normal",
    "X_train",
    "X_val",
    "X_train_scaled",
    "X_val_scaled",
    "X_train_scaled_array",
    "X_val_scaled_array",
    "scores_train",
    "scores_val",
    "X_train_reconstructed",
    "X_val_reconstructed",
    "residuals_train",
    "residuals_val",
    "T2_train",
    "T2_val",
    "SPE_train",
    "SPE_val",
    "resultados_pca_val",
    "resultados_eventos_val"
]

for variable in variables_a_eliminar:
    globals().pop(variable, None)

gc.collect()

memoria = psutil.virtual_memory()

print(
    f"Memoria disponible: "
    f"{memoria.available / (1024 ** 3):.2f} GB"
)

print(
    f"Memoria utilizada: "
    f"{memoria.percent:.1f} %"
)

In [ ]:
import joblib

ruta_checkpoint = (
    PROJECT_DIR / "Modelos" / "pca_baseline_v1.joblib"
)

checkpoint_pca = joblib.load(ruta_checkpoint)

scaler = checkpoint_pca["scaler"]
pca_model = checkpoint_pca["pca_model"]
feature_cols = checkpoint_pca["feature_cols"]

T2_threshold = checkpoint_pca["T2_threshold"]
SPE_threshold = checkpoint_pca["SPE_threshold"]

ventana = checkpoint_pca["ventana_persistencia"]
minimo_alarmas = checkpoint_pca["minimo_alarmas"]

print("Modelo recuperado correctamente.")
print("Variables del modelo:", len(feature_cols))
print("Componentes PCA:", pca_model.n_components_)
print("Umbral T²:", T2_threshold)
print("Umbral SPE:", SPE_threshold)
print(
    f"Regla temporal provisional: "
    f"{minimo_alarmas} de {ventana}"
)

## 6. Evaluación PCA en Faulty Training

Carga de datos con fallas, evaluación por corrida, detección post-falla, retrasos y análisis de las 20 perturbaciones.


In [ ]:
#Copiar el archivo con fallas al almacenamiento local
from pathlib import Path
from shutil import copy2

archivo_faulty_drive = (
    DATA_DIR / "TEP_Faulty_Training.RData"
)

archivo_faulty_local = Path(
    "/content/TEP_Faulty_Training.RData"
)

print(
    "¿Existe el archivo en Drive?:",
    archivo_faulty_drive.exists()
)

if not archivo_faulty_local.exists():
    print("Copiando archivo desde Drive...")
    copy2(
        archivo_faulty_drive,
        archivo_faulty_local
    )
    print("Copia terminada.")
else:
    print("El archivo ya existe en /content.")

print(
    "Tamaño local:",
    round(
        archivo_faulty_local.stat().st_size / (1024 ** 2),
        2
    ),
    "MB"
)

In [ ]:
#Comprobar memoria
memoria = psutil.virtual_memory()

print(
    f"Memoria disponible antes de la lectura: "
    f"{memoria.available / (1024 ** 3):.2f} GB"
)

In [ ]:
#Carga del archivo con fallas
import pyreadr

print("Iniciando lectura del archivo...")

resultado_faulty_train = pyreadr.read_r(
    str(archivo_faulty_local)
)

print("Lectura terminada.")
print(
    "Objetos internos:",
    list(resultado_faulty_train.keys())
)

In [ ]:
#Extraer el DF
nombre_objeto_faulty = next(
    iter(resultado_faulty_train.keys())
)

f_train = resultado_faulty_train[
    nombre_objeto_faulty
]

# Ya no necesitamos conservar el diccionario contenedor
del resultado_faulty_train
gc.collect()

print("Objeto seleccionado:", nombre_objeto_faulty)
print("Tipo:", type(f_train))
print("Dimensiones:", f_train.shape)

In [ ]:
display(
    f_train[
        ["faultNumber", "simulationRun", "sample"]
    ].head(10)
)

print(
    "Tipos de falla presentes:",
    sorted(
        f_train["faultNumber"]
        .astype(int)
        .unique()
    )
)

print(
    "Número de tipos de falla:",
    f_train["faultNumber"].nunique()
)

In [ ]:
#Validar corridas completas
muestras_por_corrida_faulty = (
    f_train
    .groupby(
        ["faultNumber", "simulationRun"]
    )
    .size()
)

print(
    "Número total de corridas falla-simulación:",
    len(muestras_por_corrida_faulty)
)

print(
    "Mínimo de muestras por corrida:",
    muestras_por_corrida_faulty.min()
)

print(
    "Máximo de muestras por corrida:",
    muestras_por_corrida_faulty.max()
)

print(
    "Valores distintos de muestras por corrida:",
    muestras_por_corrida_faulty
    .unique()[:10]
)

In [ ]:
#Confirmar tipos de fallas
fallas_disponibles = sorted(
    f_train["faultNumber"]
    .astype(int)
    .unique()
)

print("Fallas disponibles:", fallas_disponibles)
print("Número de tipos de falla:", len(fallas_disponibles))

In [ ]:
# En el conjunto Faulty Training, las primeras 20 muestras corresponden al periodo previo a la introducción de la falla.

MUESTRA_INICIO_FALLA_TRAIN = 21

f_train["is_fault"] = (
    f_train["sample"] >= MUESTRA_INICIO_FALLA_TRAIN
).astype("int8")

print(f_train["is_fault"].value_counts().sort_index())

In [ ]:
#Verificar cambio de estado en una corrida
verificacion_inicio_falla = f_train.loc[
    (f_train["faultNumber"] == 1)
    & (f_train["simulationRun"] == 1)
    & (f_train["sample"].between(17, 24)),
    [
        "faultNumber",
        "simulationRun",
        "sample",
        "is_fault"
    ]
]

display(verificacion_inicio_falla)

In [ ]:
#Cuantificar las muestras normales y con falla
resumen_clases_faulty_train = (
    f_train["is_fault"]
    .value_counts()
    .sort_index()
    .rename_axis("is_fault")
    .reset_index(name="numero_muestras")
)

resumen_clases_faulty_train["porcentaje"] = (
    resumen_clases_faulty_train["numero_muestras"]
    / len(f_train)
    * 100
)

display(resumen_clases_faulty_train)

In [ ]:
# Seleccionar únicamente las 500 corridas correspondientes a la falla 1
falla_1 = (
    f_train[
        f_train["faultNumber"].astype(int) == 1
    ]
    .copy()
    .sort_values(["simulationRun", "sample"])
    .reset_index(drop=True)
)

print("Dimensiones de la falla 1:", falla_1.shape)
print(
    "Número de corridas:",
    falla_1["simulationRun"].nunique()
)
print(
    "Muestras por corrida:",
    falla_1.groupby("simulationRun").size().unique()
)

In [ ]:
conteo_estado_falla1 = (
    falla_1
    .groupby(["simulationRun", "is_fault"])
    .size()
    .unstack(fill_value=0)
)

display(conteo_estado_falla1.head())

print(
    "Mínimo y máximo de muestras normales por corrida:",
    conteo_estado_falla1[0].min(),
    conteo_estado_falla1[0].max()
)

print(
    "Mínimo y máximo de muestras post-falla por corrida:",
    conteo_estado_falla1[1].min(),
    conteo_estado_falla1[1].max()
)

In [ ]:
X_falla_1 = falla_1[feature_cols]

print("Dimensiones:", X_falla_1.shape)

In [ ]:
X_falla_1_scaled = scaler.transform(X_falla_1)

In [ ]:
scores_falla_1 = pca_model.transform(
    X_falla_1_scaled
)

print(
    "Dimensiones de los scores:",
    scores_falla_1.shape
)

In [ ]:
eigenvalues = pca_model.explained_variance_

T2_falla_1 = np.sum(
    (scores_falla_1 ** 2) / eigenvalues,
    axis=1
)

In [ ]:
# Recosntrucción de observaciones usando PCA
X_falla_1_reconstructed = (
    pca_model.inverse_transform(
        scores_falla_1
    )
)

In [ ]:
#Cálculo del residuo
residuals_falla_1 = (
    X_falla_1_scaled
    - X_falla_1_reconstructed
)

In [ ]:
SPE_falla_1 = np.sum(
    residuals_falla_1 ** 2,
    axis=1
)

In [ ]:
resultados_falla_1 = falla_1[
    [
        "faultNumber",
        "simulationRun",
        "sample",
        "is_fault"
    ]
].copy()

resultados_falla_1["T2"] = T2_falla_1
resultados_falla_1["SPE"] = SPE_falla_1

resultados_falla_1["alarma_T2"] = (
    resultados_falla_1["T2"]
    > T2_threshold
)

resultados_falla_1["alarma_SPE"] = (
    resultados_falla_1["SPE"]
    > SPE_threshold
)

resultados_falla_1["alarma_puntual"] = (
    resultados_falla_1["alarma_T2"]
    |
    resultados_falla_1["alarma_SPE"]
)

display(resultados_falla_1.head(25))

In [ ]:
comparacion_pre_post_falla1 = (
    resultados_falla_1
    .groupby("is_fault")
    .agg(
        T2_media=("T2", "mean"),
        T2_mediana=("T2", "median"),
        T2_p99=("T2", lambda x: x.quantile(0.99)),
        SPE_media=("SPE", "mean"),
        SPE_mediana=("SPE", "median"),
        SPE_p99=("SPE", lambda x: x.quantile(0.99)),
        proporcion_alarma_puntual=(
            "alarma_puntual",
            "mean"
        )
    )
)

comparacion_pre_post_falla1.index = [
    "Periodo normal previo",
    "Periodo posterior a la falla"
]

display(comparacion_pre_post_falla1)

In [ ]:
ventana = 5
minimo_alarmas = 3

resultados_falla_1["alarmas_en_ultimas_5"] = (
    resultados_falla_1
    .groupby("simulationRun")["alarma_puntual"]
    .transform(
        lambda serie: (
            serie.astype(int)
            .rolling(
                window=ventana,
                min_periods=ventana
            )
            .sum()
        )
    )
)

resultados_falla_1["alarma_persistente"] = (
    resultados_falla_1["alarmas_en_ultimas_5"]
    >= minimo_alarmas
)

In [ ]:
display(
    resultados_falla_1.loc[
        (resultados_falla_1["simulationRun"] == 1)
        & (resultados_falla_1["sample"].between(15, 35)),
        [
            "sample",
            "is_fault",
            "T2",
            "SPE",
            "alarma_puntual",
            "alarmas_en_ultimas_5",
            "alarma_persistente"
        ]
    ]
)

In [ ]:
resultados_falla_1["inicio_alarma_persistente"] = (
    resultados_falla_1
    .groupby("simulationRun")["alarma_persistente"]
    .transform(
        lambda s: s & ~s.shift(fill_value=False)
    )
)

In [ ]:
#separar falsas alarmas
resultados_falla_1["inicio_falsa_alarma_previa"] = (
    resultados_falla_1["inicio_alarma_persistente"]
    & (resultados_falla_1["sample"] < 21)
)

resultados_falla_1["inicio_deteccion_post_falla"] = (
    resultados_falla_1["inicio_alarma_persistente"]
    & (resultados_falla_1["sample"] >= 21)
)

In [ ]:
#Encontrar la primera detección de cada corrida
primeras_detecciones_falla1 = (
    resultados_falla_1[
        resultados_falla_1["inicio_deteccion_post_falla"]
    ]
    .groupby("simulationRun", as_index=False)
    .agg(
        muestra_deteccion=("sample", "min")
    )
)

display(primeras_detecciones_falla1.head(10))

In [ ]:
#Calcular el retraso
primeras_detecciones_falla1["retraso_muestras"] = (
    primeras_detecciones_falla1["muestra_deteccion"]
    - 21
)

primeras_detecciones_falla1["retraso_minutos"] = (
    primeras_detecciones_falla1["retraso_muestras"]
    * 3
)

display(primeras_detecciones_falla1.head(10))

In [ ]:
total_corridas_falla1 = (
    resultados_falla_1["simulationRun"].nunique()
)

corridas_detectadas_falla1 = (
    primeras_detecciones_falla1["simulationRun"].nunique()
)

tasa_deteccion_falla1 = (
    corridas_detectadas_falla1
    / total_corridas_falla1
)

print(
    "Corridas totales:",
    total_corridas_falla1
)

print(
    "Corridas detectadas:",
    corridas_detectadas_falla1
)

print(
    f"Tasa de detección: "
    f"{tasa_deteccion_falla1:.2%}"
)

In [ ]:
#Resumen tiempos de detección
resumen_retraso_falla1 = pd.DataFrame({
    "Indicador": [
        "Corridas totales",
        "Corridas detectadas",
        "Tasa de detección",
        "Retraso medio (min)",
        "Retraso mediano (min)",
        "Percentil 95 del retraso (min)",
        "Retraso mínimo (min)",
        "Retraso máximo (min)"
    ],
    "Valor": [
        total_corridas_falla1,
        corridas_detectadas_falla1,
        tasa_deteccion_falla1,
        primeras_detecciones_falla1["retraso_minutos"].mean(),
        primeras_detecciones_falla1["retraso_minutos"].median(),
        primeras_detecciones_falla1["retraso_minutos"].quantile(0.95),
        primeras_detecciones_falla1["retraso_minutos"].min(),
        primeras_detecciones_falla1["retraso_minutos"].max()
    ]
})

display(resumen_retraso_falla1)

In [ ]:
#Medir falsas alarmas antes de fallas
corridas_con_falsa_alarma_previa = (
    resultados_falla_1.loc[
        resultados_falla_1["inicio_falsa_alarma_previa"],
        "simulationRun"
    ]
    .nunique()
)

print(
    "Corridas con falsa alarma antes de la falla:",
    corridas_con_falsa_alarma_previa,
    "de",
    total_corridas_falla1
)

In [ ]:
corridas_con_falsa_alarma_previa = (
    resultados_falla_1.loc[
        resultados_falla_1["inicio_falsa_alarma_previa"],
        "simulationRun"
    ]
    .nunique()
)

porcentaje_falsa_alarma_previa = (
    corridas_con_falsa_alarma_previa
    / total_corridas_falla1
)

print(
    "Corridas con falsa alarma antes de la falla:",
    corridas_con_falsa_alarma_previa,
    "de",
    total_corridas_falla1
)

print(
    f"Porcentaje de corridas con falsa alarma previa: "
    f"{porcentaje_falsa_alarma_previa:.2%}"
)

In [ ]:
#Función de evaluación
import gc
import numpy as np
import pandas as pd


def evaluar_falla_pca(
    df_faulty,
    numero_falla,
    feature_cols,
    scaler,
    pca_model,
    T2_threshold,
    SPE_threshold,
    ventana=5,
    minimo_alarmas=3,
    muestra_inicio_falla=21,
    minutos_por_muestra=3
):
    """
    Evalúa un tipo de falla del Tennessee Eastman Process utilizando un modelo PCA entrenado exclusivamente con operación normal.

    Retorna un diccionario con métricas de:
    - tasa de detección
    - retraso de detección
    - falsas alarmas previas a la falla
    """

    # =========================================================
    # 1. Asegurar el orden correcto de las variables
    # =========================================================

    # Si el scaler conserva los nombres usados durante el ajuste,
    # utilizamos exactamente ese mismo orden.
    if hasattr(scaler, "feature_names_in_"):
        columnas_modelo = list(scaler.feature_names_in_)
    else:
        columnas_modelo = list(feature_cols)

    # Verificar que todas las variables necesarias existan
    columnas_faltantes = [
        col for col in columnas_modelo
        if col not in df_faulty.columns
    ]

    if columnas_faltantes:
        raise ValueError(
            f"Faltan variables requeridas por el modelo: "
            f"{columnas_faltantes}"
        )

    # =========================================================
    # 2. Seleccionar únicamente la falla solicitada
    # =========================================================

    df = (
        df_faulty.loc[
            df_faulty["faultNumber"].astype(int) == numero_falla
        ]
        .copy()
        .sort_values(
            ["simulationRun", "sample"]
        )
        .reset_index(drop=True)
    )

    if df.empty:
        raise ValueError(
            f"No se encontraron datos para la falla {numero_falla}"
        )

    total_corridas = (
        df["simulationRun"].nunique()
    )

    # =========================================================
    # 3. Seleccionar las 52 variables de proceso
    # =========================================================

    X = df.loc[:, columnas_modelo].copy()

    # =========================================================
    # 4. Estandarización
    # =========================================================
    # IMPORTANTE:
    # aquí usamos transform(), NO fit_transform().
    #
    # El scaler mantiene las medias y desviaciones aprendidas
    # exclusivamente con operación normal.
    # =========================================================

    X_scaled_array = scaler.transform(X)

    # Reconstruimos un DataFrame con nombres de variables.
    # Esto evita el warning de sklearn y conserva la trazabilidad.
    X_scaled = pd.DataFrame(
        X_scaled_array,
        columns=columnas_modelo,
        index=X.index
    )

    # =========================================================
    # 5. Proyección sobre el PCA normal
    # =========================================================

    scores = pca_model.transform(X_scaled)

    eigenvalues = (
        pca_model.explained_variance_
    )

    # =========================================================
    # 6. Estadístico Hotelling T²
    # =========================================================

    T2 = np.sum(
        (scores ** 2) / eigenvalues,
        axis=1
    )

    # =========================================================
    # 7. Reconstrucción PCA y SPE/Q
    # =========================================================

    X_reconstructed = (
        pca_model.inverse_transform(scores)
    )

    residuals = (
        X_scaled.to_numpy()
        - X_reconstructed
    )

    SPE = np.sum(
        residuals ** 2,
        axis=1
    )

    # =========================================================
    # 8. Tabla temporal de resultados
    # =========================================================

    resultados = df[
        [
            "simulationRun",
            "sample"
        ]
    ].copy()

    resultados["T2"] = T2
    resultados["SPE"] = SPE

    # =========================================================
    # 9. Alarmas puntuales
    # =========================================================

    resultados["alarma_T2"] = (
        resultados["T2"]
        > T2_threshold
    )

    resultados["alarma_SPE"] = (
        resultados["SPE"]
        > SPE_threshold
    )

    # Se activa si se supera cualquiera de los dos límites
    resultados["alarma_puntual"] = (
        resultados["alarma_T2"]
        |
        resultados["alarma_SPE"]
    )

    # =========================================================
    # 10. Regla temporal de persistencia
    # =========================================================
    # Ejemplo actual:
    # al menos 3 alarmas dentro de las últimas 5 muestras.
    #
    # El groupby evita mezclar corridas diferentes.
    # =========================================================

    resultados["alarmas_ventana"] = (
        resultados
        .groupby("simulationRun")["alarma_puntual"]
        .transform(
            lambda serie: (
                serie.astype(int)
                .rolling(
                    window=ventana,
                    min_periods=ventana
                )
                .sum()
            )
        )
    )

    resultados["alarma_persistente"] = (
        resultados["alarmas_ventana"]
        >= minimo_alarmas
    )

    # =========================================================
    # 11. Detectar el INICIO de cada episodio persistente
    # =========================================================

    resultados["inicio_episodio"] = (
        resultados
        .groupby("simulationRun")["alarma_persistente"]
        .transform(
            lambda serie: (
                serie
                &
                ~serie.shift(fill_value=False)
            )
        )
    )

    # =========================================================
    # 12. Falsas alarmas PREVIAS a la introducción de la falla
    # =========================================================

    mascara_falsa_previa = (
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            < muestra_inicio_falla
        )
    )

    corridas_falsa_alarma_previa = (
        resultados.loc[
            mascara_falsa_previa,
            "simulationRun"
        ]
        .nunique()
    )

    porcentaje_falsa_alarma_previa = (
        corridas_falsa_alarma_previa
        / total_corridas
    )

    # =========================================================
    # 13. Detecciones posteriores a la falla
    # =========================================================

    mascara_deteccion = (
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            >= muestra_inicio_falla
        )
    )

    primeras_detecciones = (
        resultados.loc[
            mascara_deteccion,
            [
                "simulationRun",
                "sample"
            ]
        ]
        .groupby(
            "simulationRun",
            as_index=False
        )
        .agg(
            muestra_deteccion=(
                "sample",
                "min"
            )
        )
    )

    corridas_detectadas = len(
        primeras_detecciones
    )

    tasa_deteccion = (
        corridas_detectadas
        / total_corridas
    )

    # =========================================================
    # 14. Retraso de detección
    # =========================================================

    primeras_detecciones[
        "retraso_muestras"
    ] = (
        primeras_detecciones[
            "muestra_deteccion"
        ]
        - muestra_inicio_falla
    )

    primeras_detecciones[
        "retraso_minutos"
    ] = (
        primeras_detecciones[
            "retraso_muestras"
        ]
        * minutos_por_muestra
    )

    if corridas_detectadas > 0:

        retraso_medio = (
            primeras_detecciones[
                "retraso_minutos"
            ].mean()
        )

        retraso_mediano = (
            primeras_detecciones[
                "retraso_minutos"
            ].median()
        )

        retraso_p95 = (
            primeras_detecciones[
                "retraso_minutos"
            ].quantile(0.95)
        )

        retraso_minimo = (
            primeras_detecciones[
                "retraso_minutos"
            ].min()
        )

        retraso_maximo = (
            primeras_detecciones[
                "retraso_minutos"
            ].max()
        )

    else:

        retraso_medio = np.nan
        retraso_mediano = np.nan
        retraso_p95 = np.nan
        retraso_minimo = np.nan
        retraso_maximo = np.nan

    # =========================================================
    # 15. Crear resumen
    # =========================================================

    resumen = {
        "falla": int(numero_falla),

        "corridas_totales": int(
            total_corridas
        ),

        "corridas_detectadas": int(
            corridas_detectadas
        ),

        "tasa_deteccion": float(
            tasa_deteccion
        ),

        "retraso_medio_min": float(
            retraso_medio
        ) if not np.isnan(retraso_medio)
        else np.nan,

        "retraso_mediano_min": float(
            retraso_mediano
        ) if not np.isnan(retraso_mediano)
        else np.nan,

        "retraso_p95_min": float(
            retraso_p95
        ) if not np.isnan(retraso_p95)
        else np.nan,

        "retraso_minimo_min": float(
            retraso_minimo
        ) if not np.isnan(retraso_minimo)
        else np.nan,

        "retraso_maximo_min": float(
            retraso_maximo
        ) if not np.isnan(retraso_maximo)
        else np.nan,

        "corridas_falsa_alarma_previa": int(
            corridas_falsa_alarma_previa
        ),

        "porcentaje_falsa_alarma_previa": float(
            porcentaje_falsa_alarma_previa
        )
    }

    # =========================================================
    # 16. Liberar memoria
    # =========================================================

    del X
    del X_scaled_array
    del X_scaled
    del scores
    del X_reconstructed
    del residuals

    gc.collect()

    return resumen

In [ ]:
prueba_funcion_falla1 = evaluar_falla_pca(
    df_faulty=f_train,
    numero_falla=1,
    feature_cols=feature_cols,
    scaler=scaler,
    pca_model=pca_model,
    T2_threshold=T2_threshold,
    SPE_threshold=SPE_threshold,
    ventana=5,
    minimo_alarmas=3,
    muestra_inicio_falla=21,
    minutos_por_muestra=3
)

prueba_funcion_falla1_df = pd.DataFrame(
    [prueba_funcion_falla1]
)

display(prueba_funcion_falla1_df)

In [ ]:
#Liberar variables innecesarias
import gc

variables_temporales = [
    "falla_1",
    "X_falla_1",
    "X_falla_1_scaled",
    "scores_falla_1",
    "X_falla_1_reconstructed",
    "residuals_falla_1",
    "T2_falla_1",
    "SPE_falla_1",
    "resultados_falla_1",
    "primeras_detecciones_falla1"
]

for variable in variables_temporales:
    globals().pop(variable, None)

gc.collect()

print("Memoria temporal liberada.")

In [ ]:
#Evaluar las 20 fallas
resultados_20_fallas = []

for numero_falla in range(1, 21):

    print(f"Evaluando falla {numero_falla} de 20...")

    resumen = evaluar_falla_pca(
        df_faulty=f_train,
        numero_falla=numero_falla,
        feature_cols=feature_cols,
        scaler=scaler,
        pca_model=pca_model,
        T2_threshold=T2_threshold,
        SPE_threshold=SPE_threshold,
        ventana=5,
        minimo_alarmas=3,
        muestra_inicio_falla=21,
        minutos_por_muestra=3
    )

    resultados_20_fallas.append(resumen)

    gc.collect()

print("\nEvaluación de las 20 fallas terminada.")

In [ ]:
#Resultados
resumen_20_fallas = pd.DataFrame(
    resultados_20_fallas
)

display(resumen_20_fallas)

In [ ]:
tabla_resultados_pca = resumen_20_fallas.copy()

tabla_resultados_pca["tasa_deteccion_pct"] = (
    tabla_resultados_pca["tasa_deteccion"] * 100
)

tabla_resultados_pca[
    "falsa_alarma_previa_pct"
] = (
    tabla_resultados_pca[
        "porcentaje_falsa_alarma_previa"
    ] * 100
)

columnas_mostrar = [
    "falla",
    "corridas_totales",
    "corridas_detectadas",
    "tasa_deteccion_pct",
    "retraso_medio_min",
    "retraso_mediano_min",
    "retraso_p95_min",
    "retraso_maximo_min",
    "corridas_falsa_alarma_previa",
    "falsa_alarma_previa_pct"
]

display(
    tabla_resultados_pca[columnas_mostrar]
    .round(3)
)

In [ ]:
ruta_resultados_pca = (
    RESULTS_DIR /
    "evaluacion_PCA_20_fallas_training.csv"
)

tabla_resultados_pca.to_csv(
    ruta_resultados_pca,
    index=False
)

print(
    "Resultados guardados correctamente en:\n",
    ruta_resultados_pca
)

In [ ]:
#Clasificación de desempeño
def clasificar_desempeno(row):

    if row["tasa_deteccion_pct"] < 50:
        return "Detección deficiente"

    elif row["tasa_deteccion_pct"] < 90:
        return "Detección moderada"

    elif row["retraso_medio_min"] <= 30:
        return "Alta detección - temprana"

    elif row["retraso_medio_min"] <= 120:
        return "Alta detección - retraso moderado"

    else:
        return "Alta detección - retraso elevado"


tabla_resultados_pca["clasificacion"] = (
    tabla_resultados_pca.apply(
        clasificar_desempeno,
        axis=1
    )
)

display(
    tabla_resultados_pca[
        [
            "falla",
            "tasa_deteccion_pct",
            "retraso_medio_min",
            "retraso_p95_min",
            "clasificacion"
        ]
    ]
)

In [ ]:
resumen_global_pca = pd.DataFrame({
    "Indicador": [
        "Fallas evaluadas",
        "Fallas con detección >= 99 %",
        "Fallas con detección < 50 %",
        "Tasa media de detección entre fallas (%)",
        "Mediana de tasa de detección (%)",
        "Retraso medio entre fallas (min)",
        "Mediana del retraso entre fallas (min)"
    ],
    "Valor": [
        len(tabla_resultados_pca),

        (
            tabla_resultados_pca["tasa_deteccion_pct"]
            >= 99
        ).sum(),

        (
            tabla_resultados_pca["tasa_deteccion_pct"]
            < 50
        ).sum(),

        tabla_resultados_pca[
            "tasa_deteccion_pct"
        ].mean(),

        tabla_resultados_pca[
            "tasa_deteccion_pct"
        ].median(),

        tabla_resultados_pca[
            "retraso_medio_min"
        ].mean(),

        tabla_resultados_pca[
            "retraso_medio_min"
        ].median()
    ]
})

display(resumen_global_pca.round(3))

In [ ]:
tabla_resultados_pca.to_csv(
    RESULTS_DIR /
    "resultados_detallados_PCA_20_fallas.csv",
    index=False
)

resumen_global_pca.to_csv(
    RESULTS_DIR /
    "resumen_global_PCA.csv",
    index=False
)

print("Resultados PCA guardados.")

In [ ]:
#Detalle de la falla 4
def obtener_detalle_falla_pca(
    df_faulty,
    numero_falla,
    feature_cols,
    scaler,
    pca_model,
    T2_threshold,
    SPE_threshold,
    ventana=5,
    minimo_alarmas=3
):

    if hasattr(scaler, "feature_names_in_"):
        columnas_modelo = list(scaler.feature_names_in_)
    else:
        columnas_modelo = list(feature_cols)

    df = (
        df_faulty.loc[
            df_faulty["faultNumber"].astype(int) == numero_falla
        ]
        .sort_values(["simulationRun", "sample"])
        .reset_index(drop=True)
    )

    X = df[columnas_modelo]

    X_scaled = pd.DataFrame(
        scaler.transform(X),
        columns=columnas_modelo,
        index=df.index
    )

    scores = pca_model.transform(X_scaled)

    eigenvalues = pca_model.explained_variance_

    T2 = np.sum(
        (scores ** 2) / eigenvalues,
        axis=1
    )

    X_reconstructed = pca_model.inverse_transform(
        scores
    )

    residuals = (
        X_scaled.to_numpy()
        - X_reconstructed
    )

    SPE = np.sum(
        residuals ** 2,
        axis=1
    )

    resultados = df[
        ["simulationRun", "sample"]
    ].copy()

    resultados["T2"] = T2
    resultados["SPE"] = SPE

    resultados["alarma_puntual"] = (
        (resultados["T2"] > T2_threshold)
        |
        (resultados["SPE"] > SPE_threshold)
    )

    resultados["alarmas_ventana"] = (
        resultados
        .groupby("simulationRun")["alarma_puntual"]
        .transform(
            lambda s:
            s.astype(int)
            .rolling(
                window=ventana,
                min_periods=ventana
            )
            .sum()
        )
    )

    resultados["alarma_persistente"] = (
        resultados["alarmas_ventana"]
        >= minimo_alarmas
    )

    resultados["inicio_episodio"] = (
        resultados
        .groupby("simulationRun")["alarma_persistente"]
        .transform(
            lambda s:
            s & ~s.shift(fill_value=False)
        )
    )

    return resultados

In [ ]:
detalle_falla_4 = obtener_detalle_falla_pca(
    df_faulty=f_train,
    numero_falla=4,
    feature_cols=feature_cols,
    scaler=scaler,
    pca_model=pca_model,
    T2_threshold=T2_threshold,
    SPE_threshold=SPE_threshold,
    ventana=5,
    minimo_alarmas=3
)

In [ ]:
corridas_prealarma_f4 = sorted(
    detalle_falla_4.loc[
        detalle_falla_4["inicio_episodio"]
        & (detalle_falla_4["sample"] < 21),
        "simulationRun"
    ].unique()
)

print(
    "Corridas con episodio previo a la falla:",
    corridas_prealarma_f4
)

In [ ]:
#Identificar que corrida quedó como no detectada
corridas_detectadas_f4 = set(
    detalle_falla_4.loc[
        detalle_falla_4["inicio_episodio"]
        & (detalle_falla_4["sample"] >= 21),
        "simulationRun"
    ].unique()
)

todas_corridas_f4 = set(
    detalle_falla_4["simulationRun"].unique()
)

corridas_no_detectadas_f4 = sorted(
    todas_corridas_f4
    - corridas_detectadas_f4
)

print(
    "Corridas consideradas no detectadas:",
    corridas_no_detectadas_f4
)

In [ ]:
print(
    "Corridas con falsa alarma previa:",
    corridas_prealarma_f4
)

print(
    "Corridas no detectadas:",
    corridas_no_detectadas_f4
)

print(
    "¿Coinciden?:",
    set(corridas_prealarma_f4)
    ==
    set(corridas_no_detectadas_f4)
)

In [ ]:
fallas_dificiles = [3, 9, 15]

resumen_fallas_dificiles = []

for numero_falla in fallas_dificiles:

    detalle = obtener_detalle_falla_pca(
        df_faulty=f_train,
        numero_falla=numero_falla,
        feature_cols=feature_cols,
        scaler=scaler,
        pca_model=pca_model,
        T2_threshold=T2_threshold,
        SPE_threshold=SPE_threshold,
        ventana=5,
        minimo_alarmas=3
    )

    # Solo periodo posterior a la introducción de la falla
    post_falla = detalle[
        detalle["sample"] >= 21
    ].copy()

    resumen_fallas_dificiles.append({
        "falla": numero_falla,

        "T2_medio_post": post_falla["T2"].mean(),

        "SPE_medio_post": post_falla["SPE"].mean(),

        "porcentaje_alarmas_puntuales": (
            post_falla["alarma_puntual"].mean() * 100
        ),

        "porcentaje_T2_supera_umbral": (
            (post_falla["T2"] > T2_threshold).mean() * 100
        ),

        "porcentaje_SPE_supera_umbral": (
            (post_falla["SPE"] > SPE_threshold).mean() * 100
        )
    })

resumen_fallas_dificiles = pd.DataFrame(
    resumen_fallas_dificiles
)

display(
    resumen_fallas_dificiles.round(3)
)

In [ ]:
tabla_resultados_pca["corridas_evaluables"] = (
    tabla_resultados_pca["corridas_totales"]
    - tabla_resultados_pca["corridas_falsa_alarma_previa"]
)

tabla_resultados_pca["tasa_deteccion_evaluable_pct"] = (
    tabla_resultados_pca["corridas_detectadas"]
    / tabla_resultados_pca["corridas_evaluables"]
    * 100
)

display(
    tabla_resultados_pca[
        [
            "falla",
            "corridas_totales",
            "corridas_falsa_alarma_previa",
            "corridas_evaluables",
            "corridas_detectadas",
            "tasa_deteccion_pct",
            "tasa_deteccion_evaluable_pct",
            "retraso_medio_min"
        ]
    ].round(3)
)

In [ ]:
tabla_resultados_pca.to_csv(
    RESULTS_DIR /
    "resultados_PCA_baseline_definitivo.csv",
    index=False
)

print("Baseline PCA actualizado y guardado.")

In [ ]:
import gc
import psutil

globals().pop("f_train", None)

gc.collect()

memoria = psutil.virtual_memory()

print(
    f"Memoria disponible: "
    f"{memoria.available / (1024 ** 3):.2f} GB"
)

print(
    f"Memoria utilizada: "
    f"{memoria.percent:.1f} %"
)

## 7. Isolation Forest

Entrenamiento, calibración a una carga puntual comparable, persistencia temporal, evaluación por falla y comparación con PCA.


In [ ]:
import joblib

ruta_checkpoint = (
    PROJECT_DIR /
    "Modelos" /
    "pca_baseline_v1.joblib"
)

checkpoint_pca = joblib.load(
    ruta_checkpoint
)

corridas_train = checkpoint_pca["corridas_train"]
corridas_val = checkpoint_pca["corridas_val"]

feature_cols = checkpoint_pca["feature_cols"]

print(
    "Corridas de entrenamiento:",
    len(corridas_train)
)

print(
    "Corridas de validación:",
    len(corridas_val)
)

print(
    "Variables de proceso:",
    len(feature_cols)
)

In [ ]:
from pathlib import Path
from shutil import copy2
import pyreadr

archivo_ff_drive = (
    DATA_DIR /
    "TEP_FaultFree_Training.RData"
)

archivo_ff_local = Path(
    "/content/TEP_FaultFree_Training.RData"
)

if not archivo_ff_local.exists():

    copy2(
        archivo_ff_drive,
        archivo_ff_local
    )

resultado_ff = pyreadr.read_r(
    str(archivo_ff_local)
)

ff_train = resultado_ff[
    "fault_free_training"
].copy()

del resultado_ff
gc.collect()

print(
    "Dimensiones:",
    ff_train.shape
)

In [ ]:
df_train_if = (
    ff_train[
        ff_train["simulationRun"].astype(int)
        .isin(corridas_train)
    ]
    .copy()
    .sort_values(
        ["simulationRun", "sample"]
    )
    .reset_index(drop=True)
)

df_val_if = (
    ff_train[
        ff_train["simulationRun"].astype(int)
        .isin(corridas_val)
    ]
    .copy()
    .sort_values(
        ["simulationRun", "sample"]
    )
    .reset_index(drop=True)
)

print(
    "Entrenamiento:",
    df_train_if.shape
)

print(
    "Validación:",
    df_val_if.shape
)

In [ ]:
corridas_compartidas_if = (
    set(
        df_train_if["simulationRun"]
        .astype(int)
    )
    .intersection(
        set(
            df_val_if["simulationRun"]
            .astype(int)
        )
    )
)

print(
    "Corridas compartidas:",
    corridas_compartidas_if
)

In [ ]:
X_train_if = (
    df_train_if[feature_cols]
    .copy()
)

X_val_if = (
    df_val_if[feature_cols]
    .copy()
)

print(
    "X_train_if:",
    X_train_if.shape
)

print(
    "X_val_if:",
    X_val_if.shape
)

In [ ]:
from sklearn.ensemble import IsolationForest

if_model = IsolationForest(
    n_estimators=300,
    max_samples=2048,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

if_model.fit(
    X_train_if
)

print(
    "Isolation Forest entrenado."
)

In [ ]:
score_if_train = (
    -if_model.score_samples(
        X_train_if
    )
)

score_if_val = (
    -if_model.score_samples(
        X_val_if
    )
)

print(
    "Scores entrenamiento:",
    score_if_train.shape
)

print(
    "Scores validación:",
    score_if_val.shape
)

In [ ]:
resumen_scores_if = pd.DataFrame({
    "Conjunto": [
        "Entrenamiento",
        "Validación"
    ],

    "Media": [
        score_if_train.mean(),
        score_if_val.mean()
    ],

    "Mediana": [
        np.median(score_if_train),
        np.median(score_if_val)
    ],

    "P95": [
        np.quantile(
            score_if_train,
            0.95
        ),
        np.quantile(
            score_if_val,
            0.95
        )
    ],

    "P99": [
        np.quantile(
            score_if_train,
            0.99
        ),
        np.quantile(
            score_if_val,
            0.99
        )
    ],

    "Máximo": [
        score_if_train.max(),
        score_if_val.max()
    ]
})

display(
    resumen_scores_if
)

In [ ]:
# FAR puntual del PCA en validación normal
far_objetivo = 0.0199

# Percentil equivalente para Isolation Forest
quantile_if = 1 - far_objetivo

IF_threshold = np.quantile(
    score_if_val,
    quantile_if
)

print(
    f"FAR objetivo: {far_objetivo:.4%}"
)

print(
    f"Percentil utilizado: {quantile_if:.4%}"
)

print(
    f"Umbral Isolation Forest: {IF_threshold:.6f}"
)

In [ ]:
alarm_if_train = (
    score_if_train > IF_threshold
)

alarm_if_val = (
    score_if_val > IF_threshold
)

In [ ]:
far_if_train = alarm_if_train.mean()
far_if_val = alarm_if_val.mean()

resumen_far_if = pd.DataFrame({
    "Conjunto": [
        "Entrenamiento",
        "Validación"
    ],
    "FAR_puntual": [
        far_if_train,
        far_if_val
    ],
    "Muestras_con_alarma": [
        alarm_if_train.sum(),
        alarm_if_val.sum()
    ],
    "Muestras_totales": [
        len(alarm_if_train),
        len(alarm_if_val)
    ]
})

display(resumen_far_if)

In [ ]:
resultados_if_val = (
    df_val_if[
        ["simulationRun", "sample"]
    ]
    .copy()
    .reset_index(drop=True)
)

resultados_if_val["score_if"] = score_if_val
resultados_if_val["alarma_puntual"] = alarm_if_val

resultados_if_val = (
    resultados_if_val
    .sort_values(
        ["simulationRun", "sample"]
    )
    .reset_index(drop=True)
)

display(resultados_if_val.head())

In [ ]:
ventana = 5
minimo_alarmas = 3

resultados_if_val["alarmas_en_ultimas_5"] = (
    resultados_if_val
    .groupby("simulationRun")["alarma_puntual"]
    .transform(
        lambda serie: (
            serie.astype(int)
            .rolling(
                window=ventana,
                min_periods=ventana
            )
            .sum()
        )
    )
)

resultados_if_val["alarma_persistente_3de5"] = (
    resultados_if_val["alarmas_en_ultimas_5"]
    >= minimo_alarmas
)

In [ ]:
far_if_puntual = (
    resultados_if_val["alarma_puntual"]
    .mean()
)

far_if_persistente = (
    resultados_if_val["alarma_persistente_3de5"]
    .mean()
)

reduccion_if = (
    1 - far_if_persistente / far_if_puntual
)

resumen_persistencia_if = pd.DataFrame({
    "Indicador": [
        "FAR puntual",
        "FAR con regla 3 de 5",
        "Reducción relativa de falsas alarmas"
    ],
    "Valor": [
        far_if_puntual,
        far_if_persistente,
        reduccion_if
    ]
})

display(resumen_persistencia_if)

In [ ]:
inicio_evento_if = (
    resultados_if_val
    .groupby("simulationRun")["alarma_persistente_3de5"]
    .transform(
        lambda serie: (
            serie
            & ~serie.shift(fill_value=False)
        )
    )
)

resultados_if_val["inicio_evento"] = inicio_evento_if

resultados_if_val["id_evento"] = (
    inicio_evento_if
    .groupby(
        resultados_if_val["simulationRun"]
    )
    .cumsum()
)

resultados_if_val.loc[
    ~resultados_if_val["alarma_persistente_3de5"],
    "id_evento"
] = pd.NA

In [ ]:
episodios_falsos_if = (
    resultados_if_val
    .dropna(subset=["id_evento"])
    .groupby(
        ["simulationRun", "id_evento"],
        as_index=False
    )
    .agg(
        muestra_inicio=("sample", "min"),
        muestra_fin=("sample", "max"),
        duracion_muestras=(
            "alarma_persistente_3de5",
            "size"
        ),
        score_maximo=("score_if", "max")
    )
)

episodios_falsos_if["duracion_minutos"] = (
    episodios_falsos_if["duracion_muestras"] * 3
)

In [ ]:
horas_normales_if = (
    len(resultados_if_val) * 3 / 60
)

numero_episodios_if = len(
    episodios_falsos_if
)

corridas_con_evento_if = (
    episodios_falsos_if[
        "simulationRun"
    ].nunique()
)

resumen_eventos_if = pd.DataFrame({
    "Indicador": [
        "Episodios falsos totales",
        "Corridas con al menos un episodio",
        "Duración media de episodio (min)",
        "Duración mediana de episodio (min)",
        "Duración máxima de episodio (min)",
        "Episodios por 100 horas normales"
    ],
    "Valor": [
        numero_episodios_if,
        corridas_con_evento_if,
        episodios_falsos_if[
            "duracion_minutos"
        ].mean(),
        episodios_falsos_if[
            "duracion_minutos"
        ].median(),
        episodios_falsos_if[
            "duracion_minutos"
        ].max(),
        numero_episodios_if
        / horas_normales_if
        * 100
    ]
})

display(resumen_eventos_if)

In [ ]:
import joblib

checkpoint_if = {
    "if_model": if_model,
    "feature_cols": list(feature_cols),
    "IF_threshold": float(IF_threshold),
    "far_objetivo": float(far_objetivo),
    "ventana": 5,
    "minimo_alarmas": 3,
    "corridas_train": [int(x) for x in corridas_train],
    "corridas_val": [int(x) for x in corridas_val]
}

ruta_if = (
    MODELS_DIR /
    "isolation_forest_baseline_v1.joblib"
)

joblib.dump(
    checkpoint_if,
    ruta_if
)

print("Modelo guardado:", ruta_if)
print("¿Existe?:", ruta_if.exists())

In [ ]:
import gc

variables_a_eliminar = [
    "ff_train",
    "df_train_if",
    "df_val_if",
    "X_train_if",
    "X_val_if",
    "score_if_train",
    "score_if_val",
    "alarm_if_train",
    "alarm_if_val",
    "resultados_if_val"
]

for variable in variables_a_eliminar:
    globals().pop(variable, None)

gc.collect()

print("Memoria liberada.")

In [ ]:
from pathlib import Path
from shutil import copy2
import pyreadr

archivo_faulty_drive = (
    DATA_DIR /
    "TEP_Faulty_Training.RData"
)

archivo_faulty_local = Path(
    "/content/TEP_Faulty_Training.RData"
)

if not archivo_faulty_local.exists():
    print("Copiando archivo desde Drive...")
    copy2(
        archivo_faulty_drive,
        archivo_faulty_local
    )

print("Leyendo archivo...")

resultado_faulty = pyreadr.read_r(
    str(archivo_faulty_local)
)

f_train = resultado_faulty[
    "faulty_training"
].copy()

del resultado_faulty
gc.collect()

print("Dimensiones:", f_train.shape)

In [ ]:
#Función para evaluar fallas con Isolation Forest
def evaluar_falla_if(
    df_faulty,
    numero_falla,
    feature_cols,
    if_model,
    IF_threshold,
    ventana=5,
    minimo_alarmas=3,
    muestra_inicio_falla=21,
    minutos_por_muestra=3
):
    """
    Evalúa un tipo de falla del Tennessee Eastman Process
    mediante un Isolation Forest previamente entrenado
    exclusivamente con operación normal.
    """

    # ---------------------------------------------------------
    # 1. Seleccionar la falla
    # ---------------------------------------------------------
    df = (
        df_faulty.loc[
            df_faulty["faultNumber"].astype(int)
            == numero_falla
        ]
        .copy()
        .sort_values(
            ["simulationRun", "sample"]
        )
        .reset_index(drop=True)
    )

    if df.empty:
        raise ValueError(
            f"No hay datos para la falla {numero_falla}"
        )

    total_corridas = (
        df["simulationRun"].nunique()
    )

    # ---------------------------------------------------------
    # 2. Seleccionar variables en el mismo orden
    # ---------------------------------------------------------
    X = df.loc[:, feature_cols]

    # ---------------------------------------------------------
    # 3. Score de anomalía
    # ---------------------------------------------------------
    # sklearn: score alto = más normal.
    # Invertimos el signo:
    # score alto = más anómalo.
    score_if = -if_model.score_samples(X)

    # ---------------------------------------------------------
    # 4. Tabla temporal
    # ---------------------------------------------------------
    resultados = df[
        ["simulationRun", "sample"]
    ].copy()

    resultados["score_if"] = score_if

    resultados["alarma_puntual"] = (
        resultados["score_if"]
        > IF_threshold
    )

    # ---------------------------------------------------------
    # 5. Regla temporal
    # ---------------------------------------------------------
    resultados["alarmas_ventana"] = (
        resultados
        .groupby("simulationRun")[
            "alarma_puntual"
        ]
        .transform(
            lambda serie: (
                serie.astype(int)
                .rolling(
                    window=ventana,
                    min_periods=ventana
                )
                .sum()
            )
        )
    )

    resultados["alarma_persistente"] = (
        resultados["alarmas_ventana"]
        >= minimo_alarmas
    )

    # ---------------------------------------------------------
    # 6. Inicio de episodios
    # ---------------------------------------------------------
    resultados["inicio_episodio"] = (
        resultados
        .groupby("simulationRun")[
            "alarma_persistente"
        ]
        .transform(
            lambda serie: (
                serie
                & ~serie.shift(fill_value=False)
            )
        )
    )

    # ---------------------------------------------------------
    # 7. Falsas alarmas previas
    # ---------------------------------------------------------
    mascara_prealarma = (
        resultados["inicio_episodio"]
        & (
            resultados["sample"]
            < muestra_inicio_falla
        )
    )

    corridas_prealarma = (
        resultados.loc[
            mascara_prealarma,
            "simulationRun"
        ]
        .nunique()
    )

    # ---------------------------------------------------------
    # 8. Primer episodio posterior a la falla
    # ---------------------------------------------------------
    mascara_deteccion = (
        resultados["inicio_episodio"]
        & (
            resultados["sample"]
            >= muestra_inicio_falla
        )
    )

    primeras_detecciones = (
        resultados.loc[
            mascara_deteccion,
            ["simulationRun", "sample"]
        ]
        .groupby(
            "simulationRun",
            as_index=False
        )
        .agg(
            muestra_deteccion=("sample", "min")
        )
    )

    corridas_detectadas = len(
        primeras_detecciones
    )

    # Corridas sin alarma previa: evaluables
    corridas_evaluables = (
        total_corridas
        - corridas_prealarma
    )

    tasa_bruta = (
        corridas_detectadas
        / total_corridas
    )

    tasa_evaluable = (
        corridas_detectadas
        / corridas_evaluables
        if corridas_evaluables > 0
        else np.nan
    )

    # ---------------------------------------------------------
    # 9. Retraso
    # ---------------------------------------------------------
    primeras_detecciones[
        "retraso_minutos"
    ] = (
        primeras_detecciones[
            "muestra_deteccion"
        ]
        - muestra_inicio_falla
    ) * minutos_por_muestra

    if corridas_detectadas > 0:

        retraso_medio = (
            primeras_detecciones[
                "retraso_minutos"
            ].mean()
        )

        retraso_mediano = (
            primeras_detecciones[
                "retraso_minutos"
            ].median()
        )

        retraso_p95 = (
            primeras_detecciones[
                "retraso_minutos"
            ].quantile(0.95)
        )

        retraso_maximo = (
            primeras_detecciones[
                "retraso_minutos"
            ].max()
        )

    else:
        retraso_medio = np.nan
        retraso_mediano = np.nan
        retraso_p95 = np.nan
        retraso_maximo = np.nan

    # ---------------------------------------------------------
    # 10. Proporción de puntos post-falla señalados
    # ---------------------------------------------------------
    post_falla = (
        resultados["sample"]
        >= muestra_inicio_falla
    )

    porcentaje_alarmas_puntuales_post = (
        resultados.loc[
            post_falla,
            "alarma_puntual"
        ].mean() * 100
    )

    # ---------------------------------------------------------
    # 11. Resumen
    # ---------------------------------------------------------
    resumen = {
        "falla": int(numero_falla),
        "corridas_totales": int(total_corridas),
        "corridas_prealarma": int(corridas_prealarma),
        "corridas_evaluables": int(corridas_evaluables),
        "corridas_detectadas": int(corridas_detectadas),

        "tasa_deteccion_bruta_pct":
            float(tasa_bruta * 100),

        "tasa_deteccion_evaluable_pct":
            float(tasa_evaluable * 100),

        "alarma_puntual_post_pct":
            float(porcentaje_alarmas_puntuales_post),

        "retraso_medio_min":
            float(retraso_medio)
            if not np.isnan(retraso_medio)
            else np.nan,

        "retraso_mediano_min":
            float(retraso_mediano)
            if not np.isnan(retraso_mediano)
            else np.nan,

        "retraso_p95_min":
            float(retraso_p95)
            if not np.isnan(retraso_p95)
            else np.nan,

        "retraso_maximo_min":
            float(retraso_maximo)
            if not np.isnan(retraso_maximo)
            else np.nan
    }

    del X
    del score_if

    gc.collect()

    return resumen

In [ ]:
#Evaluación fallas 3, 9 y 15
resultados_if_dificiles = []

for numero_falla in [3, 9, 15]:

    print(
        f"Evaluando falla {numero_falla}..."
    )

    resultado = evaluar_falla_if(
        df_faulty=f_train,
        numero_falla=numero_falla,
        feature_cols=feature_cols,
        if_model=if_model,
        IF_threshold=IF_threshold,
        ventana=5,
        minimo_alarmas=3,
        muestra_inicio_falla=21,
        minutos_por_muestra=3
    )

    resultados_if_dificiles.append(
        resultado
    )

tabla_if_dificiles = pd.DataFrame(
    resultados_if_dificiles
)

display(
    tabla_if_dificiles.round(3)
)

In [ ]:
resultados_if_20_fallas = []

for numero_falla in range(1, 21):

    print(f"Evaluando falla {numero_falla} de 20...")

    resultado = evaluar_falla_if(
        df_faulty=f_train,
        numero_falla=numero_falla,
        feature_cols=feature_cols,
        if_model=if_model,
        IF_threshold=IF_threshold,
        ventana=5,
        minimo_alarmas=3,
        muestra_inicio_falla=21,
        minutos_por_muestra=3
    )

    resultados_if_20_fallas.append(resultado)

    gc.collect()

print("\nEvaluación terminada.")

In [ ]:
tabla_resultados_if = pd.DataFrame(
    resultados_if_20_fallas
)

display(
    tabla_resultados_if.round(3)
)

In [ ]:
print(
    "Fallas evaluadas:",
    len(tabla_resultados_if)
)

print(
    "Corridas totales:",
    tabla_resultados_if[
        "corridas_totales"
    ].sum()
)

In [ ]:
tabla_resultados_if.to_csv(
    RESULTS_DIR /
    "resultados_IsolationForest_20_fallas_training.csv",
    index=False
)

print("Resultados de Isolation Forest guardados.")

In [ ]:
import gc
import numpy as np
import pandas as pd


def evaluar_falla_pca(
    df_faulty,
    numero_falla,
    feature_cols,
    scaler,
    pca_model,
    T2_threshold,
    SPE_threshold,
    ventana=5,
    minimo_alarmas=3,
    muestra_inicio_falla=21,
    minutos_por_muestra=3
):
    """
    Evalúa un tipo de falla del Tennessee Eastman Process mediante
    un modelo PCA previamente entrenado exclusivamente con datos
    correspondientes a operación normal.

    La función calcula:
    - estadístico Hotelling T²
    - estadístico SPE/Q
    - alarmas puntuales
    - alarmas persistentes mediante regla k-de-n
    - falsas alarmas previas a la introducción de la falla
    - tasa bruta de detección
    - tasa de detección sobre corridas evaluables
    - retraso de detección

    Parameters
    ----------
    df_faulty : pd.DataFrame
        Dataset que contiene las corridas con fallas.

    numero_falla : int
        Número de falla que se desea evaluar.

    feature_cols : list
        Lista de las 52 variables de proceso utilizadas por el modelo.

    scaler : StandardScaler
        Escalador entrenado exclusivamente con datos normales.

    pca_model : PCA
        Modelo PCA entrenado exclusivamente con datos normales.

    T2_threshold : float
        Umbral de alarma para Hotelling T².

    SPE_threshold : float
        Umbral de alarma para SPE/Q.

    ventana : int, default=5
        Número de muestras incluidas en la ventana temporal.

    minimo_alarmas : int, default=3
        Número mínimo de alarmas puntuales requerido dentro de la ventana.

    muestra_inicio_falla : int, default=21
        Primera muestra posterior a la introducción de la falla.

    minutos_por_muestra : int, default=3
        Intervalo de muestreo del Tennessee Eastman Process.

    Returns
    -------
    dict
        Diccionario con las métricas resumidas de la falla evaluada.
    """

    # =========================================================
    # 1. Determinar el orden exacto de las variables del modelo
    # =========================================================

    if hasattr(scaler, "feature_names_in_"):
        columnas_modelo = list(scaler.feature_names_in_)
    else:
        columnas_modelo = list(feature_cols)

    # Verificar que todas las columnas requeridas estén presentes
    columnas_faltantes = [
        col
        for col in columnas_modelo
        if col not in df_faulty.columns
    ]

    if columnas_faltantes:
        raise ValueError(
            "Faltan variables requeridas por el modelo: "
            f"{columnas_faltantes}"
        )

    # =========================================================
    # 2. Seleccionar únicamente las corridas de la falla
    # =========================================================

    df = (
        df_faulty.loc[
            df_faulty["faultNumber"].astype(int)
            == int(numero_falla)
        ]
        .copy()
        .sort_values(
            ["simulationRun", "sample"]
        )
        .reset_index(drop=True)
    )

    if df.empty:
        raise ValueError(
            f"No se encontraron datos para la falla {numero_falla}."
        )

    total_corridas = int(
        df["simulationRun"].nunique()
    )

    # =========================================================
    # 3. Seleccionar las 52 variables de proceso
    # =========================================================

    X = df.loc[
        :,
        columnas_modelo
    ].copy()

    # =========================================================
    # 4. Estandarizar usando SOLO los parámetros del entrenamiento
    # =========================================================
    #
    # Se utiliza transform() y NO fit_transform(), porque las
    # medias y desviaciones deben seguir siendo exclusivamente
    # las aprendidas de la operación normal.
    # =========================================================

    X_scaled_array = scaler.transform(X)

    # Recuperar los nombres para mantener trazabilidad
    X_scaled = pd.DataFrame(
        X_scaled_array,
        columns=columnas_modelo,
        index=X.index
    )

    # =========================================================
    # 5. Proyección en el espacio PCA normal
    # =========================================================

    scores = pca_model.transform(
        X_scaled
    )

    eigenvalues = (
        pca_model.explained_variance_
    )

    # =========================================================
    # 6. Calcular Hotelling T²
    # =========================================================

    T2 = np.sum(
        (scores ** 2)
        / eigenvalues,
        axis=1
    )

    # =========================================================
    # 7. Reconstrucción PCA y cálculo de SPE/Q
    # =========================================================

    X_reconstructed = (
        pca_model.inverse_transform(
            scores
        )
    )

    residuals = (
        X_scaled.to_numpy()
        - X_reconstructed
    )

    SPE = np.sum(
        residuals ** 2,
        axis=1
    )

    # =========================================================
    # 8. Construir tabla temporal de resultados
    # =========================================================

    resultados = df[
        [
            "simulationRun",
            "sample"
        ]
    ].copy()

    resultados["T2"] = T2
    resultados["SPE"] = SPE

    # =========================================================
    # 9. Alarmas puntuales
    # =========================================================

    resultados["alarma_T2"] = (
        resultados["T2"]
        > T2_threshold
    )

    resultados["alarma_SPE"] = (
        resultados["SPE"]
        > SPE_threshold
    )

    resultados["alarma_puntual"] = (
        resultados["alarma_T2"]
        |
        resultados["alarma_SPE"]
    )

    # =========================================================
    # 10. Regla temporal de persistencia
    # =========================================================
    #
    # Por defecto:
    # al menos 3 alarmas dentro de las últimas 5 muestras.
    #
    # El cálculo se realiza de manera independiente por corrida.
    # =========================================================

    resultados["alarmas_ventana"] = (
        resultados
        .groupby(
            "simulationRun"
        )["alarma_puntual"]
        .transform(
            lambda serie: (
                serie
                .astype(int)
                .rolling(
                    window=ventana,
                    min_periods=ventana
                )
                .sum()
            )
        )
    )

    resultados["alarma_persistente"] = (
        resultados["alarmas_ventana"]
        >= minimo_alarmas
    )

    # =========================================================
    # 11. Identificar el inicio de cada episodio persistente
    # =========================================================

    resultados["inicio_episodio"] = (
        resultados
        .groupby(
            "simulationRun"
        )["alarma_persistente"]
        .transform(
            lambda serie: (
                serie
                &
                ~serie.shift(
                    fill_value=False
                )
            )
        )
    )

    # =========================================================
    # 12. Identificar corridas con falsa alarma previa
    # =========================================================
    #
    # Una corrida queda "contaminada" para el cálculo de la tasa
    # evaluable si ya tenía una alarma persistente antes de la
    # introducción de la falla.
    # =========================================================

    mascara_falsa_previa = (
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            < muestra_inicio_falla
        )
    )

    corridas_con_prealarma = set(
        resultados.loc[
            mascara_falsa_previa,
            "simulationRun"
        ].unique()
    )

    corridas_falsa_alarma_previa = len(
        corridas_con_prealarma
    )

    porcentaje_falsa_alarma_previa = (
        corridas_falsa_alarma_previa
        / total_corridas
    )

    # =========================================================
    # 13. Detectar el primer episodio NUEVO posterior a la falla
    # =========================================================

    mascara_deteccion = (
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            >= muestra_inicio_falla
        )
    )

    primeras_detecciones = (
        resultados.loc[
            mascara_deteccion,
            [
                "simulationRun",
                "sample"
            ]
        ]
        .groupby(
            "simulationRun",
            as_index=False
        )
        .agg(
            muestra_deteccion=(
                "sample",
                "min"
            )
        )
    )

    corridas_detectadas_set = set(
        primeras_detecciones[
            "simulationRun"
        ].unique()
    )

    corridas_detectadas = len(
        corridas_detectadas_set
    )

    # =========================================================
    # 14. Determinar corridas realmente evaluables
    # =========================================================

    todas_las_corridas = set(
        resultados[
            "simulationRun"
        ].unique()
    )

    # Excluir las corridas que ya estaban en alarma antes
    corridas_evaluables_set = (
        todas_las_corridas
        - corridas_con_prealarma
    )

    # Solo cuentan como detecciones evaluables aquellas que:
    # 1. fueron detectadas
    # 2. no tenían prealarma
    corridas_detectadas_evaluables_set = (
        corridas_detectadas_set
        & corridas_evaluables_set
    )

    corridas_evaluables = len(
        corridas_evaluables_set
    )

    corridas_detectadas_evaluables = len(
        corridas_detectadas_evaluables_set
    )

    # =========================================================
    # 15. Tasas de detección
    # =========================================================

    tasa_deteccion_bruta = (
        corridas_detectadas
        / total_corridas
    )

    tasa_deteccion_evaluable = (
        corridas_detectadas_evaluables
        / corridas_evaluables
        if corridas_evaluables > 0
        else np.nan
    )

    # =========================================================
    # 16. Retraso de detección
    # =========================================================

    primeras_detecciones[
        "retraso_muestras"
    ] = (
        primeras_detecciones[
            "muestra_deteccion"
        ]
        - muestra_inicio_falla
    )

    primeras_detecciones[
        "retraso_minutos"
    ] = (
        primeras_detecciones[
            "retraso_muestras"
        ]
        * minutos_por_muestra
    )

    if corridas_detectadas > 0:

        retraso_medio = (
            primeras_detecciones[
                "retraso_minutos"
            ].mean()
        )

        retraso_mediano = (
            primeras_detecciones[
                "retraso_minutos"
            ].median()
        )

        retraso_p95 = (
            primeras_detecciones[
                "retraso_minutos"
            ].quantile(0.95)
        )

        retraso_minimo = (
            primeras_detecciones[
                "retraso_minutos"
            ].min()
        )

        retraso_maximo = (
            primeras_detecciones[
                "retraso_minutos"
            ].max()
        )

    else:

        retraso_medio = np.nan
        retraso_mediano = np.nan
        retraso_p95 = np.nan
        retraso_minimo = np.nan
        retraso_maximo = np.nan

    # =========================================================
    # 17. Porcentaje de muestras post-falla con alarma puntual
    # =========================================================

    mascara_post_falla = (
        resultados["sample"]
        >= muestra_inicio_falla
    )

    porcentaje_alarmas_puntuales_post = (
        resultados.loc[
            mascara_post_falla,
            "alarma_puntual"
        ]
        .mean()
        * 100
    )

    porcentaje_T2_post = (
        resultados.loc[
            mascara_post_falla,
            "alarma_T2"
        ]
        .mean()
        * 100
    )

    porcentaje_SPE_post = (
        resultados.loc[
            mascara_post_falla,
            "alarma_SPE"
        ]
        .mean()
        * 100
    )

    # =========================================================
    # 18. Construir resumen final
    # =========================================================

    resumen = {

        "falla": int(
            numero_falla
        ),

        "corridas_totales": int(
            total_corridas
        ),

        "corridas_falsa_alarma_previa": int(
            corridas_falsa_alarma_previa
        ),

        "porcentaje_falsa_alarma_previa": float(
            porcentaje_falsa_alarma_previa
        ),

        "corridas_evaluables": int(
            corridas_evaluables
        ),

        "corridas_detectadas": int(
            corridas_detectadas
        ),

        "corridas_detectadas_evaluables": int(
            corridas_detectadas_evaluables
        ),

        "tasa_deteccion_bruta_pct": float(
            tasa_deteccion_bruta * 100
        ),

        "tasa_deteccion_evaluable_pct": float(
            tasa_deteccion_evaluable * 100
        ) if not np.isnan(
            tasa_deteccion_evaluable
        ) else np.nan,

        "alarma_puntual_post_pct": float(
            porcentaje_alarmas_puntuales_post
        ),

        "alarma_T2_post_pct": float(
            porcentaje_T2_post
        ),

        "alarma_SPE_post_pct": float(
            porcentaje_SPE_post
        ),

        "retraso_medio_min": float(
            retraso_medio
        ) if not np.isnan(
            retraso_medio
        ) else np.nan,

        "retraso_mediano_min": float(
            retraso_mediano
        ) if not np.isnan(
            retraso_mediano
        ) else np.nan,

        "retraso_p95_min": float(
            retraso_p95
        ) if not np.isnan(
            retraso_p95
        ) else np.nan,

        "retraso_minimo_min": float(
            retraso_minimo
        ) if not np.isnan(
            retraso_minimo
        ) else np.nan,

        "retraso_maximo_min": float(
            retraso_maximo
        ) if not np.isnan(
            retraso_maximo
        ) else np.nan
    }

    # =========================================================
    # 19. Liberar memoria
    # =========================================================

    del X
    del X_scaled_array
    del X_scaled
    del scores
    del X_reconstructed
    del residuals

    gc.collect()

    return resumen

In [ ]:
prueba_pca_corregida = evaluar_falla_pca(
    df_faulty=f_train,
    numero_falla=1,
    feature_cols=feature_cols,
    scaler=scaler,
    pca_model=pca_model,
    T2_threshold=T2_threshold,
    SPE_threshold=SPE_threshold,
    ventana=5,
    minimo_alarmas=3,
    muestra_inicio_falla=21,
    minutos_por_muestra=3
)

display(
    pd.DataFrame(
        [prueba_pca_corregida]
    ).round(3)
)

In [ ]:
import gc
import numpy as np
import pandas as pd


def evaluar_falla_if(
    df_faulty,
    numero_falla,
    feature_cols,
    if_model,
    IF_threshold,
    ventana=5,
    minimo_alarmas=3,
    muestra_inicio_falla=21,
    minutos_por_muestra=3
):
    """
    Evalúa un tipo de falla del Tennessee Eastman Process mediante
    un modelo Isolation Forest entrenado exclusivamente con datos
    correspondientes a operación normal.

    La función calcula:
    - score de anomalía de Isolation Forest
    - alarmas puntuales
    - alarmas persistentes mediante regla k-de-n
    - falsas alarmas previas a la introducción de la falla
    - tasa bruta de detección
    - tasa de detección sobre corridas evaluables
    - retraso de detección

    Parameters
    ----------
    df_faulty : pd.DataFrame
        Dataset que contiene las corridas con fallas.

    numero_falla : int
        Número de falla que se desea evaluar.

    feature_cols : list
        Lista de las 52 variables de proceso.

    if_model : IsolationForest
        Modelo previamente entrenado exclusivamente con datos normales.

    IF_threshold : float
        Umbral de anomalía definido a partir de validación normal.

    ventana : int, default=5
        Número de muestras incluidas en la ventana temporal.

    minimo_alarmas : int, default=3
        Número mínimo de alarmas puntuales requerido dentro de la ventana.

    muestra_inicio_falla : int, default=21
        Primera muestra posterior a la introducción de la falla.

    minutos_por_muestra : int, default=3
        Intervalo de muestreo del Tennessee Eastman Process.

    Returns
    -------
    dict
        Diccionario con las métricas resumidas de la falla evaluada.
    """

    # =========================================================
    # 1. Determinar el orden exacto de las variables
    # =========================================================

    if hasattr(if_model, "feature_names_in_"):
        columnas_modelo = list(if_model.feature_names_in_)
    else:
        columnas_modelo = list(feature_cols)

    # Comprobar que todas las variables utilizadas por el modelo
    # estén presentes en el nuevo dataset
    columnas_faltantes = [
        col
        for col in columnas_modelo
        if col not in df_faulty.columns
    ]

    if columnas_faltantes:
        raise ValueError(
            "Faltan variables requeridas por Isolation Forest: "
            f"{columnas_faltantes}"
        )

    # =========================================================
    # 2. Seleccionar únicamente las corridas de la falla
    # =========================================================

    df = (
        df_faulty.loc[
            df_faulty["faultNumber"].astype(int)
            == int(numero_falla)
        ]
        .copy()
        .sort_values(
            ["simulationRun", "sample"]
        )
        .reset_index(drop=True)
    )

    if df.empty:
        raise ValueError(
            f"No se encontraron datos para la falla {numero_falla}."
        )

    total_corridas = int(
        df["simulationRun"].nunique()
    )

    # =========================================================
    # 3. Seleccionar las 52 variables de proceso
    # =========================================================

    X = df.loc[
        :,
        columnas_modelo
    ].copy()

    # =========================================================
    # 4. Calcular el score de anomalía
    # =========================================================
    #
    # score_samples() de sklearn:
    # valores mayores = observaciones más normales.
    #
    # Invertimos el signo para trabajar con:
    # valores mayores = observaciones más anómalas.
    # =========================================================

    score_if = (
        -if_model.score_samples(X)
    )

    # =========================================================
    # 5. Construir tabla temporal
    # =========================================================

    resultados = df[
        [
            "simulationRun",
            "sample"
        ]
    ].copy()

    resultados["score_if"] = score_if

    # =========================================================
    # 6. Alarmas puntuales
    # =========================================================

    resultados["alarma_puntual"] = (
        resultados["score_if"]
        > IF_threshold
    )

    # =========================================================
    # 7. Regla temporal de persistencia
    # =========================================================
    #
    # Por defecto:
    # al menos 3 alarmas dentro de las últimas 5 muestras.
    #
    # El groupby garantiza que nunca se mezclen corridas.
    # =========================================================

    resultados["alarmas_ventana"] = (
        resultados
        .groupby(
            "simulationRun"
        )["alarma_puntual"]
        .transform(
            lambda serie: (
                serie
                .astype(int)
                .rolling(
                    window=ventana,
                    min_periods=ventana
                )
                .sum()
            )
        )
    )

    resultados["alarma_persistente"] = (
        resultados["alarmas_ventana"]
        >= minimo_alarmas
    )

    # =========================================================
    # 8. Identificar el inicio de cada episodio persistente
    # =========================================================

    resultados["inicio_episodio"] = (
        resultados
        .groupby(
            "simulationRun"
        )["alarma_persistente"]
        .transform(
            lambda serie: (
                serie
                &
                ~serie.shift(
                    fill_value=False
                )
            )
        )
    )

    # =========================================================
    # 9. Identificar corridas con falsa alarma previa
    # =========================================================

    mascara_falsa_previa = (
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            < muestra_inicio_falla
        )
    )

    corridas_con_prealarma = set(
        resultados.loc[
            mascara_falsa_previa,
            "simulationRun"
        ].unique()
    )

    corridas_falsa_alarma_previa = len(
        corridas_con_prealarma
    )

    porcentaje_falsa_alarma_previa = (
        corridas_falsa_alarma_previa
        / total_corridas
    )

    # =========================================================
    # 10. Detectar primeros episodios nuevos post-falla
    # =========================================================

    mascara_deteccion = (
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            >= muestra_inicio_falla
        )
    )

    primeras_detecciones = (
        resultados.loc[
            mascara_deteccion,
            [
                "simulationRun",
                "sample"
            ]
        ]
        .groupby(
            "simulationRun",
            as_index=False
        )
        .agg(
            muestra_deteccion=(
                "sample",
                "min"
            )
        )
    )

    corridas_detectadas_set = set(
        primeras_detecciones[
            "simulationRun"
        ].unique()
    )

    corridas_detectadas = len(
        corridas_detectadas_set
    )

    # =========================================================
    # 11. Determinar corridas realmente evaluables
    # =========================================================
    #
    # Se excluyen del denominador las corridas que ya tenían
    # alarma persistente antes de introducir la falla.
    # =========================================================

    todas_las_corridas = set(
        resultados[
            "simulationRun"
        ].unique()
    )

    corridas_evaluables_set = (
        todas_las_corridas
        - corridas_con_prealarma
    )

    # Una detección evaluable debe cumplir simultáneamente:
    # - existir una detección post-falla
    # - la corrida no tenía una prealarma
    corridas_detectadas_evaluables_set = (
        corridas_detectadas_set
        & corridas_evaluables_set
    )

    corridas_evaluables = len(
        corridas_evaluables_set
    )

    corridas_detectadas_evaluables = len(
        corridas_detectadas_evaluables_set
    )

    # =========================================================
    # 12. Tasas de detección
    # =========================================================

    tasa_deteccion_bruta = (
        corridas_detectadas
        / total_corridas
    )

    tasa_deteccion_evaluable = (
        corridas_detectadas_evaluables
        / corridas_evaluables
        if corridas_evaluables > 0
        else np.nan
    )

    # =========================================================
    # 13. Retrasos de detección
    # =========================================================

    primeras_detecciones[
        "retraso_muestras"
    ] = (
        primeras_detecciones[
            "muestra_deteccion"
        ]
        - muestra_inicio_falla
    )

    primeras_detecciones[
        "retraso_minutos"
    ] = (
        primeras_detecciones[
            "retraso_muestras"
        ]
        * minutos_por_muestra
    )

    if corridas_detectadas > 0:

        retraso_medio = (
            primeras_detecciones[
                "retraso_minutos"
            ].mean()
        )

        retraso_mediano = (
            primeras_detecciones[
                "retraso_minutos"
            ].median()
        )

        retraso_p95 = (
            primeras_detecciones[
                "retraso_minutos"
            ].quantile(0.95)
        )

        retraso_minimo = (
            primeras_detecciones[
                "retraso_minutos"
            ].min()
        )

        retraso_maximo = (
            primeras_detecciones[
                "retraso_minutos"
            ].max()
        )

    else:

        retraso_medio = np.nan
        retraso_mediano = np.nan
        retraso_p95 = np.nan
        retraso_minimo = np.nan
        retraso_maximo = np.nan

    # =========================================================
    # 14. Porcentaje de muestras post-falla con alarma puntual
    # =========================================================

    mascara_post_falla = (
        resultados["sample"]
        >= muestra_inicio_falla
    )

    porcentaje_alarmas_puntuales_post = (
        resultados.loc[
            mascara_post_falla,
            "alarma_puntual"
        ]
        .mean()
        * 100
    )

    # Score medio durante el periodo post-falla.
    # Puede ser útil posteriormente para comparar severidad.
    score_medio_post = (
        resultados.loc[
            mascara_post_falla,
            "score_if"
        ]
        .mean()
    )

    # =========================================================
    # 15. Construir resumen final
    # =========================================================

    resumen = {

        "falla": int(
            numero_falla
        ),

        "corridas_totales": int(
            total_corridas
        ),

        "corridas_falsa_alarma_previa": int(
            corridas_falsa_alarma_previa
        ),

        "porcentaje_falsa_alarma_previa": float(
            porcentaje_falsa_alarma_previa
        ),

        "corridas_evaluables": int(
            corridas_evaluables
        ),

        "corridas_detectadas": int(
            corridas_detectadas
        ),

        "corridas_detectadas_evaluables": int(
            corridas_detectadas_evaluables
        ),

        "tasa_deteccion_bruta_pct": float(
            tasa_deteccion_bruta * 100
        ),

        "tasa_deteccion_evaluable_pct": float(
            tasa_deteccion_evaluable * 100
        ) if not np.isnan(
            tasa_deteccion_evaluable
        ) else np.nan,

        "alarma_puntual_post_pct": float(
            porcentaje_alarmas_puntuales_post
        ),

        "score_if_medio_post": float(
            score_medio_post
        ),

        "retraso_medio_min": float(
            retraso_medio
        ) if not np.isnan(
            retraso_medio
        ) else np.nan,

        "retraso_mediano_min": float(
            retraso_mediano
        ) if not np.isnan(
            retraso_mediano
        ) else np.nan,

        "retraso_p95_min": float(
            retraso_p95
        ) if not np.isnan(
            retraso_p95
        ) else np.nan,

        "retraso_minimo_min": float(
            retraso_minimo
        ) if not np.isnan(
            retraso_minimo
        ) else np.nan,

        "retraso_maximo_min": float(
            retraso_maximo
        ) if not np.isnan(
            retraso_maximo
        ) else np.nan
    }

    # =========================================================
    # 16. Liberar memoria
    # =========================================================

    del X
    del score_if

    gc.collect()

    return resumen

In [ ]:
prueba_if_corregida = evaluar_falla_if(
    df_faulty=f_train,
    numero_falla=1,
    feature_cols=feature_cols,
    if_model=if_model,
    IF_threshold=IF_threshold,
    ventana=5,
    minimo_alarmas=3,
    muestra_inicio_falla=21,
    minutos_por_muestra=3
)

display(
    pd.DataFrame(
        [prueba_if_corregida]
    ).round(3)
)

In [ ]:
import gc
import numpy as np
import pandas as pd


def evaluar_falla_pca(
    df_faulty,
    numero_falla,
    feature_cols,
    scaler,
    pca_model,
    T2_threshold,
    SPE_threshold,
    ventana=5,
    minimo_alarmas=3,
    muestra_inicio_falla=21,
    minutos_por_muestra=3
):
    """
    Evalúa un tipo de falla del Tennessee Eastman Process mediante
    un modelo PCA previamente entrenado exclusivamente con datos
    de operación normal.

    Calcula:
    - Hotelling T²
    - SPE/Q
    - alarmas puntuales
    - alarmas persistentes mediante regla k-de-n
    - falsas alarmas previas a la falla
    - tasa bruta de detección
    - tasa de detección sobre corridas evaluables
    - retraso de detección sobre corridas evaluables
    """

    # =========================================================
    # 1. Orden exacto de las variables utilizadas por el modelo
    # =========================================================

    if hasattr(scaler, "feature_names_in_"):
        columnas_modelo = list(scaler.feature_names_in_)
    else:
        columnas_modelo = list(feature_cols)

    columnas_faltantes = [
        col for col in columnas_modelo
        if col not in df_faulty.columns
    ]

    if columnas_faltantes:
        raise ValueError(
            "Faltan variables requeridas por el modelo PCA: "
            f"{columnas_faltantes}"
        )

    # =========================================================
    # 2. Seleccionar únicamente la falla evaluada
    # =========================================================

    df = (
        df_faulty.loc[
            df_faulty["faultNumber"].astype(int)
            == int(numero_falla)
        ]
        .copy()
        .sort_values(
            ["simulationRun", "sample"]
        )
        .reset_index(drop=True)
    )

    if df.empty:
        raise ValueError(
            f"No se encontraron datos para la falla {numero_falla}."
        )

    total_corridas = int(
        df["simulationRun"].nunique()
    )

    # =========================================================
    # 3. Seleccionar las variables de proceso
    # =========================================================

    X = df.loc[
        :,
        columnas_modelo
    ].copy()

    # =========================================================
    # 4. Estandarizar usando únicamente el scaler normal
    # =========================================================

    X_scaled_array = scaler.transform(X)

    X_scaled = pd.DataFrame(
        X_scaled_array,
        columns=columnas_modelo,
        index=X.index
    )

    # =========================================================
    # 5. Proyección PCA
    # =========================================================

    scores = pca_model.transform(
        X_scaled
    )

    eigenvalues = (
        pca_model.explained_variance_
    )

    # =========================================================
    # 6. Hotelling T²
    # =========================================================

    T2 = np.sum(
        (scores ** 2) / eigenvalues,
        axis=1
    )

    # =========================================================
    # 7. SPE / Q
    # =========================================================

    X_reconstructed = (
        pca_model.inverse_transform(
            scores
        )
    )

    residuals = (
        X_scaled.to_numpy()
        - X_reconstructed
    )

    SPE = np.sum(
        residuals ** 2,
        axis=1
    )

    # =========================================================
    # 8. Tabla temporal de resultados
    # =========================================================

    resultados = df[
        [
            "simulationRun",
            "sample"
        ]
    ].copy()

    resultados["T2"] = T2
    resultados["SPE"] = SPE

    # =========================================================
    # 9. Alarmas puntuales
    # =========================================================

    resultados["alarma_T2"] = (
        resultados["T2"]
        > T2_threshold
    )

    resultados["alarma_SPE"] = (
        resultados["SPE"]
        > SPE_threshold
    )

    resultados["alarma_puntual"] = (
        resultados["alarma_T2"]
        |
        resultados["alarma_SPE"]
    )

    # =========================================================
    # 10. Regla temporal k-de-n
    # =========================================================

    resultados["alarmas_ventana"] = (
        resultados
        .groupby("simulationRun")[
            "alarma_puntual"
        ]
        .transform(
            lambda serie: (
                serie
                .astype(int)
                .rolling(
                    window=ventana,
                    min_periods=ventana
                )
                .sum()
            )
        )
    )

    resultados["alarma_persistente"] = (
        resultados["alarmas_ventana"]
        >= minimo_alarmas
    )

    # =========================================================
    # 11. Inicio de episodios persistentes
    # =========================================================

    resultados["inicio_episodio"] = (
        resultados
        .groupby("simulationRun")[
            "alarma_persistente"
        ]
        .transform(
            lambda serie: (
                serie
                &
                ~serie.shift(
                    fill_value=False
                )
            )
        )
    )

    # =========================================================
    # 12. Corridas con falsa alarma previa
    # =========================================================

    mascara_falsa_previa = (
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            < muestra_inicio_falla
        )
    )

    corridas_con_prealarma = set(
        resultados.loc[
            mascara_falsa_previa,
            "simulationRun"
        ].unique()
    )

    corridas_falsa_alarma_previa = len(
        corridas_con_prealarma
    )

    porcentaje_falsa_alarma_previa = (
        corridas_falsa_alarma_previa
        / total_corridas
    )

    # =========================================================
    # 13. Primer episodio nuevo posterior a la falla
    # =========================================================

    mascara_deteccion = (
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            >= muestra_inicio_falla
        )
    )

    primeras_detecciones = (
        resultados.loc[
            mascara_deteccion,
            [
                "simulationRun",
                "sample"
            ]
        ]
        .groupby(
            "simulationRun",
            as_index=False
        )
        .agg(
            muestra_deteccion=(
                "sample",
                "min"
            )
        )
    )

    corridas_detectadas_set = set(
        primeras_detecciones[
            "simulationRun"
        ].unique()
    )

    corridas_detectadas = len(
        corridas_detectadas_set
    )

    # =========================================================
    # 14. Corridas evaluables
    # =========================================================

    todas_las_corridas = set(
        resultados[
            "simulationRun"
        ].unique()
    )

    corridas_evaluables_set = (
        todas_las_corridas
        - corridas_con_prealarma
    )

    corridas_detectadas_evaluables_set = (
        corridas_detectadas_set
        & corridas_evaluables_set
    )

    corridas_evaluables = len(
        corridas_evaluables_set
    )

    corridas_detectadas_evaluables = len(
        corridas_detectadas_evaluables_set
    )

    # =========================================================
    # 15. Tasas de detección
    # =========================================================

    tasa_deteccion_bruta = (
        corridas_detectadas
        / total_corridas
    )

    tasa_deteccion_evaluable = (
        corridas_detectadas_evaluables
        / corridas_evaluables
        if corridas_evaluables > 0
        else np.nan
    )

    # =========================================================
    # 16. Detecciones válidas para calcular retrasos
    # =========================================================

    primeras_detecciones_evaluables = (
        primeras_detecciones[
            primeras_detecciones[
                "simulationRun"
            ].isin(
                corridas_evaluables_set
            )
        ]
        .copy()
    )

    # =========================================================
    # 17. Retraso de detección
    # =========================================================

    primeras_detecciones_evaluables[
        "retraso_muestras"
    ] = (
        primeras_detecciones_evaluables[
            "muestra_deteccion"
        ]
        - muestra_inicio_falla
    )

    primeras_detecciones_evaluables[
        "retraso_minutos"
    ] = (
        primeras_detecciones_evaluables[
            "retraso_muestras"
        ]
        * minutos_por_muestra
    )

    if corridas_detectadas_evaluables > 0:

        retraso_medio = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ].mean()
        )

        retraso_mediano = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ].median()
        )

        retraso_p95 = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ].quantile(0.95)
        )

        retraso_minimo = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ].min()
        )

        retraso_maximo = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ].max()
        )

    else:

        retraso_medio = np.nan
        retraso_mediano = np.nan
        retraso_p95 = np.nan
        retraso_minimo = np.nan
        retraso_maximo = np.nan

    # =========================================================
    # 18. Comportamiento puntual post-falla
    # =========================================================

    mascara_post_falla = (
        resultados["sample"]
        >= muestra_inicio_falla
    )

    porcentaje_alarmas_puntuales_post = (
        resultados.loc[
            mascara_post_falla,
            "alarma_puntual"
        ]
        .mean()
        * 100
    )

    porcentaje_T2_post = (
        resultados.loc[
            mascara_post_falla,
            "alarma_T2"
        ]
        .mean()
        * 100
    )

    porcentaje_SPE_post = (
        resultados.loc[
            mascara_post_falla,
            "alarma_SPE"
        ]
        .mean()
        * 100
    )

    # =========================================================
    # 19. Resumen final
    # =========================================================

    resumen = {

        "falla": int(
            numero_falla
        ),

        "corridas_totales": int(
            total_corridas
        ),

        "corridas_falsa_alarma_previa": int(
            corridas_falsa_alarma_previa
        ),

        "porcentaje_falsa_alarma_previa": float(
            porcentaje_falsa_alarma_previa
        ),

        "corridas_evaluables": int(
            corridas_evaluables
        ),

        "corridas_detectadas": int(
            corridas_detectadas
        ),

        "corridas_detectadas_evaluables": int(
            corridas_detectadas_evaluables
        ),

        "tasa_deteccion_bruta_pct": float(
            tasa_deteccion_bruta * 100
        ),

        "tasa_deteccion_evaluable_pct": (
            float(
                tasa_deteccion_evaluable * 100
            )
            if not np.isnan(
                tasa_deteccion_evaluable
            )
            else np.nan
        ),

        "alarma_puntual_post_pct": float(
            porcentaje_alarmas_puntuales_post
        ),

        "alarma_T2_post_pct": float(
            porcentaje_T2_post
        ),

        "alarma_SPE_post_pct": float(
            porcentaje_SPE_post
        ),

        "retraso_medio_min": (
            float(retraso_medio)
            if not np.isnan(retraso_medio)
            else np.nan
        ),

        "retraso_mediano_min": (
            float(retraso_mediano)
            if not np.isnan(retraso_mediano)
            else np.nan
        ),

        "retraso_p95_min": (
            float(retraso_p95)
            if not np.isnan(retraso_p95)
            else np.nan
        ),

        "retraso_minimo_min": (
            float(retraso_minimo)
            if not np.isnan(retraso_minimo)
            else np.nan
        ),

        "retraso_maximo_min": (
            float(retraso_maximo)
            if not np.isnan(retraso_maximo)
            else np.nan
        )
    }

    # =========================================================
    # 20. Liberar memoria
    # =========================================================

    del X
    del X_scaled_array
    del X_scaled
    del scores
    del X_reconstructed
    del residuals

    gc.collect()

    return resumen

In [ ]:
def evaluar_falla_if(
    df_faulty,
    numero_falla,
    feature_cols,
    if_model,
    IF_threshold,
    ventana=5,
    minimo_alarmas=3,
    muestra_inicio_falla=21,
    minutos_por_muestra=3
):
    """
    Evalúa un tipo de falla del Tennessee Eastman Process mediante
    un modelo Isolation Forest entrenado exclusivamente con datos
    correspondientes a operación normal.

    Calcula:
    - score de anomalía
    - alarmas puntuales
    - alarmas persistentes mediante regla k-de-n
    - falsas alarmas previas
    - tasa bruta de detección
    - tasa de detección sobre corridas evaluables
    - retraso de detección sobre corridas evaluables
    """

    # =========================================================
    # 1. Orden exacto de variables
    # =========================================================

    if hasattr(if_model, "feature_names_in_"):
        columnas_modelo = list(
            if_model.feature_names_in_
        )
    else:
        columnas_modelo = list(
            feature_cols
        )

    columnas_faltantes = [
        col for col in columnas_modelo
        if col not in df_faulty.columns
    ]

    if columnas_faltantes:
        raise ValueError(
            "Faltan variables requeridas por Isolation Forest: "
            f"{columnas_faltantes}"
        )

    # =========================================================
    # 2. Seleccionar únicamente la falla
    # =========================================================

    df = (
        df_faulty.loc[
            df_faulty["faultNumber"].astype(int)
            == int(numero_falla)
        ]
        .copy()
        .sort_values(
            ["simulationRun", "sample"]
        )
        .reset_index(drop=True)
    )

    if df.empty:
        raise ValueError(
            f"No se encontraron datos para la falla {numero_falla}."
        )

    total_corridas = int(
        df["simulationRun"].nunique()
    )

    # =========================================================
    # 3. Variables de proceso
    # =========================================================

    X = df.loc[
        :,
        columnas_modelo
    ].copy()

    # =========================================================
    # 4. Score de anomalía
    # =========================================================
    #
    # score_samples:
    # mayor valor = más normal.
    #
    # Se invierte el signo:
    # mayor valor = más anómalo.
    # =========================================================

    score_if = (
        -if_model.score_samples(X)
    )

    # =========================================================
    # 5. Tabla temporal
    # =========================================================

    resultados = df[
        [
            "simulationRun",
            "sample"
        ]
    ].copy()

    resultados["score_if"] = (
        score_if
    )

    # =========================================================
    # 6. Alarma puntual
    # =========================================================

    resultados["alarma_puntual"] = (
        resultados["score_if"]
        > IF_threshold
    )

    # =========================================================
    # 7. Regla temporal k-de-n
    # =========================================================

    resultados["alarmas_ventana"] = (
        resultados
        .groupby("simulationRun")[
            "alarma_puntual"
        ]
        .transform(
            lambda serie: (
                serie
                .astype(int)
                .rolling(
                    window=ventana,
                    min_periods=ventana
                )
                .sum()
            )
        )
    )

    resultados["alarma_persistente"] = (
        resultados["alarmas_ventana"]
        >= minimo_alarmas
    )

    # =========================================================
    # 8. Inicio de episodios
    # =========================================================

    resultados["inicio_episodio"] = (
        resultados
        .groupby("simulationRun")[
            "alarma_persistente"
        ]
        .transform(
            lambda serie: (
                serie
                &
                ~serie.shift(
                    fill_value=False
                )
            )
        )
    )

    # =========================================================
    # 9. Falsas alarmas previas
    # =========================================================

    mascara_falsa_previa = (
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            < muestra_inicio_falla
        )
    )

    corridas_con_prealarma = set(
        resultados.loc[
            mascara_falsa_previa,
            "simulationRun"
        ].unique()
    )

    corridas_falsa_alarma_previa = len(
        corridas_con_prealarma
    )

    porcentaje_falsa_alarma_previa = (
        corridas_falsa_alarma_previa
        / total_corridas
    )

    # =========================================================
    # 10. Detecciones posteriores a la falla
    # =========================================================

    mascara_deteccion = (
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            >= muestra_inicio_falla
        )
    )

    primeras_detecciones = (
        resultados.loc[
            mascara_deteccion,
            [
                "simulationRun",
                "sample"
            ]
        ]
        .groupby(
            "simulationRun",
            as_index=False
        )
        .agg(
            muestra_deteccion=(
                "sample",
                "min"
            )
        )
    )

    corridas_detectadas_set = set(
        primeras_detecciones[
            "simulationRun"
        ].unique()
    )

    corridas_detectadas = len(
        corridas_detectadas_set
    )

    # =========================================================
    # 11. Corridas evaluables
    # =========================================================

    todas_las_corridas = set(
        resultados[
            "simulationRun"
        ].unique()
    )

    corridas_evaluables_set = (
        todas_las_corridas
        - corridas_con_prealarma
    )

    corridas_detectadas_evaluables_set = (
        corridas_detectadas_set
        & corridas_evaluables_set
    )

    corridas_evaluables = len(
        corridas_evaluables_set
    )

    corridas_detectadas_evaluables = len(
        corridas_detectadas_evaluables_set
    )

    # =========================================================
    # 12. Tasas de detección
    # =========================================================

    tasa_deteccion_bruta = (
        corridas_detectadas
        / total_corridas
    )

    tasa_deteccion_evaluable = (
        corridas_detectadas_evaluables
        / corridas_evaluables
        if corridas_evaluables > 0
        else np.nan
    )

    # =========================================================
    # 13. Detecciones utilizadas para medir retraso
    # =========================================================

    primeras_detecciones_evaluables = (
        primeras_detecciones[
            primeras_detecciones[
                "simulationRun"
            ].isin(
                corridas_evaluables_set
            )
        ]
        .copy()
    )

    # =========================================================
    # 14. Retraso de detección
    # =========================================================

    primeras_detecciones_evaluables[
        "retraso_muestras"
    ] = (
        primeras_detecciones_evaluables[
            "muestra_deteccion"
        ]
        - muestra_inicio_falla
    )

    primeras_detecciones_evaluables[
        "retraso_minutos"
    ] = (
        primeras_detecciones_evaluables[
            "retraso_muestras"
        ]
        * minutos_por_muestra
    )

    if corridas_detectadas_evaluables > 0:

        retraso_medio = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ].mean()
        )

        retraso_mediano = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ].median()
        )

        retraso_p95 = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ].quantile(0.95)
        )

        retraso_minimo = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ].min()
        )

        retraso_maximo = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ].max()
        )

    else:

        retraso_medio = np.nan
        retraso_mediano = np.nan
        retraso_p95 = np.nan
        retraso_minimo = np.nan
        retraso_maximo = np.nan

    # =========================================================
    # 15. Comportamiento post-falla
    # =========================================================

    mascara_post_falla = (
        resultados["sample"]
        >= muestra_inicio_falla
    )

    porcentaje_alarmas_puntuales_post = (
        resultados.loc[
            mascara_post_falla,
            "alarma_puntual"
        ]
        .mean()
        * 100
    )

    score_medio_post = (
        resultados.loc[
            mascara_post_falla,
            "score_if"
        ]
        .mean()
    )

    # =========================================================
    # 16. Resumen final
    # =========================================================

    resumen = {

        "falla": int(
            numero_falla
        ),

        "corridas_totales": int(
            total_corridas
        ),

        "corridas_falsa_alarma_previa": int(
            corridas_falsa_alarma_previa
        ),

        "porcentaje_falsa_alarma_previa": float(
            porcentaje_falsa_alarma_previa
        ),

        "corridas_evaluables": int(
            corridas_evaluables
        ),

        "corridas_detectadas": int(
            corridas_detectadas
        ),

        "corridas_detectadas_evaluables": int(
            corridas_detectadas_evaluables
        ),

        "tasa_deteccion_bruta_pct": float(
            tasa_deteccion_bruta * 100
        ),

        "tasa_deteccion_evaluable_pct": (
            float(
                tasa_deteccion_evaluable * 100
            )
            if not np.isnan(
                tasa_deteccion_evaluable
            )
            else np.nan
        ),

        "alarma_puntual_post_pct": float(
            porcentaje_alarmas_puntuales_post
        ),

        "score_if_medio_post": float(
            score_medio_post
        ),

        "retraso_medio_min": (
            float(retraso_medio)
            if not np.isnan(retraso_medio)
            else np.nan
        ),

        "retraso_mediano_min": (
            float(retraso_mediano)
            if not np.isnan(retraso_mediano)
            else np.nan
        ),

        "retraso_p95_min": (
            float(retraso_p95)
            if not np.isnan(retraso_p95)
            else np.nan
        ),

        "retraso_minimo_min": (
            float(retraso_minimo)
            if not np.isnan(retraso_minimo)
            else np.nan
        ),

        "retraso_maximo_min": (
            float(retraso_maximo)
            if not np.isnan(retraso_maximo)
            else np.nan
        )
    }

    # =========================================================
    # 17. Liberar memoria
    # =========================================================

    del X
    del score_if

    gc.collect()

    return resumen

In [ ]:
prueba_pca_final = evaluar_falla_pca(
    df_faulty=f_train,
    numero_falla=1,
    feature_cols=feature_cols,
    scaler=scaler,
    pca_model=pca_model,
    T2_threshold=T2_threshold,
    SPE_threshold=SPE_threshold,
    ventana=5,
    minimo_alarmas=3,
    muestra_inicio_falla=21,
    minutos_por_muestra=3
)

display(
    pd.DataFrame([prueba_pca_final]).round(3)
)

In [ ]:
prueba_if_final = evaluar_falla_if(
    df_faulty=f_train,
    numero_falla=1,
    feature_cols=feature_cols,
    if_model=if_model,
    IF_threshold=IF_threshold,
    ventana=5,
    minimo_alarmas=3,
    muestra_inicio_falla=21,
    minutos_por_muestra=3
)

display(
    pd.DataFrame([prueba_if_final]).round(3)
)

In [ ]:
resultados_pca_finales = []

for numero_falla in range(1, 21):

    print(f"PCA | Evaluando falla {numero_falla} de 20...")

    resultado = evaluar_falla_pca(
        df_faulty=f_train,
        numero_falla=numero_falla,
        feature_cols=feature_cols,
        scaler=scaler,
        pca_model=pca_model,
        T2_threshold=T2_threshold,
        SPE_threshold=SPE_threshold,
        ventana=5,
        minimo_alarmas=3,
        muestra_inicio_falla=21,
        minutos_por_muestra=3
    )

    resultados_pca_finales.append(resultado)

    gc.collect()

tabla_pca_final = pd.DataFrame(
    resultados_pca_finales
)

print("\nPCA terminado.")

In [ ]:
columnas_resumen = [
    "falla",
    "corridas_totales",
    "corridas_falsa_alarma_previa",
    "corridas_evaluables",
    "corridas_detectadas_evaluables",
    "tasa_deteccion_evaluable_pct",
    "alarma_puntual_post_pct",
    "retraso_medio_min",
    "retraso_mediano_min",
    "retraso_p95_min",
    "retraso_maximo_min"
]

display(
    tabla_pca_final[
        columnas_resumen
    ].round(3)
)

In [ ]:
resultados_if_finales = []

for numero_falla in range(1, 21):

    print(
        f"Isolation Forest | "
        f"Evaluando falla {numero_falla} de 20..."
    )

    resultado = evaluar_falla_if(
        df_faulty=f_train,
        numero_falla=numero_falla,
        feature_cols=feature_cols,
        if_model=if_model,
        IF_threshold=IF_threshold,
        ventana=5,
        minimo_alarmas=3,
        muestra_inicio_falla=21,
        minutos_por_muestra=3
    )

    resultados_if_finales.append(resultado)

    gc.collect()

tabla_if_final = pd.DataFrame(
    resultados_if_finales
)

print("\nIsolation Forest terminado.")

In [ ]:
display(
    tabla_if_final[
        columnas_resumen
    ].round(3)
)

In [ ]:
tabla_pca_final.to_csv(
    RESULTS_DIR /
    "PCA_20_fallas_metricas_finales.csv",
    index=False
)

tabla_if_final.to_csv(
    RESULTS_DIR /
    "IsolationForest_20_fallas_metricas_finales.csv",
    index=False
)

print("Resultados finales guardados en Google Drive.")

In [ ]:
comparacion_modelos = pd.merge(
    tabla_pca_final[
        [
            "falla",
            "corridas_evaluables",
            "corridas_detectadas_evaluables",
            "tasa_deteccion_evaluable_pct",
            "alarma_puntual_post_pct",
            "retraso_medio_min",
            "retraso_mediano_min",
            "retraso_p95_min",
            "retraso_maximo_min"
        ]
    ],
    tabla_if_final[
        [
            "falla",
            "corridas_evaluables",
            "corridas_detectadas_evaluables",
            "tasa_deteccion_evaluable_pct",
            "alarma_puntual_post_pct",
            "retraso_medio_min",
            "retraso_mediano_min",
            "retraso_p95_min",
            "retraso_maximo_min"
        ]
    ],
    on="falla",
    suffixes=("_PCA", "_IF")
)

display(
    comparacion_modelos.round(3)
)

In [ ]:
comparacion_modelos[
    "diferencia_deteccion_IF_menos_PCA"
] = (
    comparacion_modelos[
        "tasa_deteccion_evaluable_pct_IF"
    ]
    -
    comparacion_modelos[
        "tasa_deteccion_evaluable_pct_PCA"
    ]
)

comparacion_modelos[
    "diferencia_retraso_IF_menos_PCA"
] = (
    comparacion_modelos[
        "retraso_medio_min_IF"
    ]
    -
    comparacion_modelos[
        "retraso_medio_min_PCA"
    ]
)

In [ ]:
tabla_comparativa_simple = comparacion_modelos[
    [
        "falla",
        "tasa_deteccion_evaluable_pct_PCA",
        "tasa_deteccion_evaluable_pct_IF",
        "diferencia_deteccion_IF_menos_PCA",
        "retraso_medio_min_PCA",
        "retraso_medio_min_IF",
        "diferencia_retraso_IF_menos_PCA"
    ]
].copy()

display(
    tabla_comparativa_simple.round(3)
)

In [ ]:
comparacion_modelos.to_csv(
    RESULTS_DIR /
    "Comparacion_PCA_vs_IsolationForest.csv",
    index=False
)

print("Comparación guardada.")

In [ ]:
comparacion_final = pd.DataFrame({
    "falla": tabla_pca_final["falla"],

    "deteccion_PCA_pct":
        tabla_pca_final["tasa_deteccion_evaluable_pct"],

    "deteccion_IF_pct":
        tabla_if_final["tasa_deteccion_evaluable_pct"],

    "retraso_PCA_min":
        tabla_pca_final["retraso_medio_min"],

    "retraso_IF_min":
        tabla_if_final["retraso_medio_min"]
})

comparacion_final["delta_deteccion_IF_PCA"] = (
    comparacion_final["deteccion_IF_pct"]
    - comparacion_final["deteccion_PCA_pct"]
)

comparacion_final["delta_retraso_IF_PCA"] = (
    comparacion_final["retraso_IF_min"]
    - comparacion_final["retraso_PCA_min"]
)

display(comparacion_final.round(3))

In [ ]:
resumen_comparacion = pd.DataFrame({
    "Indicador": [
        "Tasa media de detección PCA (%)",
        "Tasa media de detección IF (%)",
        "Mediana de detección PCA (%)",
        "Mediana de detección IF (%)",
        "Retraso medio PCA (min)",
        "Retraso medio IF (min)",
        "Mediana del retraso PCA (min)",
        "Mediana del retraso IF (min)",
        "Fallas donde IF mejora detección",
        "Fallas donde PCA mejora detección",
        "Fallas con igual detección",
        "Fallas donde IF es más rápido",
        "Fallas donde PCA es más rápido"
    ],
    "Valor": [
        comparacion_final["deteccion_PCA_pct"].mean(),
        comparacion_final["deteccion_IF_pct"].mean(),

        comparacion_final["deteccion_PCA_pct"].median(),
        comparacion_final["deteccion_IF_pct"].median(),

        comparacion_final["retraso_PCA_min"].mean(),
        comparacion_final["retraso_IF_min"].mean(),

        comparacion_final["retraso_PCA_min"].median(),
        comparacion_final["retraso_IF_min"].median(),

        (comparacion_final["delta_deteccion_IF_PCA"] > 0).sum(),
        (comparacion_final["delta_deteccion_IF_PCA"] < 0).sum(),
        (comparacion_final["delta_deteccion_IF_PCA"] == 0).sum(),

        (comparacion_final["delta_retraso_IF_PCA"] < 0).sum(),
        (comparacion_final["delta_retraso_IF_PCA"] > 0).sum()
    ]
})

display(resumen_comparacion.round(3))

## 8. Autoencoder

Entrenamiento del autoencoder denso, calibración del error de reconstrucción, persistencia temporal y evaluación por falla.


In [ ]:
#Construcción de un tercer modelo: Autoencoder
import gc
import psutil

globals().pop("f_train", None)

gc.collect()

memoria = psutil.virtual_memory()

print(
    f"Memoria disponible: "
    f"{memoria.available / (1024 ** 3):.2f} GB"
)

print(
    f"Memoria utilizada: "
    f"{memoria.percent:.1f} %"
)

In [ ]:
from pathlib import Path
from shutil import copy2
import pyreadr

archivo_ff_drive = (
    DATA_DIR /
    "TEP_FaultFree_Training.RData"
)

archivo_ff_local = Path(
    "/content/TEP_FaultFree_Training.RData"
)

if not archivo_ff_local.exists():
    copy2(
        archivo_ff_drive,
        archivo_ff_local
    )

resultado_ff = pyreadr.read_r(
    str(archivo_ff_local)
)

ff_train = resultado_ff[
    "fault_free_training"
].copy()

del resultado_ff
gc.collect()

print("Dimensiones:", ff_train.shape)

In [ ]:
import joblib

checkpoint_pca = joblib.load(
    MODELS_DIR /
    "pca_baseline_v1.joblib"
)

corridas_train = checkpoint_pca[
    "corridas_train"
]

corridas_val = checkpoint_pca[
    "corridas_val"
]

feature_cols = checkpoint_pca[
    "feature_cols"
]

scaler = checkpoint_pca[
    "scaler"
]

print(
    "Corridas entrenamiento:",
    len(corridas_train)
)

print(
    "Corridas validación:",
    len(corridas_val)
)

print(
    "Variables de proceso:",
    len(feature_cols)
)

In [ ]:
df_train_ae = (
    ff_train.loc[
        ff_train["simulationRun"]
        .astype(int)
        .isin(corridas_train)
    ]
    .copy()
    .sort_values(
        ["simulationRun", "sample"]
    )
    .reset_index(drop=True)
)

df_val_ae = (
    ff_train.loc[
        ff_train["simulationRun"]
        .astype(int)
        .isin(corridas_val)
    ]
    .copy()
    .sort_values(
        ["simulationRun", "sample"]
    )
    .reset_index(drop=True)
)

print(
    "Entrenamiento:",
    df_train_ae.shape
)

print(
    "Validación:",
    df_val_ae.shape
)

In [ ]:
import tensorflow as tf
import numpy as np
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(
    "TensorFlow:",
    tf.__version__
)

In [ ]:
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    Dense
)

n_features = len(
    feature_cols
)

entrada = Input(
    shape=(n_features,),
    name="entrada"
)

# Encoder
x = Dense(
    32,
    activation="relu",
    name="encoder_32"
)(entrada)

x = Dense(
    16,
    activation="relu",
    name="encoder_16"
)(x)

latent = Dense(
    8,
    activation="relu",
    name="espacio_latente"
)(x)

# Decoder
x = Dense(
    16,
    activation="relu",
    name="decoder_16"
)(latent)

x = Dense(
    32,
    activation="relu",
    name="decoder_32"
)(x)

salida = Dense(
    n_features,
    activation="linear",
    name="reconstruccion"
)(x)

autoencoder = Model(
    inputs=entrada,
    outputs=salida,
    name="autoencoder_TEP"
)

autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

autoencoder.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [ ]:
# Seleccionar únicamente las 52 variables de proceso
X_train_ae = df_train_ae[feature_cols].copy()
X_val_ae = df_val_ae[feature_cols].copy()

# Aplicar el mismo escalador aprendido con datos normales
X_train_ae_scaled = scaler.transform(X_train_ae)
X_val_ae_scaled = scaler.transform(X_val_ae)

print("Train:", X_train_ae_scaled.shape)
print("Validation:", X_val_ae_scaled.shape)

In [ ]:
history_ae = autoencoder.fit(
    X_train_ae_scaled,
    X_train_ae_scaled,

    validation_data=(
        X_val_ae_scaled,
        X_val_ae_scaled
    ),

    epochs=50,
    batch_size=512,

    shuffle=True,

    callbacks=[
        early_stopping
    ],

    verbose=1
)

In [ ]:
print(
    "Épocas ejecutadas:",
    len(history_ae.history["loss"])
)

print(
    "Loss final entrenamiento:",
    history_ae.history["loss"][-1]
)

print(
    "Loss final validación:",
    history_ae.history["val_loss"][-1]
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.plot(
    history_ae.history["loss"],
    label="Entrenamiento"
)

plt.plot(
    history_ae.history["val_loss"],
    label="Validación"
)

plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Evolución de la pérdida del autoencoder")

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
mejor_epoca = (
    np.argmin(history_ae.history["val_loss"]) + 1
)

mejor_val_loss = (
    np.min(history_ae.history["val_loss"])
)

print("Mejor época:", mejor_epoca)
print("Mejor val_loss:", mejor_val_loss)

In [ ]:
ruta_autoencoder = (
    MODELS_DIR /
    "autoencoder_baseline_v1.keras"
)

autoencoder.save(
    ruta_autoencoder
)

print("Autoencoder guardado en:")
print(ruta_autoencoder)

print(
    "¿Existe el archivo?:",
    ruta_autoencoder.exists()
)

In [ ]:
X_train_ae_reconstructed = autoencoder.predict(
    X_train_ae_scaled,
    batch_size=2048,
    verbose=1
)

X_val_ae_reconstructed = autoencoder.predict(
    X_val_ae_scaled,
    batch_size=2048,
    verbose=1
)

print(
    "Reconstrucción train:",
    X_train_ae_reconstructed.shape
)

print(
    "Reconstrucción validation:",
    X_val_ae_reconstructed.shape
)

In [ ]:
score_ae_train = np.mean(
    (
        X_train_ae_scaled
        - X_train_ae_reconstructed
    ) ** 2,
    axis=1
)

score_ae_val = np.mean(
    (
        X_val_ae_scaled
        - X_val_ae_reconstructed
    ) ** 2,
    axis=1
)

print(
    "Scores entrenamiento:",
    score_ae_train.shape
)

print(
    "Scores validación:",
    score_ae_val.shape
)

In [ ]:
resumen_scores_ae = pd.DataFrame({
    "Conjunto": [
        "Entrenamiento",
        "Validación"
    ],

    "Media": [
        score_ae_train.mean(),
        score_ae_val.mean()
    ],

    "Mediana": [
        np.median(score_ae_train),
        np.median(score_ae_val)
    ],

    "P95": [
        np.quantile(score_ae_train, 0.95),
        np.quantile(score_ae_val, 0.95)
    ],

    "P99": [
        np.quantile(score_ae_train, 0.99),
        np.quantile(score_ae_val, 0.99)
    ],

    "Máximo": [
        score_ae_train.max(),
        score_ae_val.max()
    ]
})

display(
    resumen_scores_ae
)

In [ ]:
far_objetivo = 0.0199

quantile_ae = 1 - far_objetivo

AE_threshold = np.quantile(
    score_ae_val,
    quantile_ae
)

print(
    f"FAR objetivo: {far_objetivo:.4%}"
)

print(
    f"Percentil utilizado: {quantile_ae:.4%}"
)

print(
    f"Umbral Autoencoder: {AE_threshold:.6f}"
)

In [ ]:
alarm_ae_train = (
    score_ae_train > AE_threshold
)

alarm_ae_val = (
    score_ae_val > AE_threshold
)

far_ae_train = (
    alarm_ae_train.mean()
)

far_ae_val = (
    alarm_ae_val.mean()
)

resumen_far_ae = pd.DataFrame({
    "Conjunto": [
        "Entrenamiento",
        "Validación"
    ],
    "FAR_puntual": [
        far_ae_train,
        far_ae_val
    ],
    "Muestras_con_alarma": [
        alarm_ae_train.sum(),
        alarm_ae_val.sum()
    ],
    "Muestras_totales": [
        len(alarm_ae_train),
        len(alarm_ae_val)
    ]
})

display(resumen_far_ae)

In [ ]:
resultados_ae_val = (
    df_val_ae[
        [
            "simulationRun",
            "sample"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

resultados_ae_val["score_ae"] = (
    score_ae_val
)

resultados_ae_val[
    "alarma_puntual"
] = alarm_ae_val

In [ ]:
ventana = 5
minimo_alarmas = 3

resultados_ae_val[
    "alarmas_en_ultimas_5"
] = (
    resultados_ae_val
    .groupby("simulationRun")[
        "alarma_puntual"
    ]
    .transform(
        lambda serie: (
            serie
            .astype(int)
            .rolling(
                window=ventana,
                min_periods=ventana
            )
            .sum()
        )
    )
)

resultados_ae_val[
    "alarma_persistente_3de5"
] = (
    resultados_ae_val[
        "alarmas_en_ultimas_5"
    ]
    >= minimo_alarmas
)

In [ ]:
far_ae_puntual = (
    resultados_ae_val[
        "alarma_puntual"
    ].mean()
)

far_ae_persistente = (
    resultados_ae_val[
        "alarma_persistente_3de5"
    ].mean()
)

reduccion_ae = (
    1
    - far_ae_persistente
    / far_ae_puntual
)

resumen_persistencia_ae = pd.DataFrame({
    "Indicador": [
        "FAR puntual",
        "FAR con regla 3 de 5",
        "Reducción relativa de falsas alarmas"
    ],
    "Valor": [
        far_ae_puntual,
        far_ae_persistente,
        reduccion_ae
    ]
})

display(
    resumen_persistencia_ae
)

In [ ]:
inicio_evento_ae = (
    resultados_ae_val
    .groupby("simulationRun")[
        "alarma_persistente_3de5"
    ]
    .transform(
        lambda serie: (
            serie
            &
            ~serie.shift(
                fill_value=False
            )
        )
    )
)

resultados_ae_val[
    "inicio_evento"
] = inicio_evento_ae

resultados_ae_val[
    "id_evento"
] = (
    inicio_evento_ae
    .groupby(
        resultados_ae_val[
            "simulationRun"
        ]
    )
    .cumsum()
)

resultados_ae_val.loc[
    ~resultados_ae_val[
        "alarma_persistente_3de5"
    ],
    "id_evento"
] = pd.NA

In [ ]:
episodios_falsos_ae = (
    resultados_ae_val
    .dropna(
        subset=["id_evento"]
    )
    .groupby(
        [
            "simulationRun",
            "id_evento"
        ],
        as_index=False
    )
    .agg(
        muestra_inicio=(
            "sample",
            "min"
        ),
        muestra_fin=(
            "sample",
            "max"
        ),
        duracion_muestras=(
            "alarma_persistente_3de5",
            "size"
        ),
        score_maximo=(
            "score_ae",
            "max"
        )
    )
)

episodios_falsos_ae[
    "duracion_minutos"
] = (
    episodios_falsos_ae[
        "duracion_muestras"
    ]
    * 3
)

In [ ]:
horas_normales_ae = (
    len(resultados_ae_val)
    * 3
    / 60
)

numero_episodios_ae = (
    len(episodios_falsos_ae)
)

corridas_con_evento_ae = (
    episodios_falsos_ae[
        "simulationRun"
    ].nunique()
)

resumen_eventos_ae = pd.DataFrame({
    "Indicador": [
        "Episodios falsos totales",
        "Corridas con al menos un episodio",
        "Duración media de episodio (min)",
        "Duración mediana de episodio (min)",
        "Duración máxima de episodio (min)",
        "Episodios por 100 horas normales"
    ],
    "Valor": [
        numero_episodios_ae,
        corridas_con_evento_ae,
        episodios_falsos_ae[
            "duracion_minutos"
        ].mean(),
        episodios_falsos_ae[
            "duracion_minutos"
        ].median(),
        episodios_falsos_ae[
            "duracion_minutos"
        ].max(),
        numero_episodios_ae
        / horas_normales_ae
        * 100
    ]
})

display(
    resumen_eventos_ae
)

In [ ]:
import joblib

checkpoint_ae = {
    "feature_cols": list(feature_cols),
    "AE_threshold": float(AE_threshold),
    "far_objetivo": float(far_objetivo),
    "ventana": 5,
    "minimo_alarmas": 3,
    "corridas_train": [int(x) for x in corridas_train],
    "corridas_val": [int(x) for x in corridas_val]
}

ruta_checkpoint_ae = (
    MODELS_DIR /
    "autoencoder_baseline_v1_config.joblib"
)

joblib.dump(
    checkpoint_ae,
    ruta_checkpoint_ae
)

print("Configuración del autoencoder guardada.")

In [ ]:
import gc

variables_a_eliminar = [
    "ff_train",
    "df_train_ae",
    "df_val_ae",
    "X_train_ae",
    "X_val_ae",
    "X_train_ae_scaled",
    "X_val_ae_scaled",
    "X_train_ae_reconstructed",
    "X_val_ae_reconstructed",
    "score_ae_train",
    "score_ae_val",
    "alarm_ae_train",
    "alarm_ae_val",
    "resultados_ae_val"
]

for variable in variables_a_eliminar:
    globals().pop(variable, None)

gc.collect()

print("Memoria liberada.")

In [ ]:
from pathlib import Path
from shutil import copy2
import pyreadr

archivo_faulty_drive = (
    DATA_DIR /
    "TEP_Faulty_Training.RData"
)

archivo_faulty_local = Path(
    "/content/TEP_Faulty_Training.RData"
)

if not archivo_faulty_local.exists():

    print("Copiando desde Google Drive...")

    copy2(
        archivo_faulty_drive,
        archivo_faulty_local
    )

print("Leyendo Faulty Training...")

resultado_faulty = pyreadr.read_r(
    str(archivo_faulty_local)
)

f_train = resultado_faulty[
    "faulty_training"
].copy()

del resultado_faulty
gc.collect()

print("Dimensiones:", f_train.shape)

In [ ]:
def evaluar_falla_ae(
    df_faulty,
    numero_falla,
    feature_cols,
    scaler,
    autoencoder,
    AE_threshold,
    ventana=5,
    minimo_alarmas=3,
    muestra_inicio_falla=21,
    minutos_por_muestra=3,
    batch_size=2048
):
    """
    Evalúa un tipo de falla del Tennessee Eastman Process
    mediante un autoencoder entrenado exclusivamente con
    operación normal.

    Calcula:
    - error de reconstrucción por muestra
    - alarmas puntuales
    - alarmas persistentes k-de-n
    - prealarmas
    - corridas evaluables
    - tasa de detección
    - retraso de detección
    """

    # =========================================================
    # 1. Orden de las variables
    # =========================================================

    columnas_modelo = list(feature_cols)

    columnas_faltantes = [
        col for col in columnas_modelo
        if col not in df_faulty.columns
    ]

    if columnas_faltantes:
        raise ValueError(
            "Faltan variables requeridas por el autoencoder: "
            f"{columnas_faltantes}"
        )

    # =========================================================
    # 2. Seleccionar la falla
    # =========================================================

    df = (
        df_faulty.loc[
            df_faulty["faultNumber"].astype(int)
            == int(numero_falla)
        ]
        .copy()
        .sort_values(
            ["simulationRun", "sample"]
        )
        .reset_index(drop=True)
    )

    if df.empty:
        raise ValueError(
            f"No se encontraron datos para la falla {numero_falla}."
        )

    total_corridas = int(
        df["simulationRun"].nunique()
    )

    # =========================================================
    # 3. Variables de proceso y escalado
    # =========================================================

    X = df[
        columnas_modelo
    ].copy()

    X_scaled = scaler.transform(
        X
    )

    # =========================================================
    # 4. Reconstrucción
    # =========================================================

    X_reconstructed = autoencoder.predict(
        X_scaled,
        batch_size=batch_size,
        verbose=0
    )

    # =========================================================
    # 5. Score de anomalía = MSE por observación
    # =========================================================

    score_ae = np.mean(
        (
            X_scaled
            - X_reconstructed
        ) ** 2,
        axis=1
    )

    # =========================================================
    # 6. Tabla temporal
    # =========================================================

    resultados = df[
        [
            "simulationRun",
            "sample"
        ]
    ].copy()

    resultados["score_ae"] = (
        score_ae
    )

    # =========================================================
    # 7. Alarma puntual
    # =========================================================

    resultados["alarma_puntual"] = (
        resultados["score_ae"]
        > AE_threshold
    )

    # =========================================================
    # 8. Regla temporal k-de-n
    # =========================================================

    resultados["alarmas_ventana"] = (
        resultados
        .groupby("simulationRun")[
            "alarma_puntual"
        ]
        .transform(
            lambda serie: (
                serie
                .astype(int)
                .rolling(
                    window=ventana,
                    min_periods=ventana
                )
                .sum()
            )
        )
    )

    resultados["alarma_persistente"] = (
        resultados["alarmas_ventana"]
        >= minimo_alarmas
    )

    # =========================================================
    # 9. Inicio de cada episodio
    # =========================================================

    resultados["inicio_episodio"] = (
        resultados
        .groupby("simulationRun")[
            "alarma_persistente"
        ]
        .transform(
            lambda serie: (
                serie
                &
                ~serie.shift(
                    fill_value=False
                )
            )
        )
    )

    # =========================================================
    # 10. Corridas con prealarma
    # =========================================================

    mascara_falsa_previa = (
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            < muestra_inicio_falla
        )
    )

    corridas_con_prealarma = set(
        resultados.loc[
            mascara_falsa_previa,
            "simulationRun"
        ].unique()
    )

    corridas_falsa_alarma_previa = len(
        corridas_con_prealarma
    )

    porcentaje_falsa_alarma_previa = (
        corridas_falsa_alarma_previa
        / total_corridas
    )

    # =========================================================
    # 11. Primer episodio post-falla
    # =========================================================

    mascara_deteccion = (
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            >= muestra_inicio_falla
        )
    )

    primeras_detecciones = (
        resultados.loc[
            mascara_deteccion,
            [
                "simulationRun",
                "sample"
            ]
        ]
        .groupby(
            "simulationRun",
            as_index=False
        )
        .agg(
            muestra_deteccion=(
                "sample",
                "min"
            )
        )
    )

    corridas_detectadas_set = set(
        primeras_detecciones[
            "simulationRun"
        ].unique()
    )

    corridas_detectadas = len(
        corridas_detectadas_set
    )

    # =========================================================
    # 12. Corridas evaluables
    # =========================================================

    todas_las_corridas = set(
        resultados[
            "simulationRun"
        ].unique()
    )

    corridas_evaluables_set = (
        todas_las_corridas
        - corridas_con_prealarma
    )

    corridas_detectadas_evaluables_set = (
        corridas_detectadas_set
        & corridas_evaluables_set
    )

    corridas_evaluables = len(
        corridas_evaluables_set
    )

    corridas_detectadas_evaluables = len(
        corridas_detectadas_evaluables_set
    )

    # =========================================================
    # 13. Tasas de detección
    # =========================================================

    tasa_deteccion_bruta = (
        corridas_detectadas
        / total_corridas
    )

    tasa_deteccion_evaluable = (
        corridas_detectadas_evaluables
        / corridas_evaluables
        if corridas_evaluables > 0
        else np.nan
    )

    # =========================================================
    # 14. Detecciones evaluables
    # =========================================================

    primeras_detecciones_evaluables = (
        primeras_detecciones[
            primeras_detecciones[
                "simulationRun"
            ].isin(
                corridas_evaluables_set
            )
        ]
        .copy()
    )

    # =========================================================
    # 15. Retraso
    # =========================================================

    primeras_detecciones_evaluables[
        "retraso_muestras"
    ] = (
        primeras_detecciones_evaluables[
            "muestra_deteccion"
        ]
        - muestra_inicio_falla
    )

    primeras_detecciones_evaluables[
        "retraso_minutos"
    ] = (
        primeras_detecciones_evaluables[
            "retraso_muestras"
        ]
        * minutos_por_muestra
    )

    if corridas_detectadas_evaluables > 0:

        retraso_medio = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ].mean()
        )

        retraso_mediano = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ].median()
        )

        retraso_p95 = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ].quantile(0.95)
        )

        retraso_minimo = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ].min()
        )

        retraso_maximo = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ].max()
        )

    else:

        retraso_medio = np.nan
        retraso_mediano = np.nan
        retraso_p95 = np.nan
        retraso_minimo = np.nan
        retraso_maximo = np.nan

    # =========================================================
    # 16. Comportamiento puntual post-falla
    # =========================================================

    mascara_post_falla = (
        resultados["sample"]
        >= muestra_inicio_falla
    )

    alarma_puntual_post_pct = (
        resultados.loc[
            mascara_post_falla,
            "alarma_puntual"
        ].mean()
        * 100
    )

    score_ae_medio_post = (
        resultados.loc[
            mascara_post_falla,
            "score_ae"
        ].mean()
    )

    # =========================================================
    # 17. Resumen
    # =========================================================

    resumen = {

        "falla": int(numero_falla),

        "corridas_totales":
            int(total_corridas),

        "corridas_falsa_alarma_previa":
            int(corridas_falsa_alarma_previa),

        "porcentaje_falsa_alarma_previa":
            float(porcentaje_falsa_alarma_previa),

        "corridas_evaluables":
            int(corridas_evaluables),

        "corridas_detectadas":
            int(corridas_detectadas),

        "corridas_detectadas_evaluables":
            int(corridas_detectadas_evaluables),

        "tasa_deteccion_bruta_pct":
            float(tasa_deteccion_bruta * 100),

        "tasa_deteccion_evaluable_pct": (
            float(tasa_deteccion_evaluable * 100)
            if not np.isnan(tasa_deteccion_evaluable)
            else np.nan
        ),

        "alarma_puntual_post_pct":
            float(alarma_puntual_post_pct),

        "score_ae_medio_post":
            float(score_ae_medio_post),

        "retraso_medio_min": (
            float(retraso_medio)
            if not np.isnan(retraso_medio)
            else np.nan
        ),

        "retraso_mediano_min": (
            float(retraso_mediano)
            if not np.isnan(retraso_mediano)
            else np.nan
        ),

        "retraso_p95_min": (
            float(retraso_p95)
            if not np.isnan(retraso_p95)
            else np.nan
        ),

        "retraso_minimo_min": (
            float(retraso_minimo)
            if not np.isnan(retraso_minimo)
            else np.nan
        ),

        "retraso_maximo_min": (
            float(retraso_maximo)
            if not np.isnan(retraso_maximo)
            else np.nan
        )
    }

    # =========================================================
    # 18. Liberar memoria
    # =========================================================

    del X
    del X_scaled
    del X_reconstructed
    del score_ae

    gc.collect()

    return resumen

In [ ]:
resultados_ae_dificiles = []

for numero_falla in [3, 9, 15]:

    print(
        f"Autoencoder | "
        f"Evaluando falla {numero_falla}..."
    )

    resultado = evaluar_falla_ae(
        df_faulty=f_train,
        numero_falla=numero_falla,
        feature_cols=feature_cols,
        scaler=scaler,
        autoencoder=autoencoder,
        AE_threshold=AE_threshold,
        ventana=5,
        minimo_alarmas=3,
        muestra_inicio_falla=21,
        minutos_por_muestra=3
    )

    resultados_ae_dificiles.append(
        resultado
    )

    gc.collect()

tabla_ae_dificiles = pd.DataFrame(
    resultados_ae_dificiles
)

display(
    tabla_ae_dificiles.round(3)
)

In [ ]:
resultados_ae_finales = []

for numero_falla in range(1, 21):

    print(
        f"Autoencoder | Evaluando falla "
        f"{numero_falla} de 20..."
    )

    resultado = evaluar_falla_ae(
        df_faulty=f_train,
        numero_falla=numero_falla,
        feature_cols=feature_cols,
        scaler=scaler,
        autoencoder=autoencoder,
        AE_threshold=AE_threshold,
        ventana=5,
        minimo_alarmas=3,
        muestra_inicio_falla=21,
        minutos_por_muestra=3
    )

    resultados_ae_finales.append(
        resultado
    )

    gc.collect()

tabla_ae_final = pd.DataFrame(
    resultados_ae_finales
)

print("\nEvaluación del Autoencoder terminada.")

In [ ]:
columnas_resumen_ae = [
    "falla",
    "corridas_totales",
    "corridas_falsa_alarma_previa",
    "corridas_evaluables",
    "corridas_detectadas_evaluables",
    "tasa_deteccion_evaluable_pct",
    "alarma_puntual_post_pct",
    "retraso_medio_min",
    "retraso_mediano_min",
    "retraso_p95_min",
    "retraso_maximo_min"
]

display(
    tabla_ae_final[
        columnas_resumen_ae
    ].round(3)
)

In [ ]:
tabla_ae_final.to_csv(
    RESULTS_DIR /
    "Autoencoder_20_fallas_metricas_finales.csv",
    index=False
)

print(
    "Resultados del Autoencoder guardados."
)

In [ ]:
comparacion_3_modelos = (
    tabla_pca_final[
        [
            "falla",
            "tasa_deteccion_evaluable_pct",
            "retraso_medio_min",
            "retraso_mediano_min",
            "retraso_p95_min",
            "alarma_puntual_post_pct"
        ]
    ]
    .rename(
        columns={
            "tasa_deteccion_evaluable_pct": "deteccion_PCA_pct",
            "retraso_medio_min": "retraso_medio_PCA_min",
            "retraso_mediano_min": "retraso_mediano_PCA_min",
            "retraso_p95_min": "retraso_p95_PCA_min",
            "alarma_puntual_post_pct": "alarma_post_PCA_pct"
        }
    )
    .merge(
        tabla_if_final[
            [
                "falla",
                "tasa_deteccion_evaluable_pct",
                "retraso_medio_min",
                "retraso_mediano_min",
                "retraso_p95_min",
                "alarma_puntual_post_pct"
            ]
        ].rename(
            columns={
                "tasa_deteccion_evaluable_pct": "deteccion_IF_pct",
                "retraso_medio_min": "retraso_medio_IF_min",
                "retraso_mediano_min": "retraso_mediano_IF_min",
                "retraso_p95_min": "retraso_p95_IF_min",
                "alarma_puntual_post_pct": "alarma_post_IF_pct"
            }
        ),
        on="falla"
    )
    .merge(
        tabla_ae_final[
            [
                "falla",
                "tasa_deteccion_evaluable_pct",
                "retraso_medio_min",
                "retraso_mediano_min",
                "retraso_p95_min",
                "alarma_puntual_post_pct"
            ]
        ].rename(
            columns={
                "tasa_deteccion_evaluable_pct": "deteccion_AE_pct",
                "retraso_medio_min": "retraso_medio_AE_min",
                "retraso_mediano_min": "retraso_mediano_AE_min",
                "retraso_p95_min": "retraso_p95_AE_min",
                "alarma_puntual_post_pct": "alarma_post_AE_pct"
            }
        ),
        on="falla"
    )
)

display(
    comparacion_3_modelos.round(3)
)

In [ ]:
def resumen_global_modelo(tabla, nombre):

    return {
        "Modelo": nombre,

        "Detección media (%)":
            tabla["tasa_deteccion_evaluable_pct"].mean(),

        "Mediana detección (%)":
            tabla["tasa_deteccion_evaluable_pct"].median(),

        "Fallas con detección >= 99 %":
            (
                tabla["tasa_deteccion_evaluable_pct"] >= 99
            ).sum(),

        "Fallas con detección < 50 %":
            (
                tabla["tasa_deteccion_evaluable_pct"] < 50
            ).sum(),

        "Retraso medio entre fallas (min)":
            tabla["retraso_medio_min"].mean(),

        "Mediana del retraso (min)":
            tabla["retraso_medio_min"].median()
    }


resumen_global_3_modelos = pd.DataFrame(
    [
        resumen_global_modelo(
            tabla_pca_final,
            "PCA"
        ),

        resumen_global_modelo(
            tabla_if_final,
            "Isolation Forest"
        ),

        resumen_global_modelo(
            tabla_ae_final,
            "Autoencoder"
        )
    ]
)

display(
    resumen_global_3_modelos.round(3)
)

In [ ]:
comparacion_falsas_alarmas = pd.DataFrame({
    "Modelo": [
        "PCA",
        "Isolation Forest",
        "Autoencoder"
    ],

    "FAR_puntual_pct": [
        1.99,
        1.99,
        1.99
    ],

    "FAR_persistente_pct": [
        0.204,
        1.234,
        0.230
    ],

    "Episodios_falsos": [
        36,
        106,
        37
    ],

    "Corridas_normales_afectadas": [
        28,
        42,
        30
    ],

    "Episodios_por_100h": [
        1.44,
        4.24,
        1.48
    ],

    "Duracion_media_evento_min": [
        8.50,
        17.46,
        9.32
    ]
})

display(comparacion_falsas_alarmas)

In [ ]:
comparacion_3_modelos.to_csv(
    RESULTS_DIR /
    "comparacion_PCA_IF_Autoencoder_por_falla.csv",
    index=False
)

resumen_global_3_modelos.to_csv(
    RESULTS_DIR /
    "comparacion_global_PCA_IF_Autoencoder.csv",
    index=False
)

comparacion_falsas_alarmas.to_csv(
    RESULTS_DIR /
    "comparacion_falsas_alarmas_modelos.csv",
    index=False
)

print("Comparaciones guardadas correctamente.")

## 9. Interpretabilidad de los detectores

Contribuciones PCA, SHAP para Isolation Forest y error de reconstrucción por variable para el Autoencoder.


In [ ]:
# Seleccionar falla 1 - corrida 1
caso_pca = (
    f_train.loc[
        (f_train["faultNumber"].astype(int) == 1)
        &
        (f_train["simulationRun"].astype(int) == 1)
    ]
    .copy()
    .sort_values("sample")
    .reset_index(drop=True)
)

# Variables del proceso
X_caso = caso_pca[feature_cols].copy()

# Escalado con parámetros normales
X_caso_scaled_array = scaler.transform(X_caso)

X_caso_scaled = pd.DataFrame(
    X_caso_scaled_array,
    columns=feature_cols,
    index=X_caso.index
)

# Proyección PCA
scores_caso = pca_model.transform(
    X_caso_scaled
)

eigenvalues = pca_model.explained_variance_

# T²
T2_caso = np.sum(
    (scores_caso ** 2) / eigenvalues,
    axis=1
)

# Reconstrucción
X_caso_reconstructed = (
    pca_model.inverse_transform(
        scores_caso
    )
)

# Residuales
residuals_caso = (
    X_caso_scaled.to_numpy()
    - X_caso_reconstructed
)

# SPE
SPE_caso = np.sum(
    residuals_caso ** 2,
    axis=1
)

# Construir tabla temporal
detalle_caso_pca = caso_pca[
    ["sample"]
].copy()

detalle_caso_pca["T2"] = T2_caso
detalle_caso_pca["SPE"] = SPE_caso

detalle_caso_pca["alarma_puntual"] = (
    (detalle_caso_pca["T2"] > T2_threshold)
    |
    (detalle_caso_pca["SPE"] > SPE_threshold)
)

In [ ]:
detalle_caso_pca["alarmas_ventana"] = (
    detalle_caso_pca["alarma_puntual"]
    .astype(int)
    .rolling(
        window=5,
        min_periods=5
    )
    .sum()
)

detalle_caso_pca["alarma_persistente"] = (
    detalle_caso_pca["alarmas_ventana"]
    >= 3
)

detalle_caso_pca["inicio_episodio"] = (
    detalle_caso_pca["alarma_persistente"]
    &
    ~detalle_caso_pca[
        "alarma_persistente"
    ].shift(fill_value=False)
)

In [ ]:
detecciones_caso = detalle_caso_pca.loc[
    detalle_caso_pca["inicio_episodio"]
    &
    (detalle_caso_pca["sample"] >= 21)
]

display(detecciones_caso.head())

In [ ]:
muestra_interpretar = int(
    detecciones_caso.iloc[0]["sample"]
)

print(
    "Muestra seleccionada para interpretación:",
    muestra_interpretar
)

In [ ]:
def contribuciones_pca_muestra(
    df_original,
    muestra,
    feature_cols,
    scaler,
    pca_model
):
    """
    Calcula las contribuciones por variable a T² y SPE
    para una observación concreta.
    """

    # Seleccionar observación
    fila = (
        df_original.loc[
            df_original["sample"] == muestra,
            feature_cols
        ]
        .copy()
    )

    if len(fila) != 1:
        raise ValueError(
            "La muestra seleccionada no es única."
        )

    # Escalar
    x_scaled_array = scaler.transform(
        fila
    )

    x_scaled = pd.DataFrame(
        x_scaled_array,
        columns=feature_cols
    )

    x = x_scaled.to_numpy()[0]

    # -----------------------------------------------------
    # PCA scores
    # -----------------------------------------------------

    scores = pca_model.transform(
        x_scaled
    )

    t = scores[0]

    eigenvalues = (
        pca_model.explained_variance_
    )

    # -----------------------------------------------------
    # T² total
    # -----------------------------------------------------

    T2_total = np.sum(
        (t ** 2) / eigenvalues
    )

    # -----------------------------------------------------
    # Contribuciones a T²
    # -----------------------------------------------------
    #
    # P tiene dimensiones:
    # variables x componentes
    # -----------------------------------------------------

    P = pca_model.components_.T

    vector_t2 = (
        P @ (t / eigenvalues)
    )

    contrib_T2 = (
        x * vector_t2
    )

    # -----------------------------------------------------
    # Reconstrucción y SPE
    # -----------------------------------------------------

    reconstruccion = (
        pca_model.inverse_transform(
            scores
        )[0]
    )

    residual = (
        x - reconstruccion
    )

    contrib_SPE = (
        residual ** 2
    )

    SPE_total = np.sum(
        contrib_SPE
    )

    # -----------------------------------------------------
    # Tabla
    # -----------------------------------------------------

    tabla = pd.DataFrame({
        "variable": feature_cols,

        "contrib_T2": contrib_T2,

        "abs_contrib_T2":
            np.abs(contrib_T2),

        "contrib_SPE":
            contrib_SPE
    })

    return (
        tabla,
        T2_total,
        SPE_total
    )

In [ ]:
contribuciones_pca, T2_interpretado, SPE_interpretado = (
    contribuciones_pca_muestra(
        df_original=caso_pca,
        muestra=muestra_interpretar,
        feature_cols=feature_cols,
        scaler=scaler,
        pca_model=pca_model
    )
)

print(
    "T² de la muestra:",
    T2_interpretado
)

print(
    "Umbral T²:",
    T2_threshold
)

print(
    "\nSPE de la muestra:",
    SPE_interpretado
)

print(
    "Umbral SPE:",
    SPE_threshold
)

In [ ]:
print(
    "Suma contribuciones T²:",
    contribuciones_pca[
        "contrib_T2"
    ].sum()
)

print(
    "T² calculado:",
    T2_interpretado
)

print()

print(
    "Suma contribuciones SPE:",
    contribuciones_pca[
        "contrib_SPE"
    ].sum()
)

print(
    "SPE calculado:",
    SPE_interpretado
)

In [ ]:
top_SPE = (
    contribuciones_pca
    .sort_values(
        "contrib_SPE",
        ascending=False
    )
    .head(10)
)

display(
    top_SPE[
        [
            "variable",
            "contrib_SPE"
        ]
    ]
)

In [ ]:
top_T2 = (
    contribuciones_pca
    .sort_values(
        "abs_contrib_T2",
        ascending=False
    )
    .head(10)
)

display(
    top_T2[
        [
            "variable",
            "contrib_T2",
            "abs_contrib_T2"
        ]
    ]
)

In [ ]:
import matplotlib.pyplot as plt

top_SPE_plot = (
    top_SPE
    .sort_values(
        "contrib_SPE",
        ascending=True
    )
)

plt.figure(figsize=(9, 5))

plt.barh(
    top_SPE_plot["variable"],
    top_SPE_plot["contrib_SPE"]
)

plt.xlabel(
    "Contribución al SPE/Q"
)

plt.ylabel(
    "Variable"
)

plt.title(
    f"Principales contribuciones al SPE/Q "
    f"- Falla 1, muestra {muestra_interpretar}"
)

plt.tight_layout()
plt.show()

In [ ]:
top_T2_plot = (
    top_T2
    .sort_values(
        "abs_contrib_T2",
        ascending=True
    )
)

plt.figure(figsize=(9, 5))

plt.barh(
    top_T2_plot["variable"],
    top_T2_plot["contrib_T2"]
)

plt.xlabel(
    "Contribución a Hotelling T²"
)

plt.ylabel(
    "Variable"
)

plt.title(
    f"Principales contribuciones a T² "
    f"- Falla 1, muestra {muestra_interpretar}"
)

plt.tight_layout()
plt.show()

In [ ]:
nombres_variables = {
    "xmeas_2": "Caudal alimentación D",
    "xmeas_7": "Presión del reactor",
    "xmeas_13": "Presión del separador",
    "xmeas_16": "Presión del stripper",
    "xmeas_20": "Trabajo del compresor",
    "xmeas_21": "T salida agua enfriamiento reactor",
    "xmeas_23": "Componente A - alimentación reactor",
    "xmeas_29": "Componente A - gas de purga",
    "xmeas_31": "Componente C - gas de purga",
    "xmeas_33": "Componente E - gas de purga",
    "xmeas_39": "Componente F - producto",
    "xmv_2": "Válvula/caudal alimentación E",
    "xmv_5": "Válvula recirculación compresor",
    "xmv_10": "Agua de enfriamiento del reactor"
}

In [ ]:
top_SPE_plot = top_SPE.copy()

top_SPE_plot["nombre"] = (
    top_SPE_plot["variable"]
    .map(nombres_variables)
    .fillna(top_SPE_plot["variable"])
)

top_SPE_plot = (
    top_SPE_plot
    .sort_values("contrib_SPE")
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_SPE_plot["nombre"],
    top_SPE_plot["contrib_SPE"]
)

plt.xlabel("Contribución al SPE/Q")
plt.ylabel("Variable de proceso")

plt.title(
    f"Principales contribuciones al SPE/Q "
    f"– Falla 1, muestra {muestra_interpretar}"
)

plt.tight_layout()
plt.show()

In [ ]:
top_T2_plot = top_T2.copy()

top_T2_plot["nombre"] = (
    top_T2_plot["variable"]
    .map(nombres_variables)
    .fillna(top_T2_plot["variable"])
)

top_T2_plot = (
    top_T2_plot
    .sort_values("abs_contrib_T2")
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_T2_plot["nombre"],
    top_T2_plot["contrib_T2"]
)

plt.axvline(
    x=0,
    linewidth=1
)

plt.xlabel("Contribución a Hotelling $T^2$")
plt.ylabel("Variable de proceso")

plt.title(
    f"Principales contribuciones a $T^2$ "
    f"– Falla 1, muestra {muestra_interpretar}"
)

plt.tight_layout()
plt.show()

In [ ]:
try:
    import shap
    print("SHAP instalado. Versión:", shap.__version__)

except ModuleNotFoundError:
    print("SHAP no está instalado.")

In [ ]:
import shap
print("SHAP:", shap.__version__)

In [ ]:
caso_if = (
    f_train.loc[
        (f_train["faultNumber"].astype(int) == 1)
        &
        (f_train["simulationRun"].astype(int) == 1)
    ]
    .copy()
    .sort_values("sample")
    .reset_index(drop=True)
)

X_caso_if = (
    caso_if[feature_cols]
    .copy()
)

score_caso_if = (
    -if_model.score_samples(X_caso_if)
)

detalle_caso_if = (
    caso_if[["sample"]]
    .copy()
)

detalle_caso_if["score_if"] = (
    score_caso_if
)

detalle_caso_if["alarma_puntual"] = (
    detalle_caso_if["score_if"]
    > IF_threshold
)

In [ ]:
detalle_caso_if["alarmas_ventana"] = (
    detalle_caso_if["alarma_puntual"]
    .astype(int)
    .rolling(
        window=5,
        min_periods=5
    )
    .sum()
)

detalle_caso_if["alarma_persistente"] = (
    detalle_caso_if["alarmas_ventana"]
    >= 3
)

detalle_caso_if["inicio_episodio"] = (
    detalle_caso_if["alarma_persistente"]
    &
    ~detalle_caso_if[
        "alarma_persistente"
    ].shift(fill_value=False)
)

In [ ]:
detecciones_if = (
    detalle_caso_if.loc[
        detalle_caso_if["inicio_episodio"]
        &
        (
            detalle_caso_if["sample"]
            >= 21
        )
    ]
)

display(
    detecciones_if.head()
)

muestra_interpretar_if = int(
    detecciones_if.iloc[0]["sample"]
)

print(
    "Muestra seleccionada para SHAP:",
    muestra_interpretar_if
)

In [ ]:
from pathlib import Path
import pyreadr

archivo_ff_local = Path(
    "/content/TEP_FaultFree_Training.RData"
)

resultado_normal_shap = pyreadr.read_r(
    str(archivo_ff_local)
)

ff_train_shap = (
    resultado_normal_shap[
        "fault_free_training"
    ]
    .copy()
)

del resultado_normal_shap

print(
    ff_train_shap.shape
)

In [ ]:
normal_train_shap = (
    ff_train_shap.loc[
        ff_train_shap[
            "simulationRun"
        ]
        .astype(int)
        .isin(corridas_train),
        feature_cols
    ]
)

In [ ]:
background_if = (
    normal_train_shap
    .sample(
        n=200,
        random_state=42
    )
    .reset_index(drop=True)
)

print(
    "Background SHAP:",
    background_if.shape
)

In [ ]:
def score_anomalia_if(X):

    if isinstance(X, pd.DataFrame):
        X_df = X.copy()

    else:
        X_df = pd.DataFrame(
            X,
            columns=feature_cols
        )

    return (
        -if_model.score_samples(
            X_df[feature_cols]
        )
    )

In [ ]:
X_interpretar_if = (
    caso_if.loc[
        caso_if["sample"]
        == muestra_interpretar_if,
        feature_cols
    ]
    .copy()
)

print(
    "Dimensiones:",
    X_interpretar_if.shape
)

print(
    "Score de anomalía:",
    score_anomalia_if(
        X_interpretar_if
    )[0]
)

print(
    "Umbral IF:",
    IF_threshold
)

In [ ]:
explainer_if = shap.Explainer(
    score_anomalia_if,
    background_if,
    algorithm="permutation",
    feature_names=feature_cols
)

In [ ]:
shap_if = explainer_if(
    X_interpretar_if,
    max_evals=(
        2 * len(feature_cols) + 1
    )
)

In [ ]:
# Crear explícitamente el masker con las 200 muestras normales
masker_if = shap.maskers.Independent(
    background_if,
    max_samples=200
)

# Crear el explicador SHAP
explainer_if = shap.Explainer(
    score_anomalia_if,
    masker_if,
    algorithm="permutation",
    feature_names=feature_cols
)

print(
    "Background utilizado por SHAP:",
    background_if.shape
)

In [ ]:
shap_if = explainer_if(
    X_interpretar_if,
    max_evals=(
        2 * len(feature_cols) + 1
    )
)

In [ ]:
valor_base_if = float(
    np.asarray(
        shap_if.base_values
    ).reshape(-1)[0]
)

valores_shap_if = (
    shap_if.values[0]
)

score_reconstruido_if = (
    valor_base_if
    + valores_shap_if.sum()
)

score_real_if = (
    score_anomalia_if(
        X_interpretar_if
    )[0]
)

print(
    "Valor base SHAP:",
    valor_base_if
)

print(
    "Suma contribuciones SHAP:",
    valores_shap_if.sum()
)

print(
    "Score reconstruido:",
    score_reconstruido_if
)

print(
    "Score real Isolation Forest:",
    score_real_if
)

print(
    "Diferencia:",
    abs(
        score_reconstruido_if
        - score_real_if
    )
)

In [ ]:
tabla_shap_if = pd.DataFrame({
    "variable": feature_cols,
    "shap_value": valores_shap_if,
    "abs_shap": np.abs(
        valores_shap_if
    )
})

top_shap_if = (
    tabla_shap_if
    .sort_values(
        "abs_shap",
        ascending=False
    )
    .head(10)
)

display(
    top_shap_if
)

In [ ]:
top_shap_plot = (
    top_shap_if
    .sort_values(
        "abs_shap",
        ascending=True
    )
)

plt.figure(
    figsize=(9, 5)
)

plt.barh(
    top_shap_plot["variable"],
    top_shap_plot["shap_value"]
)

plt.axvline(
    x=0,
    linewidth=1
)

plt.xlabel(
    "Contribución SHAP al score de anomalía"
)

plt.ylabel(
    "Variable"
)

plt.title(
    f"Interpretación SHAP de Isolation Forest "
    f"– Falla 1, muestra {muestra_interpretar_if}"
)

plt.tight_layout()
plt.show()

In [ ]:
nombres_variables.update({
    "xmeas_1": "Alimentación A - corriente 1",
    "xmeas_7": "Presión del reactor",
    "xmeas_13": "Presión del separador",
    "xmeas_16": "Presión del stripper",
    "xmeas_20": "Trabajo del compresor",
    "xmeas_23": "Componente A - alimentación reactor",
    "xmeas_25": "Componente C - alimentación reactor",
    "xmeas_29": "Componente A - gas de purga",
    "xmeas_31": "Componente C - gas de purga",
    "xmv_5": "Válvula recirculación compresor"
})

In [ ]:
top_shap_plot = top_shap_if.copy()

top_shap_plot["nombre"] = (
    top_shap_plot["variable"]
    .map(nombres_variables)
    .fillna(top_shap_plot["variable"])
)

top_shap_plot = (
    top_shap_plot
    .sort_values("abs_shap")
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_shap_plot["nombre"],
    top_shap_plot["shap_value"]
)

plt.axvline(
    x=0,
    linewidth=1
)

plt.xlabel(
    "Contribución SHAP al score de anomalía"
)

plt.ylabel(
    "Variable de proceso"
)

plt.title(
    f"Interpretación SHAP de Isolation Forest "
    f"– Falla 1, muestra {muestra_interpretar_if}"
)

plt.tight_layout()
plt.show()

In [ ]:
variables_shap = set(
    top_shap_if["variable"]
)

variables_spe = set(
    top_SPE["variable"]
)

variables_t2 = set(
    top_T2["variable"]
)

coincidencia_shap_spe = (
    variables_shap
    & variables_spe
)

coincidencia_shap_t2 = (
    variables_shap
    & variables_t2
)

coincidencia_shap_pca_total = (
    variables_shap
    & (
        variables_spe
        | variables_t2
    )
)

print(
    "Coincidencias SHAP - SPE:",
    sorted(coincidencia_shap_spe)
)

print(
    "\nCoincidencias SHAP - T²:",
    sorted(coincidencia_shap_t2)
)

print(
    "\nVariables SHAP presentes en alguna "
    "explicación PCA:",
    sorted(coincidencia_shap_pca_total)
)

print(
    "\nNúmero de coincidencias:",
    len(coincidencia_shap_pca_total),
    "de",
    len(variables_shap)
)

In [ ]:
# Seleccionar falla 1, corrida 1
caso_ae = (
    f_train.loc[
        (f_train["faultNumber"].astype(int) == 1)
        &
        (f_train["simulationRun"].astype(int) == 1)
    ]
    .copy()
    .sort_values("sample")
    .reset_index(drop=True)
)

# Variables del proceso
X_caso_ae = caso_ae[feature_cols].copy()

# Escalar con el mismo scaler normal
X_caso_ae_scaled = scaler.transform(
    X_caso_ae
)

# Reconstrucción
X_caso_ae_reconstructed = autoencoder.predict(
    X_caso_ae_scaled,
    batch_size=512,
    verbose=0
)

# Score MSE por muestra
score_caso_ae = np.mean(
    (
        X_caso_ae_scaled
        - X_caso_ae_reconstructed
    ) ** 2,
    axis=1
)

# Tabla temporal
detalle_caso_ae = (
    caso_ae[["sample"]]
    .copy()
)

detalle_caso_ae["score_ae"] = (
    score_caso_ae
)

detalle_caso_ae["alarma_puntual"] = (
    detalle_caso_ae["score_ae"]
    > AE_threshold
)

In [ ]:
detalle_caso_ae["alarmas_ventana"] = (
    detalle_caso_ae["alarma_puntual"]
    .astype(int)
    .rolling(
        window=5,
        min_periods=5
    )
    .sum()
)

detalle_caso_ae["alarma_persistente"] = (
    detalle_caso_ae["alarmas_ventana"]
    >= 3
)

detalle_caso_ae["inicio_episodio"] = (
    detalle_caso_ae["alarma_persistente"]
    &
    ~detalle_caso_ae[
        "alarma_persistente"
    ].shift(fill_value=False)
)

In [ ]:
detecciones_ae = (
    detalle_caso_ae.loc[
        detalle_caso_ae["inicio_episodio"]
        &
        (detalle_caso_ae["sample"] >= 21)
    ]
)

display(detecciones_ae.head())

muestra_interpretar_ae = int(
    detecciones_ae.iloc[0]["sample"]
)

print(
    "Muestra seleccionada para interpretar AE:",
    muestra_interpretar_ae
)

In [ ]:
def contribuciones_ae_muestra(
    df_original,
    muestra,
    feature_cols,
    scaler,
    autoencoder
):
    """
    Calcula la contribución de cada variable al score MSE
    del autoencoder para una observación determinada.
    """

    # Seleccionar observación
    fila = (
        df_original.loc[
            df_original["sample"] == muestra,
            feature_cols
        ]
        .copy()
    )

    if len(fila) != 1:
        raise ValueError(
            "La muestra seleccionada no es única."
        )

    # Escalar
    x_scaled = scaler.transform(fila)

    # Reconstruir
    x_reconstructed = autoencoder.predict(
        x_scaled,
        verbose=0
    )

    # Residuo por variable
    residual = (
        x_scaled[0]
        - x_reconstructed[0]
    )

    # Error cuadrático por variable
    error_cuadratico = (
        residual ** 2
    )

    # Contribución exacta al MSE
    contribucion_score = (
        error_cuadratico
        / len(feature_cols)
    )

    # Score total
    score_total = np.mean(
        error_cuadratico
    )

    tabla = pd.DataFrame({
        "variable": feature_cols,
        "valor_escalado": x_scaled[0],
        "reconstruccion": x_reconstructed[0],
        "residual": residual,
        "error_cuadratico": error_cuadratico,
        "contribucion_score": contribucion_score
    })

    return tabla, score_total

In [ ]:
contribuciones_ae, score_ae_interpretado = (
    contribuciones_ae_muestra(
        df_original=caso_ae,
        muestra=muestra_interpretar_ae,
        feature_cols=feature_cols,
        scaler=scaler,
        autoencoder=autoencoder
    )
)

print(
    "Score AE:",
    score_ae_interpretado
)

print(
    "Umbral AE:",
    AE_threshold
)

In [ ]:
print(
    "Suma de contribuciones:",
    contribuciones_ae[
        "contribucion_score"
    ].sum()
)

print(
    "Score real del Autoencoder:",
    score_ae_interpretado
)

print(
    "Diferencia:",
    abs(
        contribuciones_ae[
            "contribucion_score"
        ].sum()
        - score_ae_interpretado
    )
)

In [ ]:
top_ae = (
    contribuciones_ae
    .sort_values(
        "contribucion_score",
        ascending=False
    )
    .head(10)
)

display(
    top_ae[
        [
            "variable",
            "valor_escalado",
            "reconstruccion",
            "residual",
            "contribucion_score"
        ]
    ]
)

In [ ]:
top_ae_plot = top_ae.copy()

top_ae_plot["nombre"] = (
    top_ae_plot["variable"]
    .map(nombres_variables)
    .fillna(top_ae_plot["variable"])
)

top_ae_plot = (
    top_ae_plot
    .sort_values(
        "contribucion_score"
    )
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_ae_plot["nombre"],
    top_ae_plot["contribucion_score"]
)

plt.xlabel(
    "Contribución al error de reconstrucción"
)

plt.ylabel(
    "Variable de proceso"
)

plt.title(
    f"Principales contribuciones al score del Autoencoder "
    f"– Falla 1, muestra {muestra_interpretar_ae}"
)

plt.tight_layout()
plt.show()

In [ ]:
variables_ae = set(
    top_ae["variable"]
)

variables_shap = set(
    top_shap_if["variable"]
)

variables_pca = (
    set(top_SPE["variable"])
    |
    set(top_T2["variable"])
)

print(
    "Coincidencias AE - SHAP:",
    sorted(
        variables_ae
        & variables_shap
    )
)

print(
    "\nCoincidencias AE - PCA:",
    sorted(
        variables_ae
        & variables_pca
    )
)

variables_comunes_3_modelos = (
    variables_ae
    & variables_shap
    & variables_pca
)

print(
    "\nVariables presentes en los tres enfoques:",
    sorted(
        variables_comunes_3_modelos
    )
)

print(
    "\nNúmero de variables comunes a los tres:",
    len(variables_comunes_3_modelos)
)

## 10. Evaluación independiente en FaultFree Testing

Estimación de falsas alarmas en operación normal independiente para los tres detectores.


In [ ]:
import gc
import psutil

globals().pop("f_train", None)

gc.collect()

memoria = psutil.virtual_memory()

print(
    f"Memoria disponible: "
    f"{memoria.available / (1024 ** 3):.2f} GB"
)

print(
    f"Memoria utilizada: "
    f"{memoria.percent:.1f} %"
)

In [ ]:
from pathlib import Path
from shutil import copy2
import pyreadr

archivo_ff_test_drive = (
    DATA_DIR /
    "TEP_FaultFree_Testing.RData"
)

archivo_ff_test_local = Path(
    "/content/TEP_FaultFree_Testing.RData"
)

if not archivo_ff_test_local.exists():

    print("Copiando archivo desde Google Drive...")

    copy2(
        archivo_ff_test_drive,
        archivo_ff_test_local
    )

print("Leyendo FaultFree Testing...")

resultado_ff_test = pyreadr.read_r(
    str(archivo_ff_test_local)
)

print(
    "Objetos encontrados:",
    list(resultado_ff_test.keys())
)

In [ ]:
nombre_objeto_ff_test = next(
    iter(resultado_ff_test.keys())
)

ff_test = (
    resultado_ff_test[
        nombre_objeto_ff_test
    ]
    .copy()
)

del resultado_ff_test
gc.collect()

print(
    "Objeto:",
    nombre_objeto_ff_test
)

print(
    "Dimensiones:",
    ff_test.shape
)

In [ ]:
print(
    "Corridas:",
    ff_test["simulationRun"].nunique()
)

print(
    "Fault numbers:",
    sorted(
        ff_test["faultNumber"]
        .astype(int)
        .unique()
    )
)

muestras_por_corrida_ff_test = (
    ff_test
    .groupby("simulationRun")
    .size()
)

print(
    "Mínimo de muestras por corrida:",
    muestras_por_corrida_ff_test.min()
)

print(
    "Máximo de muestras por corrida:",
    muestras_por_corrida_ff_test.max()
)

print(
    "Valores nulos:",
    ff_test.isna().sum().sum()
)

print(
    "Duplicados exactos:",
    ff_test.duplicated().sum()
)

In [ ]:
columnas_faltantes_test = [
    col
    for col in feature_cols
    if col not in ff_test.columns
]

print(
    "Variables del modelo:",
    len(feature_cols)
)

print(
    "Variables faltantes en testing:",
    columnas_faltantes_test
)

In [ ]:
ff_test = (
    ff_test
    .sort_values(
        ["simulationRun", "sample"]
    )
    .reset_index(drop=True)
)

print(ff_test.shape)
print(
    ff_test[
        ["simulationRun", "sample"]
    ].head()
)

In [ ]:
def evaluar_alarmas_normales(
    df_metadata,
    alarma_puntual,
    nombre_modelo,
    ventana=5,
    minimo_alarmas=3,
    minutos_por_muestra=3
):
    """
    Evalúa la carga de falsas alarmas de un detector
    sobre corridas completamente normales.
    """

    resultados = (
        df_metadata[
            ["simulationRun", "sample"]
        ]
        .copy()
        .reset_index(drop=True)
    )

    resultados["alarma_puntual"] = (
        np.asarray(
            alarma_puntual,
            dtype=bool
        )
    )

    # ---------------------------------------------------------
    # Regla temporal k-de-n
    # ---------------------------------------------------------

    resultados["alarmas_ventana"] = (
        resultados
        .groupby("simulationRun")[
            "alarma_puntual"
        ]
        .transform(
            lambda serie: (
                serie
                .astype(int)
                .rolling(
                    window=ventana,
                    min_periods=ventana
                )
                .sum()
            )
        )
    )

    resultados["alarma_persistente"] = (
        resultados["alarmas_ventana"]
        >= minimo_alarmas
    )

    # ---------------------------------------------------------
    # Inicio de episodios
    # ---------------------------------------------------------

    resultados["inicio_evento"] = (
        resultados
        .groupby("simulationRun")[
            "alarma_persistente"
        ]
        .transform(
            lambda serie: (
                serie
                &
                ~serie.shift(
                    fill_value=False
                )
            )
        )
    )

    resultados["id_evento"] = (
        resultados["inicio_evento"]
        .groupby(
            resultados["simulationRun"]
        )
        .cumsum()
    )

    resultados.loc[
        ~resultados["alarma_persistente"],
        "id_evento"
    ] = pd.NA

    # ---------------------------------------------------------
    # Episodios falsos
    # ---------------------------------------------------------

    episodios = (
        resultados
        .dropna(
            subset=["id_evento"]
        )
        .groupby(
            [
                "simulationRun",
                "id_evento"
            ],
            as_index=False
        )
        .agg(
            muestra_inicio=(
                "sample",
                "min"
            ),
            muestra_fin=(
                "sample",
                "max"
            ),
            duracion_muestras=(
                "alarma_persistente",
                "size"
            )
        )
    )

    episodios["duracion_minutos"] = (
        episodios[
            "duracion_muestras"
        ]
        * minutos_por_muestra
    )

    # ---------------------------------------------------------
    # Métricas
    # ---------------------------------------------------------

    horas_normales = (
        len(resultados)
        * minutos_por_muestra
        / 60
    )

    numero_episodios = len(
        episodios
    )

    corridas_afectadas = (
        episodios[
            "simulationRun"
        ].nunique()
        if numero_episodios > 0
        else 0
    )

    far_puntual = (
        resultados[
            "alarma_puntual"
        ].mean()
    )

    far_persistente = (
        resultados[
            "alarma_persistente"
        ].mean()
    )

    resumen = {
        "Modelo":
            nombre_modelo,

        "Muestras_normales":
            len(resultados),

        "Horas_normales":
            horas_normales,

        "FAR_puntual_pct":
            far_puntual * 100,

        "FAR_persistente_pct":
            far_persistente * 100,

        "Episodios_falsos":
            numero_episodios,

        "Corridas_afectadas":
            corridas_afectadas,

        "Corridas_afectadas_pct":
            corridas_afectadas
            / resultados[
                "simulationRun"
            ].nunique()
            * 100,

        "Episodios_por_100h":
            numero_episodios
            / horas_normales
            * 100,

        "Duracion_media_min":
            (
                episodios[
                    "duracion_minutos"
                ].mean()
                if numero_episodios > 0
                else 0
            ),

        "Duracion_mediana_min":
            (
                episodios[
                    "duracion_minutos"
                ].median()
                if numero_episodios > 0
                else 0
            ),

        "Duracion_maxima_min":
            (
                episodios[
                    "duracion_minutos"
                ].max()
                if numero_episodios > 0
                else 0
            )
    }

    return resultados, episodios, resumen

In [ ]:
#Evaluación PCA
X_test_pca = (
    ff_test[
        feature_cols
    ]
    .copy()
)

# Escalado
X_test_pca_scaled_array = (
    scaler.transform(
        X_test_pca
    )
)

X_test_pca_scaled = pd.DataFrame(
    X_test_pca_scaled_array,
    columns=feature_cols,
    index=X_test_pca.index
)

# Proyección PCA
scores_pca_test = (
    pca_model.transform(
        X_test_pca_scaled
    )
)

eigenvalues = (
    pca_model.explained_variance_
)

# Hotelling T²
T2_test = np.sum(
    (
        scores_pca_test ** 2
    ) / eigenvalues,
    axis=1
)

# Reconstrucción
X_test_pca_reconstructed = (
    pca_model.inverse_transform(
        scores_pca_test
    )
)

# SPE/Q
SPE_test = np.sum(
    (
        X_test_pca_scaled_array
        - X_test_pca_reconstructed
    ) ** 2,
    axis=1
)

# Alarma puntual
alarma_pca_test = (
    (T2_test > T2_threshold)
    |
    (SPE_test > SPE_threshold)
)

print(
    "FAR puntual PCA antes de persistencia:",
    alarma_pca_test.mean()
)

In [ ]:
resultados_pca_test_normal, \
episodios_pca_test_normal, \
resumen_pca_test_normal = evaluar_alarmas_normales(
    df_metadata=ff_test,
    alarma_puntual=alarma_pca_test,
    nombre_modelo="PCA",
    ventana=5,
    minimo_alarmas=3,
    minutos_por_muestra=3
)

display(
    pd.DataFrame(
        [resumen_pca_test_normal]
    ).round(4)
)

In [ ]:
del X_test_pca
del X_test_pca_scaled_array
del X_test_pca_scaled
del scores_pca_test
del X_test_pca_reconstructed

gc.collect()

In [ ]:
#Evaluación isolation forest
X_test_if = (
    ff_test[
        feature_cols
    ]
)

score_if_test = (
    -if_model.score_samples(
        X_test_if
    )
)

alarma_if_test = (
    score_if_test
    > IF_threshold
)

print(
    "FAR puntual IF antes de persistencia:",
    alarma_if_test.mean()
)

In [ ]:
resultados_if_test_normal, \
episodios_if_test_normal, \
resumen_if_test_normal = evaluar_alarmas_normales(
    df_metadata=ff_test,
    alarma_puntual=alarma_if_test,
    nombre_modelo="Isolation Forest",
    ventana=5,
    minimo_alarmas=3,
    minutos_por_muestra=3
)

display(
    pd.DataFrame(
        [resumen_if_test_normal]
    ).round(4)
)

In [ ]:
del X_test_if
del score_if_test

gc.collect()

In [ ]:
X_test_ae = (
    ff_test[
        feature_cols
    ]
)

X_test_ae_scaled = (
    scaler.transform(
        X_test_ae
    )
)

X_test_ae_reconstructed = (
    autoencoder.predict(
        X_test_ae_scaled,
        batch_size=2048,
        verbose=1
    )
)

score_ae_test = np.mean(
    (
        X_test_ae_scaled
        - X_test_ae_reconstructed
    ) ** 2,
    axis=1
)

alarma_ae_test = (
    score_ae_test
    > AE_threshold
)

print(
    "FAR puntual Autoencoder antes de persistencia:",
    alarma_ae_test.mean()
)

In [ ]:
resultados_ae_test_normal, \
episodios_ae_test_normal, \
resumen_ae_test_normal = evaluar_alarmas_normales(
    df_metadata=ff_test,
    alarma_puntual=alarma_ae_test,
    nombre_modelo="Autoencoder",
    ventana=5,
    minimo_alarmas=3,
    minutos_por_muestra=3
)

display(
    pd.DataFrame(
        [resumen_ae_test_normal]
    ).round(4)
)

In [ ]:
del X_test_ae
del X_test_ae_scaled
del X_test_ae_reconstructed
del score_ae_test

gc.collect()

In [ ]:
comparacion_test_normal = pd.DataFrame(
    [
        resumen_pca_test_normal,
        resumen_if_test_normal,
        resumen_ae_test_normal
    ]
)

display(
    comparacion_test_normal.round(4)
)

In [ ]:
comparacion_validacion_testing = pd.DataFrame({
    "Modelo": [
        "PCA",
        "Isolation Forest",
        "Autoencoder"
    ],

    "Episodios_100h_validacion": [
        1.44,
        4.24,
        1.48
    ],

    "Episodios_100h_testing": [
        resumen_pca_test_normal[
            "Episodios_por_100h"
        ],
        resumen_if_test_normal[
            "Episodios_por_100h"
        ],
        resumen_ae_test_normal[
            "Episodios_por_100h"
        ]
    ],

    "FAR_persistente_testing_pct": [
        resumen_pca_test_normal[
            "FAR_persistente_pct"
        ],
        resumen_if_test_normal[
            "FAR_persistente_pct"
        ],
        resumen_ae_test_normal[
            "FAR_persistente_pct"
        ]
    ]
})

display(
    comparacion_validacion_testing.round(4)
)

In [ ]:
comparacion_test_normal.to_csv(
    RESULTS_DIR /
    "evaluacion_final_FaultFree_Testing.csv",
    index=False
)

comparacion_validacion_testing.to_csv(
    RESULTS_DIR /
    "comparacion_validacion_vs_testing_falsas_alarmas.csv",
    index=False
)

print(
    "Evaluación normal de testing guardada correctamente."
)

In [ ]:
comparacion_test_normal.to_csv(
    RESULTS_DIR /
    "evaluacion_final_FaultFree_Testing.csv",
    index=False
)

comparacion_validacion_testing.to_csv(
    RESULTS_DIR /
    "comparacion_validacion_vs_testing_falsas_alarmas.csv",
    index=False
)

print("Resultados normales de testing guardados.")

In [ ]:
import gc
import psutil

variables_a_eliminar = [
    "ff_test",

    "alarma_pca_test",
    "T2_test",
    "SPE_test",

    "alarma_if_test",

    "alarma_ae_test",

    "resultados_pca_test_normal",
    "resultados_if_test_normal",
    "resultados_ae_test_normal",

    "episodios_pca_test_normal",
    "episodios_if_test_normal",
    "episodios_ae_test_normal"
]

for variable in variables_a_eliminar:
    globals().pop(variable, None)

gc.collect()

memoria = psutil.virtual_memory()

print(
    f"Memoria disponible: "
    f"{memoria.available / (1024 ** 3):.2f} GB"
)

print(
    f"Memoria utilizada: "
    f"{memoria.percent:.1f} %"
)

## 11. Recuperación del entorno y Faulty Testing

Recuperación de modelos/checkpoints, preparación del conjunto Faulty Testing y procesamiento por particiones.


In [ ]:
from pathlib import Path

archivo_faulty_test_drive = (
    DATA_DIR /
    "TEP_Faulty_Testing.RData"
)

print(
    "¿Existe?:",
    archivo_faulty_test_drive.exists()
)

print(
    "Tamaño en Drive:",
    round(
        archivo_faulty_test_drive.stat().st_size
        / (1024 ** 2),
        2
    ),
    "MB"
)

In [ ]:
memoria = psutil.virtual_memory()

print(
    "RAM disponible antes de cargar:",
    round(
        memoria.available / (1024 ** 3),
        2
    ),
    "GB"
)

In [ ]:
from shutil import copy2

archivo_faulty_test_local = Path(
    "/content/TEP_Faulty_Testing.RData"
)

if not archivo_faulty_test_local.exists():

    print("Copiando Faulty Testing desde Drive...")

    copy2(
        archivo_faulty_test_drive,
        archivo_faulty_test_local
    )

    print("Copia terminada.")

else:
    print(
        "El archivo ya está disponible en /content."
    )

print(
    "Tamaño local:",
    round(
        archivo_faulty_test_local.stat().st_size
        / (1024 ** 2),
        2
    ),
    "MB"
)

In [ ]:
import pyreadr

print("Iniciando lectura de Faulty Testing...")

resultado_faulty_test = pyreadr.read_r(
    str(archivo_faulty_test_local)
)

print("Lectura terminada.")

print(
    "Objetos encontrados:",
    list(resultado_faulty_test.keys())
)

In [ ]:
print("PROJECT_DIR existe:", "PROJECT_DIR" in globals())
print("scaler existe:", "scaler" in globals())
print("pca_model existe:", "pca_model" in globals())
print("if_model existe:", "if_model" in globals())
print("autoencoder existe:", "autoencoder" in globals())

In [ ]:
# ============================================================
# RECUPERACIÓN DEL PROYECTO DESPUÉS DE REINICIO DE COLAB
# ============================================================

from google.colab import drive
from pathlib import Path
import joblib
import tensorflow as tf
import numpy as np
import pandas as pd
import gc

# ------------------------------------------------------------
# 1. Montar Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")

# ------------------------------------------------------------
# 2. Definir rutas del proyecto
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/TFM_Tennessee_Eastman"
)

DATA_DIR = PROJECT_DIR / "Datos"
MODELS_DIR = PROJECT_DIR / "Modelos"
RESULTS_DIR = PROJECT_DIR / "Resultados"

print("Proyecto:", PROJECT_DIR)
print("¿Existe PROJECT_DIR?:", PROJECT_DIR.exists())
print("¿Existe DATA_DIR?:", DATA_DIR.exists())
print("¿Existe MODELS_DIR?:", MODELS_DIR.exists())
print("¿Existe RESULTS_DIR?:", RESULTS_DIR.exists())

# ------------------------------------------------------------
# 3. Recuperar PCA
# ------------------------------------------------------------

ruta_pca = (
    MODELS_DIR /
    "pca_baseline_v1.joblib"
)

checkpoint_pca = joblib.load(
    ruta_pca
)

scaler = checkpoint_pca["scaler"]
pca_model = checkpoint_pca["pca_model"]
feature_cols = checkpoint_pca["feature_cols"]

T2_threshold = checkpoint_pca["T2_threshold"]
SPE_threshold = checkpoint_pca["SPE_threshold"]

corridas_train = checkpoint_pca["corridas_train"]
corridas_val = checkpoint_pca["corridas_val"]

# Recuperar configuración temporal
ventana = checkpoint_pca.get(
    "ventana_persistencia",
    5
)

minimo_alarmas = checkpoint_pca.get(
    "minimo_alarmas",
    3
)

print("\nPCA recuperado correctamente.")

# ------------------------------------------------------------
# 4. Recuperar Isolation Forest
# ------------------------------------------------------------

ruta_if = (
    MODELS_DIR /
    "isolation_forest_baseline_v1.joblib"
)

checkpoint_if = joblib.load(
    ruta_if
)

if_model = checkpoint_if["if_model"]
IF_threshold = checkpoint_if["IF_threshold"]

print("Isolation Forest recuperado correctamente.")

# ------------------------------------------------------------
# 5. Recuperar Autoencoder
# ------------------------------------------------------------

ruta_autoencoder = (
    MODELS_DIR /
    "autoencoder_baseline_v1.keras"
)

autoencoder = tf.keras.models.load_model(
    ruta_autoencoder,
    compile=False
)

ruta_config_ae = (
    MODELS_DIR /
    "autoencoder_baseline_v1_config.joblib"
)

checkpoint_ae = joblib.load(
    ruta_config_ae
)

AE_threshold = checkpoint_ae["AE_threshold"]

print("Autoencoder recuperado correctamente.")

# ------------------------------------------------------------
# 6. Resumen de parámetros recuperados
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("RESUMEN DEL SISTEMA RECUPERADO")
print("=" * 55)

print(
    "Variables de proceso:",
    len(feature_cols)
)

print(
    "Componentes PCA:",
    pca_model.n_components_
)

print(
    f"Umbral T²: {T2_threshold:.6f}"
)

print(
    f"Umbral SPE: {SPE_threshold:.6f}"
)

print(
    f"Umbral Isolation Forest: {IF_threshold:.6f}"
)

print(
    f"Umbral Autoencoder: {AE_threshold:.6f}"
)

print(
    f"Regla temporal: "
    f"{minimo_alarmas} de {ventana}"
)

print(
    "Corridas normales de entrenamiento:",
    len(corridas_train)
)

print(
    "Corridas normales de validación:",
    len(corridas_val)
)

gc.collect()

print("\nRecuperación terminada.")

In [ ]:
print("PROJECT_DIR existe:", "PROJECT_DIR" in globals())
print("scaler existe:", "scaler" in globals())
print("pca_model existe:", "pca_model" in globals())
print("if_model existe:", "if_model" in globals())
print("autoencoder existe:", "autoencoder" in globals())

print()
print("PCA variables:", len(feature_cols))
print("PCA componentes:", pca_model.n_components_)
print("AE input:", autoencoder.input_shape)
print("AE output:", autoencoder.output_shape)

In [ ]:
%pip install -q kagglehub

In [ ]:
import kagglehub
from pathlib import Path

ruta_dataset_csv = kagglehub.dataset_download(
    "afrniomelo/tep-csv"
)

ruta_dataset_csv = Path(ruta_dataset_csv)

print("Dataset descargado en:")
print(ruta_dataset_csv)

In [ ]:
archivos_csv = sorted(
    ruta_dataset_csv.rglob("*.csv")
)

print(
    f"Archivos CSV encontrados: {len(archivos_csv)}\n"
)

for archivo in archivos_csv:
    tamaño_mb = archivo.stat().st_size / (1024 ** 2)

    print(
        f"{archivo.name:35s} | "
        f"{tamaño_mb:.2f} MB"
    )

In [ ]:
candidatos_faulty_test = [
    archivo
    for archivo in archivos_csv
    if (
        "Faulty" in archivo.name
        and "Testing" in archivo.name
    )
]

print("Candidatos encontrados:")

for archivo in candidatos_faulty_test:
    print(
        archivo,
        "|",
        round(
            archivo.stat().st_size / (1024 ** 2),
            2
        ),
        "MB"
    )

In [ ]:
import pandas as pd
from pathlib import Path

archivo_faulty_test_csv = candidatos_faulty_test[0]

print("Archivo seleccionado:")
print(archivo_faulty_test_csv)

print(
    "Tamaño:",
    round(
        archivo_faulty_test_csv.stat().st_size / (1024 ** 2),
        2
    ),
    "MB"
)

# Leer únicamente 5 filas
preview_faulty_test = pd.read_csv(
    archivo_faulty_test_csv,
    nrows=5
)

print("\nDimensiones de la muestra:")
print(preview_faulty_test.shape)

print("\nColumnas:")
print(preview_faulty_test.columns.tolist())

display(preview_faulty_test)

In [ ]:
print(
    preview_faulty_test[
        [
            "faultNumber",
            "simulationRun",
            "sample"
        ]
    ]
)

print("\nTipos:")
print(
    preview_faulty_test[
        [
            "faultNumber",
            "simulationRun",
            "sample"
        ]
    ].dtypes
)

In [ ]:
columnas_faltantes_csv = [
    col
    for col in feature_cols
    if col not in preview_faulty_test.columns
]

print(
    "Número de columnas:",
    len(preview_faulty_test.columns)
)

print(
    "Variables del modelo encontradas:",
    len(feature_cols) - len(columnas_faltantes_csv)
)

print(
    "Variables faltantes:",
    columnas_faltantes_csv
)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc

PARTITION_DIR = (
    PROJECT_DIR
    / "Datos_Procesados"
    / "Faulty_Testing_Parquet_v1"
)

PARTITION_DIR.parent.mkdir(
    parents=True,
    exist_ok=True
)

print("Carpeta de salida:")
print(PARTITION_DIR)

print(
    "¿Ya existe?:",
    PARTITION_DIR.exists()
)

In [ ]:
try:
    import pyarrow
    import pyarrow.parquet as pq

    print(
        "PyArrow disponible:",
        pyarrow.__version__
    )

except ModuleNotFoundError:
    print("PyArrow no está instalado.")

In [ ]:
from collections import defaultdict

# ============================================================
# CONFIGURACIÓN
# ============================================================

CHUNKSIZE = 200_000

columnas_necesarias = (
    [
        "faultNumber",
        "simulationRun",
        "sample"
    ]
    + list(feature_cols)
)

# Contadores para validar simultáneamente el dataset
filas_por_falla = defaultdict(int)

corridas_por_falla = {
    i: set()
    for i in range(1, 21)
}

muestra_min_por_falla = {
    i: np.inf
    for i in range(1, 21)
}

muestra_max_por_falla = {
    i: -np.inf
    for i in range(1, 21)
}

# ============================================================
# SEGURIDAD
# ============================================================

if PARTITION_DIR.exists():

    archivos_existentes = list(
        PARTITION_DIR.rglob("*.parquet")
    )

    if len(archivos_existentes) > 0:

        raise RuntimeError(
            "La carpeta de particiones ya contiene archivos "
            "Parquet. No se sobrescribirá automáticamente."
        )

else:

    PARTITION_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

# ============================================================
# LECTURA OUT-OF-CORE
# ============================================================

print("Iniciando partición de Faulty Testing...")
print(
    f"Tamaño de bloque: {CHUNKSIZE:,} filas"
)
print()

lector_csv = pd.read_csv(
    archivo_faulty_test_csv,
    usecols=columnas_necesarias,
    chunksize=CHUNKSIZE
)

total_filas_procesadas = 0

for numero_chunk, chunk in enumerate(
    lector_csv,
    start=1
):

    # Mantener identificadores en tipos pequeños
    # sin modificar las 52 variables del proceso.
    chunk["faultNumber"] = (
        chunk["faultNumber"].astype("int16")
    )

    chunk["simulationRun"] = (
        chunk["simulationRun"].astype("int16")
    )

    chunk["sample"] = (
        chunk["sample"].astype("int16")
    )

    # --------------------------------------------------------
    # Separar las fallas presentes en este bloque
    # --------------------------------------------------------

    for falla, bloque_falla in chunk.groupby(
        "faultNumber",
        sort=False
    ):

        falla = int(falla)

        carpeta_falla = (
            PARTITION_DIR
            / f"falla_{falla:02d}"
        )

        carpeta_falla.mkdir(
            parents=True,
            exist_ok=True
        )

        ruta_parquet = (
            carpeta_falla
            / f"part_{numero_chunk:04d}.parquet"
        )

        bloque_falla.to_parquet(
            ruta_parquet,
            index=False,
            engine="pyarrow",
            compression="zstd"
        )

        # ----------------------------------------------------
        # Estadísticas de validación
        # ----------------------------------------------------

        filas_por_falla[falla] += len(
            bloque_falla
        )

        corridas_por_falla[falla].update(
            bloque_falla[
                "simulationRun"
            ].astype(int).unique()
        )

        muestra_min_por_falla[falla] = min(
            muestra_min_por_falla[falla],
            int(
                bloque_falla[
                    "sample"
                ].min()
            )
        )

        muestra_max_por_falla[falla] = max(
            muestra_max_por_falla[falla],
            int(
                bloque_falla[
                    "sample"
                ].max()
            )
        )

    total_filas_procesadas += len(chunk)

    print(
        f"Chunk {numero_chunk:02d} | "
        f"Filas acumuladas: "
        f"{total_filas_procesadas:,}"
    )

    del chunk
    gc.collect()

print()
print("PARTICIÓN TERMINADA.")
print(
    "Total de filas procesadas:",
    f"{total_filas_procesadas:,}"
)

In [ ]:
validacion_particiones = pd.DataFrame({
    "falla": range(1, 21),

    "filas": [
        filas_por_falla[i]
        for i in range(1, 21)
    ],

    "corridas": [
        len(corridas_por_falla[i])
        for i in range(1, 21)
    ],

    "muestra_min": [
        muestra_min_por_falla[i]
        for i in range(1, 21)
    ],

    "muestra_max": [
        muestra_max_por_falla[i]
        for i in range(1, 21)
    ]
})

display(validacion_particiones)

In [ ]:
print(
    "Filas totales:",
    validacion_particiones[
        "filas"
    ].sum()
)

print(
    "Número de fallas:",
    len(
        validacion_particiones
    )
)

## 12. Evaluación final por falla en testing

Funciones de evaluación para PCA, Isolation Forest y Autoencoder, checkpointing y consolidación de resultados finales.


In [ ]:
import numpy as np
import pandas as pd
import gc


def resumir_deteccion_testing(
    df,
    alarma_puntual,
    nombre_modelo,
    muestra_inicio_falla=161,
    ventana=5,
    minimo_alarmas=3,
    minutos_por_muestra=3
):
    """
    Resume el desempeño de un detector sobre una única falla
    del conjunto Faulty_Testing.

    La falla comienza en la muestra 161.

    Se calculan:
    - prealarmas
    - corridas evaluables
    - tasa de detección
    - retraso de detección
    - proporción de alarmas puntuales post-falla
    """

    resultados = (
        df[
            [
                "simulationRun",
                "sample"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    resultados["alarma_puntual"] = np.asarray(
        alarma_puntual,
        dtype=bool
    )

    # =========================================================
    # Regla temporal 3-de-5
    # =========================================================

    resultados["alarmas_ventana"] = (
        resultados
        .groupby("simulationRun")[
            "alarma_puntual"
        ]
        .transform(
            lambda serie: (
                serie
                .astype(int)
                .rolling(
                    window=ventana,
                    min_periods=ventana
                )
                .sum()
            )
        )
    )

    resultados["alarma_persistente"] = (
        resultados["alarmas_ventana"]
        >= minimo_alarmas
    )

    # =========================================================
    # Inicio de episodios persistentes
    # =========================================================

    resultados["inicio_episodio"] = (
        resultados
        .groupby("simulationRun")[
            "alarma_persistente"
        ]
        .transform(
            lambda serie: (
                serie
                &
                ~serie.shift(
                    fill_value=False
                )
            )
        )
    )

    # =========================================================
    # Corridas con prealarma
    # =========================================================

    mascara_prealarma = (
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            < muestra_inicio_falla
        )
    )

    corridas_con_prealarma = set(
        resultados.loc[
            mascara_prealarma,
            "simulationRun"
        ].unique()
    )

    # =========================================================
    # Primer episodio iniciado después de la falla
    # =========================================================

    mascara_deteccion = (
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            >= muestra_inicio_falla
        )
    )

    primeras_detecciones = (
        resultados.loc[
            mascara_deteccion,
            [
                "simulationRun",
                "sample"
            ]
        ]
        .groupby(
            "simulationRun",
            as_index=False
        )
        .agg(
            muestra_deteccion=(
                "sample",
                "min"
            )
        )
    )

    corridas_detectadas_set = set(
        primeras_detecciones[
            "simulationRun"
        ]
    )

    # =========================================================
    # Corridas evaluables
    # =========================================================

    todas_corridas = set(
        resultados[
            "simulationRun"
        ].unique()
    )

    corridas_evaluables_set = (
        todas_corridas
        - corridas_con_prealarma
    )

    corridas_detectadas_evaluables_set = (
        corridas_detectadas_set
        & corridas_evaluables_set
    )

    total_corridas = len(
        todas_corridas
    )

    corridas_evaluables = len(
        corridas_evaluables_set
    )

    corridas_detectadas_evaluables = len(
        corridas_detectadas_evaluables_set
    )

    # =========================================================
    # Retraso solo sobre corridas evaluables
    # =========================================================

    primeras_detecciones_evaluables = (
        primeras_detecciones.loc[
            primeras_detecciones[
                "simulationRun"
            ].isin(
                corridas_evaluables_set
            )
        ]
        .copy()
    )

    primeras_detecciones_evaluables[
        "retraso_muestras"
    ] = (
        primeras_detecciones_evaluables[
            "muestra_deteccion"
        ]
        - muestra_inicio_falla
    )

    primeras_detecciones_evaluables[
        "retraso_minutos"
    ] = (
        primeras_detecciones_evaluables[
            "retraso_muestras"
        ]
        * minutos_por_muestra
    )

    if corridas_detectadas_evaluables > 0:

        retrasos = (
            primeras_detecciones_evaluables[
                "retraso_minutos"
            ]
        )

        retraso_medio = retrasos.mean()
        retraso_mediano = retrasos.median()
        retraso_p95 = retrasos.quantile(0.95)
        retraso_minimo = retrasos.min()
        retraso_maximo = retrasos.max()

    else:

        retraso_medio = np.nan
        retraso_mediano = np.nan
        retraso_p95 = np.nan
        retraso_minimo = np.nan
        retraso_maximo = np.nan

    # =========================================================
    # Alarmas puntuales durante el periodo post-falla
    # =========================================================

    mascara_post = (
        resultados["sample"]
        >= muestra_inicio_falla
    )

    alarma_post_pct = (
        resultados.loc[
            mascara_post,
            "alarma_puntual"
        ].mean()
        * 100
    )

    # =========================================================
    # Resumen
    # =========================================================

    resumen = {

        "Modelo": nombre_modelo,

        "corridas_totales":
            total_corridas,

        "corridas_prealarma":
            len(corridas_con_prealarma),

        "corridas_evaluables":
            corridas_evaluables,

        "corridas_detectadas_evaluables":
            corridas_detectadas_evaluables,

        "tasa_deteccion_evaluable_pct": (
            corridas_detectadas_evaluables
            / corridas_evaluables
            * 100
            if corridas_evaluables > 0
            else np.nan
        ),

        "alarma_puntual_post_pct":
            alarma_post_pct,

        "retraso_medio_min":
            retraso_medio,

        "retraso_mediano_min":
            retraso_mediano,

        "retraso_p95_min":
            retraso_p95,

        "retraso_minimo_min":
            retraso_minimo,

        "retraso_maximo_min":
            retraso_maximo
    }

    return resumen

In [ ]:
def evaluar_pca_testing(
    df,
    scaler,
    pca_model,
    feature_cols,
    T2_threshold,
    SPE_threshold,
    muestra_inicio_falla=161
):

    # ---------------------------------------------------------
    # Variables
    # ---------------------------------------------------------

    X = df[
        feature_cols
    ]

    # ---------------------------------------------------------
    # Estandarización
    # ---------------------------------------------------------

    X_scaled_array = scaler.transform(
        X
    )

    X_scaled_df = pd.DataFrame(
        X_scaled_array,
        columns=feature_cols,
        copy=False
    )

    # ---------------------------------------------------------
    # PCA
    # ---------------------------------------------------------

    scores = pca_model.transform(
        X_scaled_df
    )

    eigenvalues = (
        pca_model.explained_variance_
    )

    T2 = np.sum(
        (scores ** 2)
        / eigenvalues,
        axis=1
    )

    # ---------------------------------------------------------
    # Reconstrucción y SPE
    # ---------------------------------------------------------

    X_reconstructed = (
        pca_model.inverse_transform(
            scores
        )
    )

    SPE = np.sum(
        (
            X_scaled_array
            - X_reconstructed
        ) ** 2,
        axis=1
    )

    alarma = (
        (T2 > T2_threshold)
        |
        (SPE > SPE_threshold)
    )

    resumen = resumir_deteccion_testing(
        df=df,
        alarma_puntual=alarma,
        nombre_modelo="PCA",
        muestra_inicio_falla=muestra_inicio_falla
    )

    del X
    del X_scaled_array
    del X_scaled_df
    del scores
    del X_reconstructed
    del T2
    del SPE
    del alarma

    gc.collect()

    return resumen

In [ ]:
def evaluar_if_testing(
    df,
    if_model,
    feature_cols,
    IF_threshold,
    muestra_inicio_falla=161
):

    X = df[
        feature_cols
    ]

    score_if = (
        -if_model.score_samples(
            X
        )
    )

    alarma = (
        score_if
        > IF_threshold
    )

    resumen = resumir_deteccion_testing(
        df=df,
        alarma_puntual=alarma,
        nombre_modelo="Isolation Forest",
        muestra_inicio_falla=muestra_inicio_falla
    )

    del X
    del score_if
    del alarma

    gc.collect()

    return resumen

In [ ]:
def evaluar_ae_testing(
    df,
    scaler,
    autoencoder,
    feature_cols,
    AE_threshold,
    muestra_inicio_falla=161,
    batch_size=2048
):

    X = df[
        feature_cols
    ]

    X_scaled = scaler.transform(
        X
    )

    X_reconstructed = (
        autoencoder.predict(
            X_scaled,
            batch_size=batch_size,
            verbose=0
        )
    )

    score_ae = np.mean(
        (
            X_scaled
            - X_reconstructed
        ) ** 2,
        axis=1
    )

    alarma = (
        score_ae
        > AE_threshold
    )

    resumen = resumir_deteccion_testing(
        df=df,
        alarma_puntual=alarma,
        nombre_modelo="Autoencoder",
        muestra_inicio_falla=muestra_inicio_falla
    )

    del X
    del X_scaled
    del X_reconstructed
    del score_ae
    del alarma

    gc.collect()

    return resumen

In [ ]:
ruta_falla_1 = (
    PARTITION_DIR /
    "falla_01"
)

falla_1_test = pd.read_parquet(
    ruta_falla_1
)

falla_1_test = (
    falla_1_test
    .sort_values(
        [
            "simulationRun",
            "sample"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Dimensiones:",
    falla_1_test.shape
)

print(
    "Corridas:",
    falla_1_test[
        "simulationRun"
    ].nunique()
)

print(
    "Muestra mínima:",
    falla_1_test[
        "sample"
    ].min()
)

print(
    "Muestra máxima:",
    falla_1_test[
        "sample"
    ].max()
)

print(
    "Falla:",
    falla_1_test[
        "faultNumber"
    ].unique()
)

In [ ]:
resultado_pca_f1_test = evaluar_pca_testing(
    df=falla_1_test,
    scaler=scaler,
    pca_model=pca_model,
    feature_cols=feature_cols,
    T2_threshold=T2_threshold,
    SPE_threshold=SPE_threshold,
    muestra_inicio_falla=161
)

display(
    pd.DataFrame(
        [resultado_pca_f1_test]
    ).round(3)
)

In [ ]:
resultado_if_f1_test = evaluar_if_testing(
    df=falla_1_test,
    if_model=if_model,
    feature_cols=feature_cols,
    IF_threshold=IF_threshold,
    muestra_inicio_falla=161
)

display(
    pd.DataFrame(
        [resultado_if_f1_test]
    ).round(3)
)

In [ ]:
resultado_ae_f1_test = evaluar_ae_testing(
    df=falla_1_test,
    scaler=scaler,
    autoencoder=autoencoder,
    feature_cols=feature_cols,
    AE_threshold=AE_threshold,
    muestra_inicio_falla=161
)

display(
    pd.DataFrame(
        [resultado_ae_f1_test]
    ).round(3)
)

In [ ]:
import pandas as pd
import numpy as np
import gc
from pathlib import Path

# ============================================================
# ARCHIVO DE CHECKPOINT
# ============================================================

RUTA_CHECKPOINT_TEST = (
    RESULTS_DIR /
    "evaluacion_final_Faulty_Testing_checkpoint.csv"
)

# ============================================================
# RECUPERAR RESULTADOS ANTERIORES SI EXISTEN
# ============================================================

if RUTA_CHECKPOINT_TEST.exists():

    resultados_testing = (
        pd.read_csv(
            RUTA_CHECKPOINT_TEST
        )
        .to_dict("records")
    )

    print(
        "Checkpoint encontrado."
    )

    print(
        "Resultados recuperados:",
        len(resultados_testing)
    )

else:

    resultados_testing = []

    print(
        "No existe checkpoint previo. "
        "Se inicia evaluación desde cero."
    )

In [ ]:
def combinacion_ya_evaluada(
    resultados,
    falla,
    modelo
):

    return any(
        (
            int(r["falla"]) == int(falla)
            and
            r["Modelo"] == modelo
        )
        for r in resultados
    )

In [ ]:
for numero_falla in range(1, 21):

    print()
    print("=" * 60)
    print(
        f"FALLA {numero_falla} DE 20"
    )
    print("=" * 60)

    # --------------------------------------------------------
    # Comprobar si los tres modelos ya están terminados
    # --------------------------------------------------------

    modelos_esperados = [
        "PCA",
        "Isolation Forest",
        "Autoencoder"
    ]

    ya_completos = all(
        combinacion_ya_evaluada(
            resultados_testing,
            numero_falla,
            modelo
        )
        for modelo in modelos_esperados
    )

    if ya_completos:

        print(
            f"Falla {numero_falla} ya estaba "
            "completamente evaluada. Se omite."
        )

        continue

    # --------------------------------------------------------
    # Cargar SOLO esta falla
    # --------------------------------------------------------

    ruta_falla = (
        PARTITION_DIR /
        f"falla_{numero_falla:02d}"
    )

    df_falla = pd.read_parquet(
        ruta_falla
    )

    df_falla = (
        df_falla
        .sort_values(
            [
                "simulationRun",
                "sample"
            ]
        )
        .reset_index(drop=True)
    )

    print(
        "Datos cargados:",
        df_falla.shape
    )

    # ========================================================
    # PCA
    # ========================================================

    if not combinacion_ya_evaluada(
        resultados_testing,
        numero_falla,
        "PCA"
    ):

        print("  → PCA...")

        resultado_pca = evaluar_pca_testing(
            df=df_falla,
            scaler=scaler,
            pca_model=pca_model,
            feature_cols=feature_cols,
            T2_threshold=T2_threshold,
            SPE_threshold=SPE_threshold,
            muestra_inicio_falla=161
        )

        resultado_pca["falla"] = (
            numero_falla
        )

        resultados_testing.append(
            resultado_pca
        )

        # Guardar inmediatamente
        pd.DataFrame(
            resultados_testing
        ).to_csv(
            RUTA_CHECKPOINT_TEST,
            index=False
        )

        print("    PCA terminado y guardado.")

        gc.collect()

    # ========================================================
    # ISOLATION FOREST
    # ========================================================

    if not combinacion_ya_evaluada(
        resultados_testing,
        numero_falla,
        "Isolation Forest"
    ):

        print("  → Isolation Forest...")

        resultado_if = evaluar_if_testing(
            df=df_falla,
            if_model=if_model,
            feature_cols=feature_cols,
            IF_threshold=IF_threshold,
            muestra_inicio_falla=161
        )

        resultado_if["falla"] = (
            numero_falla
        )

        resultados_testing.append(
            resultado_if
        )

        pd.DataFrame(
            resultados_testing
        ).to_csv(
            RUTA_CHECKPOINT_TEST,
            index=False
        )

        print(
            "    Isolation Forest terminado y guardado."
        )

        gc.collect()

    # ========================================================
    # AUTOENCODER
    # ========================================================

    if not combinacion_ya_evaluada(
        resultados_testing,
        numero_falla,
        "Autoencoder"
    ):

        print("  → Autoencoder...")

        resultado_ae = evaluar_ae_testing(
            df=df_falla,
            scaler=scaler,
            autoencoder=autoencoder,
            feature_cols=feature_cols,
            AE_threshold=AE_threshold,
            muestra_inicio_falla=161,
            batch_size=2048
        )

        resultado_ae["falla"] = (
            numero_falla
        )

        resultados_testing.append(
            resultado_ae
        )

        pd.DataFrame(
            resultados_testing
        ).to_csv(
            RUTA_CHECKPOINT_TEST,
            index=False
        )

        print(
            "    Autoencoder terminado y guardado."
        )

        gc.collect()

    # --------------------------------------------------------
    # Liberar completamente esta falla
    # --------------------------------------------------------

    del df_falla

    gc.collect()

    print(
        f"Falla {numero_falla} completada."
    )

print()
print("=" * 60)
print("EVALUACIÓN FINAL TERMINADA")
print("=" * 60)

In [ ]:
resultados_finales_testing = pd.read_csv(
    RUTA_CHECKPOINT_TEST
)

resultados_finales_testing = (
    resultados_finales_testing
    .sort_values(
        [
            "falla",
            "Modelo"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Filas de resultados:",
    len(resultados_finales_testing)
)

print(
    "Fallas:",
    resultados_finales_testing[
        "falla"
    ].nunique()
)

print(
    "Modelos:",
    resultados_finales_testing[
        "Modelo"
    ].unique()
)

In [ ]:
tabla_deteccion_testing = (
    resultados_finales_testing
    .pivot(
        index="falla",
        columns="Modelo",
        values="tasa_deteccion_evaluable_pct"
    )
    .reset_index()
)

tabla_deteccion_testing.columns.name = None

display(
    tabla_deteccion_testing.round(3)
)

In [ ]:
tabla_retraso_testing = (
    resultados_finales_testing
    .pivot(
        index="falla",
        columns="Modelo",
        values="retraso_medio_min"
    )
    .reset_index()
)

tabla_retraso_testing.columns.name = None

display(
    tabla_retraso_testing.round(3)
)

In [ ]:
resumen_final_modelos = []

for modelo in [
    "PCA",
    "Isolation Forest",
    "Autoencoder"
]:

    datos_modelo = (
        resultados_finales_testing.loc[
            resultados_finales_testing["Modelo"]
            == modelo
        ]
        .copy()
    )

    total_evaluables = (
        datos_modelo[
            "corridas_evaluables"
        ].sum()
    )

    total_detectadas = (
        datos_modelo[
            "corridas_detectadas_evaluables"
        ].sum()
    )

    tasa_global = (
        total_detectadas
        / total_evaluables
        * 100
    )

    # Retraso ponderado por número de corridas detectadas
    retraso_global_ponderado = (
        (
            datos_modelo["retraso_medio_min"]
            *
            datos_modelo[
                "corridas_detectadas_evaluables"
            ]
        ).sum()
        /
        datos_modelo[
            "corridas_detectadas_evaluables"
        ].sum()
    )

    resumen_final_modelos.append({
        "Modelo": modelo,

        "Corridas evaluables":
            total_evaluables,

        "Corridas detectadas":
            total_detectadas,

        "Tasa global detección (%)":
            tasa_global,

        "Detección media entre fallas (%)":
            datos_modelo[
                "tasa_deteccion_evaluable_pct"
            ].mean(),

        "Mediana detección (%)":
            datos_modelo[
                "tasa_deteccion_evaluable_pct"
            ].median(),

        "Fallas detección >= 99 %":
            (
                datos_modelo[
                    "tasa_deteccion_evaluable_pct"
                ] >= 99
            ).sum(),

        "Retraso medio entre fallas (min)":
            datos_modelo[
                "retraso_medio_min"
            ].mean(),

        "Retraso ponderado por corridas (min)":
            retraso_global_ponderado,

        "Mediana retraso entre fallas (min)":
            datos_modelo[
                "retraso_medio_min"
            ].median()
    })


resumen_final_modelos = pd.DataFrame(
    resumen_final_modelos
)

display(
    resumen_final_modelos.round(3)
)

In [ ]:
ruta_test_normal = (
    RESULTS_DIR /
    "evaluacion_final_FaultFree_Testing.csv"
)

print("Ruta:", ruta_test_normal)
print("¿Existe?:", ruta_test_normal.exists())

In [ ]:
comparacion_test_normal = pd.read_csv(
    ruta_test_normal
)

display(
    comparacion_test_normal.round(4)
)

In [ ]:
resumen_final_completo = (
    resumen_final_modelos
    .merge(
        comparacion_test_normal[
            [
                "Modelo",
                "FAR_puntual_pct",
                "FAR_persistente_pct",
                "Episodios_por_100h",
                "Corridas_afectadas_pct",
                "Duracion_media_min"
            ]
        ],
        on="Modelo",
        how="left"
    )
)

display(
    resumen_final_completo.round(3)
)

In [ ]:
ruta_resumen_final = (
    RESULTS_DIR /
    "resumen_final_completo_modelos_testing.csv"
)

resumen_final_completo.to_csv(
    ruta_resumen_final,
    index=False
)

print(
    "Resumen final guardado en:",
    ruta_resumen_final
)

## 13. Precision–Recall y selección final

Cálculo de scores continuos, AP/AUC-PR por falla y consolidación de los criterios empleados durante la selección del detector principal.


In [ ]:
import pyreadr
import gc
from pathlib import Path
from shutil import copy2

archivo_ff_test_drive = (
    DATA_DIR /
    "TEP_FaultFree_Testing.RData"
)

archivo_ff_test_local = Path(
    "/content/TEP_FaultFree_Testing.RData"
)

if not archivo_ff_test_local.exists():

    copy2(
        archivo_ff_test_drive,
        archivo_ff_test_local
    )

resultado_ff_test = pyreadr.read_r(
    str(archivo_ff_test_local)
)

nombre_objeto_ff_test = next(
    iter(resultado_ff_test.keys())
)

ff_test_auc = (
    resultado_ff_test[
        nombre_objeto_ff_test
    ]
    .copy()
)

del resultado_ff_test
gc.collect()

print("Dimensiones:", ff_test_auc.shape)
print(
    "Corridas:",
    ff_test_auc["simulationRun"].nunique()
)

In [ ]:
X_normal_pca = (
    ff_test_auc[feature_cols]
)

X_normal_pca_scaled_array = (
    scaler.transform(
        X_normal_pca
    )
)

X_normal_pca_scaled = pd.DataFrame(
    X_normal_pca_scaled_array,
    columns=feature_cols,
    copy=False
)

scores_normal_pca = (
    pca_model.transform(
        X_normal_pca_scaled
    )
)

eigenvalues = (
    pca_model.explained_variance_
)

T2_normal = np.sum(
    (
        scores_normal_pca ** 2
    ) / eigenvalues,
    axis=1
)

X_normal_pca_rec = (
    pca_model.inverse_transform(
        scores_normal_pca
    )
)

SPE_normal = np.sum(
    (
        X_normal_pca_scaled_array
        - X_normal_pca_rec
    ) ** 2,
    axis=1
)

score_normal_pca = np.maximum(
    T2_normal / T2_threshold,
    SPE_normal / SPE_threshold
)

print(
    "Scores normales PCA:",
    score_normal_pca.shape
)

print(
    "Mediana:",
    np.median(score_normal_pca)
)

print(
    "P99:",
    np.quantile(score_normal_pca, 0.99)
)

In [ ]:
del X_normal_pca
del X_normal_pca_scaled_array
del X_normal_pca_scaled
del scores_normal_pca
del X_normal_pca_rec
del T2_normal
del SPE_normal

gc.collect()

In [ ]:
X_normal_if = (
    ff_test_auc[feature_cols]
)

score_normal_if = (
    -if_model.score_samples(
        X_normal_if
    )
)

print(
    "Scores normales IF:",
    score_normal_if.shape
)

print(
    "Mediana:",
    np.median(score_normal_if)
)

print(
    "P99:",
    np.quantile(score_normal_if, 0.99)
)

del X_normal_if
gc.collect()

In [ ]:
X_normal_ae = (
    ff_test_auc[feature_cols]
)

X_normal_ae_scaled = (
    scaler.transform(
        X_normal_ae
    )
)

X_normal_ae_rec = (
    autoencoder.predict(
        X_normal_ae_scaled,
        batch_size=2048,
        verbose=0
    )
)

score_normal_ae = np.mean(
    (
        X_normal_ae_scaled
        - X_normal_ae_rec
    ) ** 2,
    axis=1
)

print(
    "Scores normales AE:",
    score_normal_ae.shape
)

print(
    "Mediana:",
    np.median(score_normal_ae)
)

print(
    "P99:",
    np.quantile(score_normal_ae, 0.99)
)

del X_normal_ae
del X_normal_ae_scaled
del X_normal_ae_rec

gc.collect()

In [ ]:
def calcular_scores_falla_auc(
    df,
    feature_cols,
    scaler,
    pca_model,
    T2_threshold,
    SPE_threshold,
    if_model,
    autoencoder
):

    X = df[feature_cols]

    # ======================================================
    # PCA
    # ======================================================

    X_scaled = scaler.transform(X)

    X_scaled_df = pd.DataFrame(
        X_scaled,
        columns=feature_cols,
        copy=False
    )

    scores_pca = pca_model.transform(
        X_scaled_df
    )

    eigenvalues = (
        pca_model.explained_variance_
    )

    T2 = np.sum(
        (scores_pca ** 2)
        / eigenvalues,
        axis=1
    )

    X_rec_pca = (
        pca_model.inverse_transform(
            scores_pca
        )
    )

    SPE = np.sum(
        (
            X_scaled
            - X_rec_pca
        ) ** 2,
        axis=1
    )

    score_pca = np.maximum(
        T2 / T2_threshold,
        SPE / SPE_threshold
    )

    # ======================================================
    # ISOLATION FOREST
    # ======================================================

    score_if = (
        -if_model.score_samples(X)
    )

    # ======================================================
    # AUTOENCODER
    # ======================================================

    X_rec_ae = (
        autoencoder.predict(
            X_scaled,
            batch_size=2048,
            verbose=0
        )
    )

    score_ae = np.mean(
        (
            X_scaled
            - X_rec_ae
        ) ** 2,
        axis=1
    )

    # Liberar auxiliares
    del X_scaled_df
    del scores_pca
    del X_rec_pca
    del T2
    del SPE
    del X_rec_ae

    gc.collect()

    return (
        score_pca,
        score_if,
        score_ae
    )

In [ ]:
falla_auc_1 = pd.read_parquet(
    PARTITION_DIR /
    "falla_01"
)

falla_auc_1 = (
    falla_auc_1
    .sort_values(
        ["simulationRun", "sample"]
    )
    .reset_index(drop=True)
)

# Solo periodo realmente post-falla
falla_auc_1_post = (
    falla_auc_1.loc[
        falla_auc_1["sample"] >= 161
    ]
    .reset_index(drop=True)
)

print(
    "Muestras post-falla:",
    len(falla_auc_1_post)
)

In [ ]:
score_f1_pca, \
score_f1_if, \
score_f1_ae = calcular_scores_falla_auc(
    df=falla_auc_1_post,
    feature_cols=feature_cols,
    scaler=scaler,
    pca_model=pca_model,
    T2_threshold=T2_threshold,
    SPE_threshold=SPE_threshold,
    if_model=if_model,
    autoencoder=autoencoder
)

In [ ]:
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    auc
)

In [ ]:
def calcular_metricas_pr(
    score_normal,
    score_falla
):

    y_true = np.concatenate([
        np.zeros(
            len(score_normal),
            dtype=np.int8
        ),
        np.ones(
            len(score_falla),
            dtype=np.int8
        )
    ])

    y_score = np.concatenate([
        score_normal,
        score_falla
    ])

    precision, recall, _ = (
        precision_recall_curve(
            y_true,
            y_score
        )
    )

    average_precision = (
        average_precision_score(
            y_true,
            y_score
        )
    )

    auc_pr = auc(
        recall,
        precision
    )

    prevalencia = (
        y_true.mean()
    )

    return {
        "AP": average_precision,
        "AUC_PR": auc_pr,
        "Prevalencia": prevalencia
    }

In [ ]:
pr_pca_f1 = calcular_metricas_pr(
    score_normal_pca,
    score_f1_pca
)

pr_if_f1 = calcular_metricas_pr(
    score_normal_if,
    score_f1_if
)

pr_ae_f1 = calcular_metricas_pr(
    score_normal_ae,
    score_f1_ae
)

tabla_pr_f1 = pd.DataFrame([
    {
        "Modelo": "PCA",
        **pr_pca_f1
    },
    {
        "Modelo": "Isolation Forest",
        **pr_if_f1
    },
    {
        "Modelo": "Autoencoder",
        **pr_ae_f1
    }
])

display(
    tabla_pr_f1.round(4)
)

In [ ]:
RUTA_PR_CHECKPOINT = (
    RESULTS_DIR /
    "metricas_PR_Faulty_Testing_checkpoint.csv"
)

# Recuperar avance previo si existe
if RUTA_PR_CHECKPOINT.exists():

    resultados_pr = (
        pd.read_csv(
            RUTA_PR_CHECKPOINT
        )
        .to_dict("records")
    )

    fallas_ya_procesadas = set(
        int(x["falla"])
        for x in resultados_pr
    )

    print(
        "Checkpoint encontrado."
    )

    print(
        "Fallas ya procesadas:",
        sorted(fallas_ya_procesadas)
    )

else:

    resultados_pr = []
    fallas_ya_procesadas = set()

    print(
        "No existe checkpoint previo."
    )

In [ ]:
for numero_falla in range(1, 21):

    if numero_falla in fallas_ya_procesadas:

        print(
            f"Falla {numero_falla} ya procesada. "
            "Se omite."
        )

        continue

    print()
    print("=" * 55)
    print(
        f"PRECISION-RECALL | FALLA "
        f"{numero_falla} DE 20"
    )
    print("=" * 55)

    # ========================================================
    # 1. Cargar solo esta falla
    # ========================================================

    df_falla = pd.read_parquet(
        PARTITION_DIR /
        f"falla_{numero_falla:02d}"
    )

    df_falla = (
        df_falla
        .sort_values(
            ["simulationRun", "sample"]
        )
        .reset_index(drop=True)
    )

    # ========================================================
    # 2. Solo periodo post-falla
    # ========================================================

    df_post = (
        df_falla.loc[
            df_falla["sample"] >= 161
        ]
        .reset_index(drop=True)
    )

    print(
        "Muestras post-falla:",
        len(df_post)
    )

    # Deben ser 400.000
    if len(df_post) != 400_000:

        raise ValueError(
            f"La falla {numero_falla} tiene "
            f"{len(df_post)} muestras post-falla, "
            "pero se esperaban 400000."
        )

    # ========================================================
    # 3. Scores de los tres modelos
    # ========================================================

    score_pca, score_if, score_ae = (
        calcular_scores_falla_auc(
            df=df_post,
            feature_cols=feature_cols,
            scaler=scaler,
            pca_model=pca_model,
            T2_threshold=T2_threshold,
            SPE_threshold=SPE_threshold,
            if_model=if_model,
            autoencoder=autoencoder
        )
    )

    # ========================================================
    # 4. Precision-Recall
    # ========================================================

    metricas_pca = calcular_metricas_pr(
        score_normal_pca,
        score_pca
    )

    metricas_if = calcular_metricas_pr(
        score_normal_if,
        score_if
    )

    metricas_ae = calcular_metricas_pr(
        score_normal_ae,
        score_ae
    )

    # ========================================================
    # 5. Guardar resultados
    # ========================================================

    resultados_pr.append({
        "falla": numero_falla,

        "AP_PCA":
            metricas_pca["AP"],

        "AUC_PR_PCA":
            metricas_pca["AUC_PR"],

        "AP_IF":
            metricas_if["AP"],

        "AUC_PR_IF":
            metricas_if["AUC_PR"],

        "AP_AE":
            metricas_ae["AP"],

        "AUC_PR_AE":
            metricas_ae["AUC_PR"],

        "Prevalencia":
            metricas_pca["Prevalencia"]
    })

    # Checkpoint inmediato
    pd.DataFrame(
        resultados_pr
    ).to_csv(
        RUTA_PR_CHECKPOINT,
        index=False
    )

    print(
        f"PCA AP={metricas_pca['AP']:.4f} | "
        f"IF AP={metricas_if['AP']:.4f} | "
        f"AE AP={metricas_ae['AP']:.4f}"
    )

    # ========================================================
    # 6. Liberar memoria
    # ========================================================

    del df_falla
    del df_post

    del score_pca
    del score_if
    del score_ae

    gc.collect()

print()
print("Evaluación Precision-Recall terminada.")

In [ ]:
tabla_pr_20 = (
    pd.read_csv(
        RUTA_PR_CHECKPOINT
    )
    .sort_values("falla")
    .reset_index(drop=True)
)

display(
    tabla_pr_20.round(4)
)

In [ ]:
print(
    "Fallas evaluadas:",
    tabla_pr_20["falla"].nunique()
)

print(
    "Prevalencia:",
    tabla_pr_20["Prevalencia"].unique()
)

In [ ]:
resumen_pr_modelos = pd.DataFrame({
    "Modelo": [
        "PCA",
        "Isolation Forest",
        "Autoencoder"
    ],

    "AP_media": [
        tabla_pr_20["AP_PCA"].mean(),
        tabla_pr_20["AP_IF"].mean(),
        tabla_pr_20["AP_AE"].mean()
    ],

    "AP_mediana": [
        tabla_pr_20["AP_PCA"].median(),
        tabla_pr_20["AP_IF"].median(),
        tabla_pr_20["AP_AE"].median()
    ],

    "AP_minima": [
        tabla_pr_20["AP_PCA"].min(),
        tabla_pr_20["AP_IF"].min(),
        tabla_pr_20["AP_AE"].min()
    ],

    "AUC_PR_media": [
        tabla_pr_20["AUC_PR_PCA"].mean(),
        tabla_pr_20["AUC_PR_IF"].mean(),
        tabla_pr_20["AUC_PR_AE"].mean()
    ]
})

display(
    resumen_pr_modelos.round(4)
)

In [ ]:
tabla_pr_20.to_csv(
    RESULTS_DIR /
    "PrecisionRecall_20_fallas_Testing.csv",
    index=False
)

resumen_pr_modelos.to_csv(
    RESULTS_DIR /
    "Resumen_PrecisionRecall_modelos.csv",
    index=False
)

print(
    "Resultados Precision-Recall guardados."
)

In [ ]:
tabla_ap = tabla_pr_20[
    [
        "falla",
        "AP_PCA",
        "AP_IF",
        "AP_AE"
    ]
].copy()

display(
    tabla_ap.round(4)
)

In [ ]:
tabla_seleccion_final = (
    resumen_final_completo
    .merge(
        resumen_pr_modelos[
            [
                "Modelo",
                "AP_media",
                "AP_mediana",
                "AP_minima"
            ]
        ],
        on="Modelo",
        how="left"
    )
)

columnas_seleccion = [
    "Modelo",
    "Tasa global detección (%)",
    "Fallas detección >= 99 %",
    "Retraso ponderado por corridas (min)",
    "FAR_persistente_pct",
    "Episodios_por_100h",
    "AP_media",
    "AP_mediana"
]

tabla_seleccion_final = (
    tabla_seleccion_final[
        columnas_seleccion
    ]
)

display(
    tabla_seleccion_final.round(3)
)

In [ ]:
tabla_seleccion_final.to_csv(
    RESULTS_DIR /
    "seleccion_final_modelo_TFM.csv",
    index=False
)

print(
    "Tabla de selección final guardada."
)

## 14. Prototipo final de alerta temprana basado en PCA

Procesamiento de una corrida, visualización temporal, confirmación 3-de-5 e interpretación mediante contribuciones PCA.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc

PARTITION_DIR = (
    PROJECT_DIR
    / "Datos_Procesados"
    / "Faulty_Testing_Parquet_v1"
)

print("Ruta:", PARTITION_DIR)
print("¿Existe?:", PARTITION_DIR.exists())

In [ ]:
def procesar_corrida_prototipo_pca(
    numero_falla,
    numero_corrida,
    partition_dir,
    feature_cols,
    scaler,
    pca_model,
    T2_threshold,
    SPE_threshold,
    muestra_inicio_falla=161,
    ventana=5,
    minimo_alarmas=3,
    minutos_por_muestra=3
):
    """
    Procesa una corrida individual del conjunto Faulty_Testing
    utilizando el detector PCA final del TFM.

    Devuelve:
    - evolución temporal de T², SPE y score normalizado;
    - alarmas puntuales;
    - alarma persistente 3-de-5;
    - primera alarma posterior a la falla;
    - retraso de detección;
    - información general de la corrida.
    """

    # =========================================================
    # 1. Cargar únicamente la falla seleccionada
    # =========================================================

    ruta_falla = (
        partition_dir
        / f"falla_{numero_falla:02d}"
    )

    if not ruta_falla.exists():
        raise FileNotFoundError(
            f"No existe la partición de la falla {numero_falla}."
        )

    df_falla = pd.read_parquet(
        ruta_falla
    )

    # =========================================================
    # 2. Seleccionar una única corrida
    # =========================================================

    df = (
        df_falla.loc[
            df_falla["simulationRun"].astype(int)
            == int(numero_corrida)
        ]
        .copy()
        .sort_values("sample")
        .reset_index(drop=True)
    )

    del df_falla
    gc.collect()

    if df.empty:
        raise ValueError(
            f"No existe la corrida {numero_corrida} "
            f"para la falla {numero_falla}."
        )

    # Verificación estructural
    if len(df) != 960:
        print(
            f"Advertencia: la corrida contiene {len(df)} "
            "muestras y se esperaban 960."
        )

    # =========================================================
    # 3. Variables de proceso
    # =========================================================

    X = df[
        feature_cols
    ]

    # =========================================================
    # 4. Estandarización
    # =========================================================

    X_scaled_array = scaler.transform(
        X
    )

    X_scaled = pd.DataFrame(
        X_scaled_array,
        columns=feature_cols,
        index=df.index
    )

    # =========================================================
    # 5. Proyección PCA
    # =========================================================

    scores = pca_model.transform(
        X_scaled
    )

    eigenvalues = (
        pca_model.explained_variance_
    )

    # =========================================================
    # 6. Hotelling T²
    # =========================================================

    T2 = np.sum(
        (scores ** 2)
        / eigenvalues,
        axis=1
    )

    # =========================================================
    # 7. SPE / Q
    # =========================================================

    X_reconstructed = (
        pca_model.inverse_transform(
            scores
        )
    )

    residuals = (
        X_scaled_array
        - X_reconstructed
    )

    SPE = np.sum(
        residuals ** 2,
        axis=1
    )

    # =========================================================
    # 8. Score único normalizado
    # =========================================================
    #
    # score > 1 significa que T² o SPE superó su límite.
    # =========================================================

    score_pca = np.maximum(
        T2 / T2_threshold,
        SPE / SPE_threshold
    )

    # =========================================================
    # 9. Tabla temporal
    # =========================================================

    resultados = df[
        [
            "faultNumber",
            "simulationRun",
            "sample"
        ]
    ].copy()

    resultados["tiempo_h"] = (
        (resultados["sample"] - 1)
        * minutos_por_muestra
        / 60
    )

    resultados["T2"] = T2
    resultados["SPE"] = SPE
    resultados["score_pca"] = score_pca

    # =========================================================
    # 10. Alarma puntual
    # =========================================================

    resultados["alarma_puntual"] = (
        resultados["score_pca"] > 1
    )

    # =========================================================
    # 11. Regla temporal 3-de-5
    # =========================================================

    resultados["alarmas_ventana"] = (
        resultados["alarma_puntual"]
        .astype(int)
        .rolling(
            window=ventana,
            min_periods=ventana
        )
        .sum()
    )

    resultados["alarma_persistente"] = (
        resultados["alarmas_ventana"]
        >= minimo_alarmas
    )

    # =========================================================
    # 12. Inicio de cada episodio
    # =========================================================

    resultados["inicio_episodio"] = (
        resultados["alarma_persistente"]
        &
        ~resultados[
            "alarma_persistente"
        ].shift(fill_value=False)
    )

    # =========================================================
    # 13. ¿Existe una prealarma?
    # =========================================================

    episodios_previos = resultados.loc[
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            < muestra_inicio_falla
        )
    ]

    existe_prealarma = (
        len(episodios_previos) > 0
    )

    # =========================================================
    # 14. Primera detección posterior a la falla
    # =========================================================

    detecciones_post = resultados.loc[
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            >= muestra_inicio_falla
        )
    ]

    if len(detecciones_post) > 0:

        muestra_deteccion = int(
            detecciones_post.iloc[0][
                "sample"
            ]
        )

        retraso_min = (
            muestra_deteccion
            - muestra_inicio_falla
        ) * minutos_por_muestra

        detectada = True

    else:

        muestra_deteccion = None
        retraso_min = None
        detectada = False

    # =========================================================
    # 15. Estado final del sistema
    # =========================================================

    if detectada:
        estado = "ALERTA CONFIRMADA"
    else:
        estado = "SIN ALERTA CONFIRMADA"

    # =========================================================
    # 16. Resumen
    # =========================================================

    resumen = {
        "Falla": int(numero_falla),
        "Corrida": int(numero_corrida),

        "Muestra_inicio_falla":
            int(muestra_inicio_falla),

        "Tiempo_inicio_falla_h":
            (
                (muestra_inicio_falla - 1)
                * minutos_por_muestra
                / 60
            ),

        "Existe_prealarma":
            bool(existe_prealarma),

        "Detectada":
            bool(detectada),

        "Muestra_deteccion":
            muestra_deteccion,

        "Retraso_deteccion_min":
            retraso_min,

        "Estado":
            estado
    }

    # =========================================================
    # 17. Guardar información necesaria para interpretación
    # =========================================================

    datos_interpretacion = {
        "df_original": df,
        "X_scaled": X_scaled,
        "scores": scores,
        "residuals": residuals
    }

    return (
        resultados,
        resumen,
        datos_interpretacion
    )

In [ ]:
resultados_proto, \
resumen_proto, \
datos_proto = procesar_corrida_prototipo_pca(
    numero_falla=1,
    numero_corrida=1,
    partition_dir=PARTITION_DIR,
    feature_cols=feature_cols,
    scaler=scaler,
    pca_model=pca_model,
    T2_threshold=T2_threshold,
    SPE_threshold=SPE_threshold,
    muestra_inicio_falla=161,
    ventana=5,
    minimo_alarmas=3,
    minutos_por_muestra=3
)

In [ ]:
display(
    pd.DataFrame(
        [resumen_proto]
    )
)

In [ ]:
display(
    resultados_proto.loc[
        (
            resultados_proto["sample"]
            >= 155
        )
        &
        (
            resultados_proto["sample"]
            <= 175
        ),
        [
            "sample",
            "tiempo_h",
            "T2",
            "SPE",
            "score_pca",
            "alarma_puntual",
            "alarmas_ventana",
            "alarma_persistente",
            "inicio_episodio"
        ]
    ].round(3)
)

In [ ]:
import matplotlib.pyplot as plt


def visualizar_alerta_temprana_pca(
    resultados,
    resumen,
    mostrar_alarmas_puntuales=True
):
    """
    Visualiza el comportamiento temporal del sistema
    de alerta temprana basado en PCA.
    """

    # ---------------------------------------------------------
    # Información general
    # ---------------------------------------------------------

    falla = resumen["Falla"]
    corrida = resumen["Corrida"]

    muestra_inicio_falla = (
        resumen["Muestra_inicio_falla"]
    )

    tiempo_inicio_falla = (
        resumen["Tiempo_inicio_falla_h"]
    )

    detectada = resumen["Detectada"]

    muestra_deteccion = (
        resumen["Muestra_deteccion"]
    )

    retraso = (
        resumen["Retraso_deteccion_min"]
    )

    # ---------------------------------------------------------
    # Figura
    # ---------------------------------------------------------

    plt.figure(
        figsize=(12, 6)
    )

    # Score temporal
    plt.plot(
        resultados["tiempo_h"],
        resultados["score_pca"],
        label="Score de anomalía PCA",
        linewidth=1.5
    )

    # Umbral
    plt.axhline(
        y=1,
        linestyle="--",
        linewidth=1.5,
        label="Umbral de anomalía"
    )

    # Inicio real de la falla
    plt.axvline(
        x=tiempo_inicio_falla,
        linestyle="--",
        linewidth=1.5,
        label="Inicio de la falla"
    )

    # ---------------------------------------------------------
    # Primera alarma confirmada
    # ---------------------------------------------------------

    if detectada:

        tiempo_deteccion_h = (
            (muestra_deteccion - 1)
            * 3
            / 60
        )

        plt.axvline(
            x=tiempo_deteccion_h,
            linestyle="-.",
            linewidth=1.5,
            label=(
                f"Alarma confirmada "
                f"({retraso} min)"
            )
        )

    # ---------------------------------------------------------
    # Alarmas puntuales
    # ---------------------------------------------------------

    if mostrar_alarmas_puntuales:

        puntos_alarma = (
            resultados.loc[
                resultados["alarma_puntual"]
            ]
        )

        plt.scatter(
            puntos_alarma["tiempo_h"],
            puntos_alarma["score_pca"],
            s=12,
            label="Alarma puntual"
        )

    # ---------------------------------------------------------
    # Formato
    # ---------------------------------------------------------

    plt.xlabel(
        "Tiempo de simulación (h)"
    )

    plt.ylabel(
        "Score de anomalía normalizado"
    )

    plt.title(
        f"Sistema de alerta temprana PCA "
        f"– Falla {falla}, corrida {corrida}"
    )

    plt.legend()

    plt.grid(
        True,
        alpha=0.3
    )

    plt.tight_layout()

    plt.show()

In [ ]:
visualizar_alerta_temprana_pca(
    resultados=resultados_proto,
    resumen=resumen_proto
)

In [ ]:
def visualizar_zoom_deteccion_pca(
    resultados,
    resumen,
    minutos_antes=30,
    minutos_despues=60
):
    """
    Amplía la región temporal alrededor del inicio de la falla.
    """

    muestra_inicio = (
        resumen["Muestra_inicio_falla"]
    )

    muestras_antes = int(
        minutos_antes / 3
    )

    muestras_despues = int(
        minutos_despues / 3
    )

    muestra_min = max(
        1,
        muestra_inicio - muestras_antes
    )

    muestra_max = min(
        resultados["sample"].max(),
        muestra_inicio + muestras_despues
    )

    zona = resultados.loc[
        (
            resultados["sample"]
            >= muestra_min
        )
        &
        (
            resultados["sample"]
            <= muestra_max
        )
    ].copy()

    plt.figure(
        figsize=(11, 6)
    )

    plt.plot(
        zona["tiempo_h"],
        zona["score_pca"],
        marker="o",
        markersize=4,
        label="Score PCA"
    )

    plt.axhline(
        y=1,
        linestyle="--",
        label="Umbral"
    )

    # Inicio de falla
    plt.axvline(
        x=resumen["Tiempo_inicio_falla_h"],
        linestyle="--",
        label="Inicio de falla"
    )

    # Detección
    if resumen["Detectada"]:

        tiempo_deteccion = (
            (
                resumen["Muestra_deteccion"]
                - 1
            )
            * 3
            / 60
        )

        plt.axvline(
            x=tiempo_deteccion,
            linestyle="-.",
            label=(
                "Alarma confirmada "
                f"({resumen['Retraso_deteccion_min']} min)"
            )
        )

    puntos_alarma = zona.loc[
        zona["alarma_puntual"]
    ]

    plt.scatter(
        puntos_alarma["tiempo_h"],
        puntos_alarma["score_pca"],
        s=40,
        label="Alarma puntual"
    )

    plt.xlabel(
        "Tiempo de simulación (h)"
    )

    plt.ylabel(
        "Score de anomalía normalizado"
    )

    plt.title(
        f"Detección temprana – "
        f"Falla {resumen['Falla']}, "
        f"corrida {resumen['Corrida']}"
    )

    plt.legend()

    plt.grid(
        True,
        alpha=0.3
    )

    plt.tight_layout()

    plt.show()

In [ ]:
visualizar_zoom_deteccion_pca(
    resultados_proto,
    resumen_proto
)

In [ ]:
def mostrar_estado_prototipo(resumen):

    if resumen["Detectada"]:

        estado_operativo = (
            "ALERTA CONFIRMADA"
        )

        mensaje = (
            f"Falla detectada en la muestra "
            f"{resumen['Muestra_deteccion']} "
            f"con un retraso de "
            f"{resumen['Retraso_deteccion_min']} min."
        )

    else:

        estado_operativo = (
            "SIN ALERTA CONFIRMADA"
        )

        mensaje = (
            "No se confirmó una condición anómala "
            "durante la corrida."
        )

    panel = pd.DataFrame({
        "Indicador": [
            "Falla simulada",
            "Corrida",
            "Estado del sistema",
            "Inicio de la falla",
            "Muestra de detección",
            "Retraso de detección",
            "Prealarma antes de la falla"
        ],

        "Valor": [
            resumen["Falla"],
            resumen["Corrida"],
            estado_operativo,
            (
                f"{resumen['Tiempo_inicio_falla_h']:.2f} h"
            ),
            (
                resumen["Muestra_deteccion"]
                if resumen["Detectada"]
                else "No detectada"
            ),
            (
                f"{resumen['Retraso_deteccion_min']} min"
                if resumen["Detectada"]
                else "N.A."
            ),
            (
                "Sí"
                if resumen["Existe_prealarma"]
                else "No"
            )
        ]
    })

    display(panel)

    print()
    print(mensaje)

In [ ]:
mostrar_estado_prototipo(
    resumen_proto
)

In [ ]:
nombres_variables_tep = {

    # ========================================================
    # VARIABLES MEDIDAS XMEAS
    # ========================================================

    "xmeas_1": "Caudal alimentación A - corriente 1",
    "xmeas_2": "Caudal alimentación D - corriente 2",
    "xmeas_3": "Caudal alimentación E - corriente 3",
    "xmeas_4": "Caudal alimentación A/C - corriente 4",
    "xmeas_5": "Caudal de recirculación",
    "xmeas_6": "Caudal de alimentación al reactor",

    "xmeas_7": "Presión del reactor",
    "xmeas_8": "Nivel del reactor",
    "xmeas_9": "Temperatura del reactor",

    "xmeas_10": "Caudal de purga",

    "xmeas_11": "Temperatura del separador",
    "xmeas_12": "Nivel del separador",
    "xmeas_13": "Presión del separador",
    "xmeas_14": "Caudal de salida del separador",

    "xmeas_15": "Nivel del stripper",
    "xmeas_16": "Presión del stripper",
    "xmeas_17": "Caudal de salida del stripper",
    "xmeas_18": "Temperatura del stripper",
    "xmeas_19": "Caudal de vapor al stripper",

    "xmeas_20": "Trabajo del compresor",

    "xmeas_21": "T salida agua enfriamiento reactor",
    "xmeas_22": "T salida agua enfriamiento separador",

    # Composición alimentación al reactor
    "xmeas_23": "Componente A - alimentación reactor",
    "xmeas_24": "Componente B - alimentación reactor",
    "xmeas_25": "Componente C - alimentación reactor",
    "xmeas_26": "Componente D - alimentación reactor",
    "xmeas_27": "Componente E - alimentación reactor",
    "xmeas_28": "Componente F - alimentación reactor",

    # Composición gas de purga
    "xmeas_29": "Componente A - gas de purga",
    "xmeas_30": "Componente B - gas de purga",
    "xmeas_31": "Componente C - gas de purga",
    "xmeas_32": "Componente D - gas de purga",
    "xmeas_33": "Componente E - gas de purga",
    "xmeas_34": "Componente F - gas de purga",
    "xmeas_35": "Componente G - gas de purga",
    "xmeas_36": "Componente H - gas de purga",

    # Composición producto
    "xmeas_37": "Componente D - producto",
    "xmeas_38": "Componente E - producto",
    "xmeas_39": "Componente F - producto",
    "xmeas_40": "Componente G - producto",
    "xmeas_41": "Componente H - producto",

    # ========================================================
    # VARIABLES MANIPULADAS XMV
    # ========================================================

    "xmv_1": "Válvula alimentación D",
    "xmv_2": "Válvula alimentación E",
    "xmv_3": "Válvula alimentación A",
    "xmv_4": "Válvula alimentación A/C",

    "xmv_5": "Válvula recirculación compresor",
    "xmv_6": "Válvula de purga",

    "xmv_7": "Válvula salida separador",
    "xmv_8": "Válvula producto stripper",

    "xmv_9": "Válvula vapor stripper",

    "xmv_10": "Agua de enfriamiento reactor",
    "xmv_11": "Agua de enfriamiento condensador"
}

In [ ]:
def interpretar_alerta_prototipo_pca(
    datos_interpretacion,
    resumen,
    feature_cols,
    pca_model,
    T2_threshold,
    SPE_threshold,
    nombres_variables,
    top_n=10
):
    """
    Interpreta automáticamente la primera alarma confirmada
    mediante contribuciones PCA.

    Devuelve:
    - estadístico dominante
    - contribuciones T²
    - contribuciones SPE/Q
    - tabla con las principales variables
    """

    if not resumen["Detectada"]:

        print(
            "La corrida no presenta una alarma confirmada. "
            "No se genera interpretación."
        )

        return None

    # ========================================================
    # 1. Muestra donde se confirma la alarma
    # ========================================================

    muestra = int(
        resumen["Muestra_deteccion"]
    )

    df_original = (
        datos_interpretacion[
            "df_original"
        ]
    )

    X_scaled = (
        datos_interpretacion[
            "X_scaled"
        ]
    )

    scores = (
        datos_interpretacion[
            "scores"
        ]
    )

    residuals = (
        datos_interpretacion[
            "residuals"
        ]
    )

    # Posición de la muestra dentro de la corrida
    posiciones = np.where(
        df_original["sample"]
        .to_numpy()
        == muestra
    )[0]

    if len(posiciones) != 1:

        raise ValueError(
            "No fue posible identificar de forma única "
            "la muestra de detección."
        )

    posicion = posiciones[0]

    # ========================================================
    # 2. Datos PCA de esa muestra
    # ========================================================

    x = (
        X_scaled.iloc[
            posicion
        ]
        .to_numpy()
    )

    t = scores[
        posicion
    ]

    residual = residuals[
        posicion
    ]

    eigenvalues = (
        pca_model.explained_variance_
    )

    # ========================================================
    # 3. T² y SPE
    # ========================================================

    T2 = np.sum(
        (t ** 2)
        / eigenvalues
    )

    SPE = np.sum(
        residual ** 2
    )

    ratio_T2 = (
        T2 / T2_threshold
    )

    ratio_SPE = (
        SPE / SPE_threshold
    )

    # ========================================================
    # 4. Estadístico dominante
    # ========================================================

    if ratio_T2 >= ratio_SPE:

        estadistico_dominante = "Hotelling T²"

    else:

        estadistico_dominante = "SPE/Q"

    # ========================================================
    # 5. Contribuciones T²
    # ========================================================

    P = (
        pca_model.components_.T
    )

    vector_T2 = (
        P @ (
            t / eigenvalues
        )
    )

    contrib_T2 = (
        x * vector_T2
    )

    # ========================================================
    # 6. Contribuciones SPE
    # ========================================================

    contrib_SPE = (
        residual ** 2
    )

    # ========================================================
    # 7. Tabla completa
    # ========================================================

    tabla_contribuciones = pd.DataFrame({

        "variable":
            feature_cols,

        "nombre":
            [
                nombres_variables.get(
                    variable,
                    variable
                )
                for variable in feature_cols
            ],

        "contrib_T2":
            contrib_T2,

        "abs_contrib_T2":
            np.abs(
                contrib_T2
            ),

        "contrib_SPE":
            contrib_SPE
    })

    # ========================================================
    # 8. Elegir contribución relevante para esta alarma
    # ========================================================

    if estadistico_dominante == "SPE/Q":

        tabla_top = (
            tabla_contribuciones
            .sort_values(
                "contrib_SPE",
                ascending=False
            )
            .head(top_n)
            .copy()
        )

        tabla_top[
            "contribucion_alarma"
        ] = (
            tabla_top[
                "contrib_SPE"
            ]
        )

    else:

        tabla_top = (
            tabla_contribuciones
            .sort_values(
                "abs_contrib_T2",
                ascending=False
            )
            .head(top_n)
            .copy()
        )

        tabla_top[
            "contribucion_alarma"
        ] = (
            tabla_top[
                "contrib_T2"
            ]
        )

    # ========================================================
    # 9. Resumen de la alarma
    # ========================================================

    resumen_interpretacion = {
        "Muestra": muestra,

        "T2":
            T2,

        "Umbral_T2":
            T2_threshold,

        "T2_normalizado":
            ratio_T2,

        "SPE":
            SPE,

        "Umbral_SPE":
            SPE_threshold,

        "SPE_normalizado":
            ratio_SPE,

        "Estadistico_dominante":
            estadistico_dominante
    }

    return (
        tabla_contribuciones,
        tabla_top,
        resumen_interpretacion
    )

In [ ]:
contribuciones_proto, \
top_variables_proto, \
resumen_interpretacion_proto = (
    interpretar_alerta_prototipo_pca(
        datos_interpretacion=datos_proto,
        resumen=resumen_proto,
        feature_cols=feature_cols,
        pca_model=pca_model,
        T2_threshold=T2_threshold,
        SPE_threshold=SPE_threshold,
        nombres_variables=nombres_variables_tep,
        top_n=10
    )
)

In [ ]:
display(
    pd.DataFrame(
        [resumen_interpretacion_proto]
    ).round(3)
)

In [ ]:
display(
    top_variables_proto[
        [
            "variable",
            "nombre",
            "contribucion_alarma"
        ]
    ].round(4)
)

In [ ]:
def visualizar_variables_alerta_pca(
    tabla_top,
    resumen_interpretacion
):

    estadistico = (
        resumen_interpretacion[
            "Estadistico_dominante"
        ]
    )

    muestra = (
        resumen_interpretacion[
            "Muestra"
        ]
    )

    datos_plot = (
        tabla_top
        .sort_values(
            "contribucion_alarma",
            key=lambda x: np.abs(x)
        )
    )

    plt.figure(
        figsize=(10, 6)
    )

    plt.barh(
        datos_plot["nombre"],
        datos_plot[
            "contribucion_alarma"
        ]
    )

    if estadistico == "Hotelling T²":

        plt.axvline(
            x=0,
            linewidth=1
        )

    plt.xlabel(
        f"Contribución a {estadistico}"
    )

    plt.ylabel(
        "Variable de proceso"
    )

    plt.title(
        f"Variables asociadas a la alerta "
        f"– {estadistico}, muestra {muestra}"
    )

    plt.tight_layout()

    plt.show()

In [ ]:
def ejecutar_prototipo(
    falla,
    corrida,
    mostrar_vista_completa=True,
    mostrar_zoom=True,
    mostrar_interpretacion=True,
    top_n=10
):
    """
    Ejecuta el prototipo final de alerta temprana basado en PCA.

    Para una falla y corrida seleccionadas:
    1. Procesa las 52 variables del proceso.
    2. Calcula T², SPE/Q y score de anomalía.
    3. Aplica la regla temporal 3-de-5.
    4. Identifica la primera alarma confirmada.
    5. Calcula el retraso de detección.
    6. Muestra la evolución temporal.
    7. Interpreta la alarma mediante contribuciones PCA.

    Parameters
    ----------
    falla : int
        Número de falla del Tennessee Eastman Process (1-20).

    corrida : int
        Número de corrida o simulationRun (1-500).

    mostrar_vista_completa : bool
        Mostrar evolución completa de la corrida.

    mostrar_zoom : bool
        Mostrar ampliación alrededor del inicio de la falla.

    mostrar_interpretacion : bool
        Mostrar variables responsables de la alarma.

    top_n : int
        Número de variables principales para la interpretación.

    Returns
    -------
    dict
        Resultados del prototipo.
    """

    # ========================================================
    # 0. Validaciones
    # ========================================================

    if not 1 <= int(falla) <= 20:
        raise ValueError(
            "La falla debe estar entre 1 y 20."
        )

    if not 1 <= int(corrida) <= 500:
        raise ValueError(
            "La corrida debe estar entre 1 y 500."
        )

    print("=" * 70)
    print("SISTEMA DE ALERTA TEMPRANA - PCA")
    print("=" * 70)

    print(
        f"Falla seleccionada: {falla}"
    )

    print(
        f"Corrida seleccionada: {corrida}"
    )

    print()

    # ========================================================
    # 1. Procesar corrida
    # ========================================================

    resultados, resumen, datos_interpretacion = (
        procesar_corrida_prototipo_pca(
            numero_falla=falla,
            numero_corrida=corrida,
            partition_dir=PARTITION_DIR,
            feature_cols=feature_cols,
            scaler=scaler,
            pca_model=pca_model,
            T2_threshold=T2_threshold,
            SPE_threshold=SPE_threshold,
            muestra_inicio_falla=161,
            ventana=5,
            minimo_alarmas=3,
            minutos_por_muestra=3
        )
    )

    # ========================================================
    # 2. Panel de estado
    # ========================================================

    print("\nESTADO DEL SISTEMA")
    print("-" * 70)

    mostrar_estado_prototipo(
        resumen
    )

    # ========================================================
    # 3. Visualización temporal completa
    # ========================================================

    if mostrar_vista_completa:

        print("\nEVOLUCIÓN TEMPORAL")
        print("-" * 70)

        visualizar_alerta_temprana_pca(
            resultados=resultados,
            resumen=resumen,
            mostrar_alarmas_puntuales=True
        )

    # ========================================================
    # 4. Zoom alrededor del inicio de falla
    # ========================================================

    if mostrar_zoom:

        print("\nDETALLE DE LA DETECCIÓN")
        print("-" * 70)

        visualizar_zoom_deteccion_pca(
            resultados=resultados,
            resumen=resumen,
            minutos_antes=30,
            minutos_despues=60
        )

    # ========================================================
    # 5. Interpretabilidad
    # ========================================================

    tabla_contribuciones = None
    top_variables = None
    resumen_interpretacion = None

    if mostrar_interpretacion:

        print("\nINTERPRETACIÓN DE LA ALERTA")
        print("-" * 70)

        if resumen["Detectada"]:

            (
                tabla_contribuciones,
                top_variables,
                resumen_interpretacion
            ) = interpretar_alerta_prototipo_pca(
                datos_interpretacion=datos_interpretacion,
                resumen=resumen,
                feature_cols=feature_cols,
                pca_model=pca_model,
                T2_threshold=T2_threshold,
                SPE_threshold=SPE_threshold,
                nombres_variables=nombres_variables_tep,
                top_n=top_n
            )

            # -----------------------------------------------
            # Resumen del estadístico que origina la alarma
            # -----------------------------------------------

            display(
                pd.DataFrame(
                    [resumen_interpretacion]
                ).round(3)
            )

            # -----------------------------------------------
            # Variables principales
            # -----------------------------------------------

            print(
                "\nPrincipales variables asociadas "
                "a la alerta:"
            )

            display(
                top_variables[
                    [
                        "variable",
                        "nombre",
                        "contribucion_alarma"
                    ]
                ].round(4)
            )

            # -----------------------------------------------
            # Contribution chart
            # -----------------------------------------------

            visualizar_variables_alerta_pca(
                tabla_top=top_variables,
                resumen_interpretacion=resumen_interpretacion
            )

        else:

            print(
                "No existe una alerta confirmada "
                "posterior a la introducción de la falla."
            )

            print(
                "Por tanto, no se genera un "
                "contribution chart."
            )

    # ========================================================
    # 6. Resumen textual final
    # ========================================================

    print("\n" + "=" * 70)
    print("RESUMEN")
    print("=" * 70)

    if resumen["Detectada"]:

        print(
            f"Estado: ALERTA CONFIRMADA"
        )

        print(
            f"Inicio de la falla: "
            f"{resumen['Tiempo_inicio_falla_h']:.2f} h"
        )

        print(
            f"Muestra de detección: "
            f"{resumen['Muestra_deteccion']}"
        )

        print(
            f"Retraso: "
            f"{resumen['Retraso_deteccion_min']} min"
        )

        if resumen_interpretacion is not None:

            print(
                f"Estadístico dominante: "
                f"{resumen_interpretacion['Estadistico_dominante']}"
            )

            print(
                f"Principal variable asociada: "
                f"{top_variables.iloc[0]['nombre']}"
            )

    else:

        print(
            "Estado: SIN ALERTA CONFIRMADA"
        )

    if resumen["Existe_prealarma"]:

        print(
            "Advertencia: esta corrida presentó "
            "una alarma persistente antes de la falla."
        )

    # ========================================================
    # 7. Retorno de resultados
    # ========================================================

    return {
        "serie_temporal":
            resultados,

        "resumen":
            resumen,

        "interpretacion":
            resumen_interpretacion,

        "contribuciones":
            tabla_contribuciones,

        "top_variables":
            top_variables
    }

In [ ]:
prototipo_f1_c1 = ejecutar_prototipo(
    falla=1,
    corrida=1
)

In [ ]:
prototipo_f4_c1 = ejecutar_prototipo(
    falla=4,
    corrida=1
)

In [ ]:
prototipo_f3_c1 = ejecutar_prototipo(
    falla=3,
    corrida=1
)

In [ ]:
variables_revisar = [
    "ff_train_clean",
    "df_train_normal",
    "df_val_normal",
    "df_train_clean",
    "df_val_clean",
    "feature_cols",
    "scaler",
    "f_train",
    "corridas_train",
    "corridas_val",
    "pca_model"
]

for nombre in variables_revisar:
    if nombre in globals():
        objeto = globals()[nombre]

        if hasattr(objeto, "shape"):
            print(
                f"{nombre:20s} → existe | shape = {objeto.shape}"
            )
        else:
            try:
                print(
                    f"{nombre:20s} → existe | longitud = {len(objeto)}"
                )
            except:
                print(
                    f"{nombre:20s} → existe"
                )
    else:
        print(
            f"{nombre:20s} → NO existe"
        )

## 15. Análisis de sensibilidad del PCA

Revisión del número de componentes, percentiles de umbral y reglas de persistencia para documentar la robustez de las decisiones metodológicas.


In [ ]:
from google.colab import drive
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import gc

# Montar Drive
drive.mount("/content/drive")

# Rutas
PROJECT_DIR = Path(
    "/content/drive/MyDrive/TFM_Tennessee_Eastman"
)

DATA_DIR = PROJECT_DIR / "Datos"
MODELS_DIR = PROJECT_DIR / "Modelos"
RESULTS_DIR = PROJECT_DIR / "Resultados"

# Cargar checkpoint PCA original
ruta_pca = (
    MODELS_DIR /
    "pca_baseline_v1.joblib"
)

checkpoint_pca = joblib.load(
    ruta_pca
)

print("Claves guardadas en el checkpoint:")
print(checkpoint_pca.keys())

In [ ]:
scaler = checkpoint_pca["scaler"]
pca_model = checkpoint_pca["pca_model"]
feature_cols = checkpoint_pca["feature_cols"]

corridas_train = checkpoint_pca["corridas_train"]
corridas_val = checkpoint_pca["corridas_val"]

T2_threshold = checkpoint_pca["T2_threshold"]
SPE_threshold = checkpoint_pca["SPE_threshold"]

ventana = checkpoint_pca.get(
    "ventana_persistencia",
    5
)

minimo_alarmas = checkpoint_pca.get(
    "minimo_alarmas",
    3
)

print("Variables de proceso:", len(feature_cols))
print("Corridas de entrenamiento:", len(corridas_train))
print("Corridas de validación:", len(corridas_val))
print("Componentes PCA original:", pca_model.n_components_)
print("Umbral T² original:", T2_threshold)
print("Umbral SPE original:", SPE_threshold)
print("Regla temporal:", minimo_alarmas, "de", ventana)

In [ ]:
!pip install -q pyreadr
import pyreadr

archivo_ff_train = (
    DATA_DIR /
    "TEP_FaultFree_Training.RData"
)

print("¿Existe archivo?:", archivo_ff_train.exists())

resultado_ff_train = pyreadr.read_r(
    str(archivo_ff_train)
)

print("Objetos encontrados:")
print(resultado_ff_train.keys())

In [ ]:
import numpy as np
import pandas as pd
import pyreadr
import joblib
import gc

In [ ]:
# ============================================================
# RECUPERACIÓN COMPLETA PARA ANÁLISIS DE SENSIBILIDAD DEL PCA
# ============================================================

!pip install -q pyreadr

from google.colab import drive
from pathlib import Path

import pyreadr
import joblib
import pandas as pd
import numpy as np
import gc

# ------------------------------------------------------------
# 1. Montar Google Drive
# ------------------------------------------------------------

drive.mount(
    "/content/drive",
    force_remount=False
)

# ------------------------------------------------------------
# 2. Definir rutas del proyecto
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/TFM_Tennessee_Eastman"
)

DATA_DIR = PROJECT_DIR / "Datos"
MODELS_DIR = PROJECT_DIR / "Modelos"
RESULTS_DIR = PROJECT_DIR / "Resultados"

print("PROJECT_DIR existe:", PROJECT_DIR.exists())
print("DATA_DIR existe:", DATA_DIR.exists())
print("MODELS_DIR existe:", MODELS_DIR.exists())

# ------------------------------------------------------------
# 3. Recuperar checkpoint original del PCA
# ------------------------------------------------------------

ruta_pca = (
    MODELS_DIR /
    "pca_baseline_v1.joblib"
)

checkpoint_pca = joblib.load(
    ruta_pca
)

scaler = checkpoint_pca["scaler"]
pca_model = checkpoint_pca["pca_model"]
feature_cols = checkpoint_pca["feature_cols"]

corridas_train = checkpoint_pca["corridas_train"]
corridas_val = checkpoint_pca["corridas_val"]

T2_threshold = checkpoint_pca["T2_threshold"]
SPE_threshold = checkpoint_pca["SPE_threshold"]

ventana = checkpoint_pca.get(
    "ventana_persistencia",
    5
)

minimo_alarmas = checkpoint_pca.get(
    "minimo_alarmas",
    3
)

print("\nCheckpoint PCA recuperado.")
print("Variables de proceso:", len(feature_cols))
print("Corridas train:", len(corridas_train))
print("Corridas validación:", len(corridas_val))
print("Componentes PCA original:", pca_model.n_components_)

# ------------------------------------------------------------
# 4. Cargar FaultFree Training
# ------------------------------------------------------------

archivo_ff_train = (
    DATA_DIR /
    "TEP_FaultFree_Training.RData"
)

print(
    "\nFaultFree Training existe:",
    archivo_ff_train.exists()
)

resultado_ff_train = pyreadr.read_r(
    str(archivo_ff_train)
)

nombre_objeto = next(
    iter(resultado_ff_train.keys())
)

ff_train = (
    resultado_ff_train[
        nombre_objeto
    ]
    .copy()
)

del resultado_ff_train
gc.collect()

print(
    "Dimensiones FaultFree Training:",
    ff_train.shape
)

print(
    "Corridas totales:",
    ff_train["simulationRun"].nunique()
)

print(
    "Fault numbers:",
    ff_train["faultNumber"].unique()
)

# ------------------------------------------------------------
# 5. Reconstruir EXACTAMENTE el split original
# ------------------------------------------------------------

df_train_normal = (
    ff_train.loc[
        ff_train["simulationRun"].isin(
            corridas_train
        )
    ]
    .sort_values(
        [
            "simulationRun",
            "sample"
        ]
    )
    .reset_index(drop=True)
)

df_val_normal = (
    ff_train.loc[
        ff_train["simulationRun"].isin(
            corridas_val
        )
    ]
    .sort_values(
        [
            "simulationRun",
            "sample"
        ]
    )
    .reset_index(drop=True)
)

print(
    "\nTrain normal:",
    df_train_normal.shape
)

print(
    "Validation normal:",
    df_val_normal.shape
)

print(
    "Corridas train:",
    df_train_normal[
        "simulationRun"
    ].nunique()
)

print(
    "Corridas validation:",
    df_val_normal[
        "simulationRun"
    ].nunique()
)

# ------------------------------------------------------------
# 6. Verificar que no hay corridas compartidas
# ------------------------------------------------------------

corridas_compartidas = (
    set(
        df_train_normal[
            "simulationRun"
        ]
    )
    .intersection(
        set(
            df_val_normal[
                "simulationRun"
            ]
        )
    )
)

print(
    "Corridas compartidas:",
    corridas_compartidas
)

# ------------------------------------------------------------
# 7. Construir matrices con las 52 variables de proceso
# ------------------------------------------------------------

X_train_pca = (
    df_train_normal[
        feature_cols
    ].copy()
)

X_val_pca = (
    df_val_normal[
        feature_cols
    ].copy()
)

# ------------------------------------------------------------
# 8. Aplicar EL MISMO scaler del modelo original
# ------------------------------------------------------------

X_train_pca_scaled = scaler.transform(
    X_train_pca
)

X_val_pca_scaled = scaler.transform(
    X_val_pca
)

print(
    "\nX_train_pca_scaled:",
    X_train_pca_scaled.shape
)

print(
    "X_val_pca_scaled:",
    X_val_pca_scaled.shape
)

print("\nRECUPERACIÓN COMPLETADA.")

In [ ]:
from sklearn.decomposition import PCA

componentes_evaluar = [31, 36, 41]

modelos_pca_sensibilidad = {}

for n_componentes in componentes_evaluar:

    pca_temp = PCA(
        n_components=n_componentes
    )

    pca_temp.fit(
        X_train_pca_scaled
    )

    modelos_pca_sensibilidad[
        n_componentes
    ] = pca_temp

    varianza = (
        pca_temp
        .explained_variance_ratio_
        .sum()
        * 100
    )

    print(
        f"{n_componentes} componentes → "
        f"{varianza:.3f} % de varianza explicada"
    )

In [ ]:
def calcular_T2_SPE(
    X_scaled,
    pca_model
):
    """
    Calcula Hotelling T² y SPE/Q
    para observaciones previamente escaladas.
    """

    # Proyección en componentes principales
    scores = pca_model.transform(
        X_scaled
    )

    # Varianza asociada a cada componente
    eigenvalues = (
        pca_model.explained_variance_
    )

    # Hotelling T²
    T2 = np.sum(
        (scores ** 2)
        / eigenvalues,
        axis=1
    )

    # Reconstrucción desde el espacio PCA
    X_reconstructed = (
        pca_model.inverse_transform(
            scores
        )
    )

    # Error cuadrático de reconstrucción
    SPE = np.sum(
        (
            X_scaled
            - X_reconstructed
        ) ** 2,
        axis=1
    )

    return T2, SPE

In [ ]:
def calcular_T2_SPE(
    X_scaled,
    pca_model
):
    """
    Calcula Hotelling T² y SPE/Q
    para observaciones previamente escaladas.
    """

    # Proyección en componentes principales
    scores = pca_model.transform(
        X_scaled
    )

    # Varianza asociada a cada componente
    eigenvalues = (
        pca_model.explained_variance_
    )

    # Hotelling T²
    T2 = np.sum(
        (scores ** 2)
        / eigenvalues,
        axis=1
    )

    # Reconstrucción desde el espacio PCA
    X_reconstructed = (
        pca_model.inverse_transform(
            scores
        )
    )

    # Error cuadrático de reconstrucción
    SPE = np.sum(
        (
            X_scaled
            - X_reconstructed
        ) ** 2,
        axis=1
    )

    return T2, SPE

In [ ]:
umbrales_componentes = {}

for n_componentes, pca_temp in modelos_pca_sensibilidad.items():

    T2_train_temp, SPE_train_temp = (
        calcular_T2_SPE(
            X_train_pca_scaled,
            pca_temp
        )
    )

    T2_lim_temp = np.quantile(
        T2_train_temp,
        0.99
    )

    SPE_lim_temp = np.quantile(
        SPE_train_temp,
        0.99
    )

    umbrales_componentes[
        n_componentes
    ] = {
        "T2_threshold":
            T2_lim_temp,

        "SPE_threshold":
            SPE_lim_temp
    }

    print(
        f"{n_componentes} componentes | "
        f"T² lim = {T2_lim_temp:.4f} | "
        f"SPE lim = {SPE_lim_temp:.4f}"
    )

In [ ]:
resultados_componentes_normal = []

for n_componentes, pca_temp in modelos_pca_sensibilidad.items():

    # Estadísticos en validación normal
    T2_val, SPE_val = calcular_T2_SPE(
        X_val_pca_scaled,
        pca_temp
    )

    T2_lim = (
        umbrales_componentes[
            n_componentes
        ]["T2_threshold"]
    )

    SPE_lim = (
        umbrales_componentes[
            n_componentes
        ]["SPE_threshold"]
    )

    # Alarmas individuales
    alarma_T2 = (
        T2_val > T2_lim
    )

    alarma_SPE = (
        SPE_val > SPE_lim
    )

    # PCA final usa condición OR
    alarma_combinada = (
        alarma_T2
        |
        alarma_SPE
    )

    resultados_componentes_normal.append({

        "Componentes":
            n_componentes,

        "Varianza_explicada_pct":
            pca_temp
            .explained_variance_ratio_
            .sum()
            * 100,

        "T2_threshold":
            T2_lim,

        "SPE_threshold":
            SPE_lim,

        "FAR_T2_pct":
            alarma_T2.mean()
            * 100,

        "FAR_SPE_pct":
            alarma_SPE.mean()
            * 100,

        "FAR_combinada_pct":
            alarma_combinada.mean()
            * 100
    })


tabla_sensibilidad_componentes_normal = (
    pd.DataFrame(
        resultados_componentes_normal
    )
    .sort_values(
        "Componentes"
    )
    .reset_index(drop=True)
)

display(
    tabla_sensibilidad_componentes_normal.round(4)
)

In [ ]:
def aplicar_regla_persistencia(
    metadata,
    alarma_puntual,
    ventana=5,
    minimo_alarmas=3
):

    resultado = (
        metadata[
            [
                "simulationRun",
                "sample"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    resultado[
        "alarma_puntual"
    ] = np.asarray(
        alarma_puntual,
        dtype=bool
    )

    resultado[
        "alarmas_ventana"
    ] = (
        resultado
        .groupby(
            "simulationRun"
        )["alarma_puntual"]
        .transform(
            lambda serie:
                serie
                .astype(int)
                .rolling(
                    window=ventana,
                    min_periods=ventana
                )
                .sum()
        )
    )

    resultado[
        "alarma_persistente"
    ] = (
        resultado[
            "alarmas_ventana"
        ]
        >= minimo_alarmas
    )

    return resultado

In [ ]:
resultados_persistencia_componentes = []

for n_componentes, pca_temp in modelos_pca_sensibilidad.items():

    T2_val, SPE_val = calcular_T2_SPE(
        X_val_pca_scaled,
        pca_temp
    )

    T2_lim = (
        umbrales_componentes[
            n_componentes
        ]["T2_threshold"]
    )

    SPE_lim = (
        umbrales_componentes[
            n_componentes
        ]["SPE_threshold"]
    )

    alarma_combinada = (
        (T2_val > T2_lim)
        |
        (SPE_val > SPE_lim)
    )

    resultado_persistencia = (
        aplicar_regla_persistencia(
            metadata=df_val_normal,
            alarma_puntual=alarma_combinada,
            ventana=5,
            minimo_alarmas=3
        )
    )

    # FAR persistente por muestra
    FAR_persistente = (
        resultado_persistencia[
            "alarma_persistente"
        ].mean()
        * 100
    )

    # Identificar inicio de cada episodio falso
    resultado_persistencia[
        "inicio_evento"
    ] = (
        resultado_persistencia[
            "alarma_persistente"
        ]
        &
        ~resultado_persistencia
        .groupby(
            "simulationRun"
        )[
            "alarma_persistente"
        ]
        .shift(
            fill_value=False
        )
    )

    episodios_falsos = int(
        resultado_persistencia[
            "inicio_evento"
        ].sum()
    )

    # 3 minutos por observación
    horas_normales = (
        len(
            resultado_persistencia
        )
        * 3
        / 60
    )

    episodios_100h = (
        episodios_falsos
        / horas_normales
        * 100
    )

    # Corridas que presentan al menos un episodio falso
    corridas_afectadas = (
        resultado_persistencia.loc[
            resultado_persistencia[
                "inicio_evento"
            ],
            "simulationRun"
        ]
        .nunique()
    )

    resultados_persistencia_componentes.append({

        "Componentes":
            n_componentes,

        "FAR_persistente_pct":
            FAR_persistente,

        "Episodios_falsos":
            episodios_falsos,

        "Episodios_por_100h":
            episodios_100h,

        "Corridas_afectadas":
            corridas_afectadas,

        "Corridas_afectadas_pct":
            corridas_afectadas
            / 100
            * 100
    })

In [ ]:
tabla_persistencia_componentes = (
    pd.DataFrame(
        resultados_persistencia_componentes
    )
)

tabla_sensibilidad_componentes = (
    tabla_sensibilidad_componentes_normal
    .merge(
        tabla_persistencia_componentes,
        on="Componentes"
    )
)

display(
    tabla_sensibilidad_componentes.round(4)
)

In [ ]:
from pathlib import Path

# Buscar si el dataset CSV ya sigue disponible
posibles_rutas = [
    Path("/kaggle/input/tep-csv"),
    Path("/root/.cache/kagglehub"),
    DATA_DIR
]

archivos_faulty_train = []

for ruta in posibles_rutas:
    if ruta.exists():
        archivos_faulty_train.extend(
            ruta.rglob("TEP_Faulty_Training.csv")
        )

print("Archivos encontrados:")

for archivo in archivos_faulty_train:
    print(
        archivo,
        "|",
        round(
            archivo.stat().st_size / (1024 ** 2),
            2
        ),
        "MB"
    )

In [ ]:
!pip install -q kagglehub

import kagglehub

ruta_tep_csv = Path(
    kagglehub.dataset_download(
        "afrniomelo/tep-csv"
    )
)

archivo_faulty_train_csv = (
    ruta_tep_csv /
    "TEP_Faulty_Training.csv"
)

print(archivo_faulty_train_csv)
print(
    "Existe:",
    archivo_faulty_train_csv.exists()
)

In [ ]:
PARTITION_TRAIN_DIR = (
    PROJECT_DIR
    / "Datos_Procesados"
    / "Faulty_Training_Parquet_v1"
)

PARTITION_TRAIN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(PARTITION_TRAIN_DIR)

In [ ]:
from collections import defaultdict
import gc

CHUNKSIZE = 200_000

columnas_necesarias = (
    [
        "faultNumber",
        "simulationRun",
        "sample"
    ]
    + list(feature_cols)
)

filas_por_falla_train = defaultdict(int)

lector = pd.read_csv(
    archivo_faulty_train_csv,
    usecols=columnas_necesarias,
    chunksize=CHUNKSIZE
)

total = 0

for numero_chunk, chunk in enumerate(
    lector,
    start=1
):

    chunk["faultNumber"] = (
        chunk["faultNumber"].astype("int16")
    )

    chunk["simulationRun"] = (
        chunk["simulationRun"].astype("int16")
    )

    chunk["sample"] = (
        chunk["sample"].astype("int16")
    )

    for falla, bloque in chunk.groupby(
        "faultNumber",
        sort=False
    ):

        falla = int(falla)

        carpeta = (
            PARTITION_TRAIN_DIR
            / f"falla_{falla:02d}"
        )

        carpeta.mkdir(
            parents=True,
            exist_ok=True
        )

        bloque.to_parquet(
            carpeta /
            f"part_{numero_chunk:04d}.parquet",
            index=False,
            compression="zstd"
        )

        filas_por_falla_train[
            falla
        ] += len(bloque)

    total += len(chunk)

    print(
        f"Chunk {numero_chunk:02d} | "
        f"{total:,} filas"
    )

    del chunk
    gc.collect()

print("\nTotal:", total)

In [ ]:
validacion_faulty_train = pd.DataFrame({
    "falla": range(1, 21),

    "filas": [
        filas_por_falla_train[i]
        for i in range(1, 21)
    ]
})

display(
    validacion_faulty_train
)

In [ ]:
def evaluar_falla_pca_sensibilidad(
    df_falla,
    pca_model,
    scaler,
    feature_cols,
    T2_threshold,
    SPE_threshold,
    muestra_inicio_falla=21,
    ventana=5,
    minimo_alarmas=3,
    minutos_por_muestra=3
):
    """
    Evalúa un PCA sobre las 500 corridas de una falla de Faulty_Training.

    Se excluyen de la tasa evaluable las corridas que presentan
    una alarma persistente antes de la introducción de la falla.
    """

    # ---------------------------------------------------------
    # 1. Orden temporal
    # ---------------------------------------------------------

    df = (
        df_falla
        .sort_values(
            ["simulationRun", "sample"]
        )
        .reset_index(drop=True)
    )

    # ---------------------------------------------------------
    # 2. Variables de proceso y escalado
    # ---------------------------------------------------------

    X = df[feature_cols]

    X_scaled = scaler.transform(X)

    # ---------------------------------------------------------
    # 3. T² y SPE
    # ---------------------------------------------------------

    T2, SPE = calcular_T2_SPE(
        X_scaled,
        pca_model
    )

    # ---------------------------------------------------------
    # 4. Alarma puntual PCA
    # ---------------------------------------------------------

    alarma_puntual = (
        (T2 > T2_threshold)
        |
        (SPE > SPE_threshold)
    )

    # ---------------------------------------------------------
    # 5. Regla temporal 3-de-5
    # ---------------------------------------------------------

    resultado = df[
        [
            "simulationRun",
            "sample"
        ]
    ].copy()

    resultado["alarma_puntual"] = (
        alarma_puntual
    )

    resultado["alarmas_ventana"] = (
        resultado
        .groupby(
            "simulationRun"
        )["alarma_puntual"]
        .transform(
            lambda s:
                s.astype(int)
                .rolling(
                    window=ventana,
                    min_periods=ventana
                )
                .sum()
        )
    )

    resultado["alarma_persistente"] = (
        resultado["alarmas_ventana"]
        >= minimo_alarmas
    )

    # Inicio de cada episodio persistente
    resultado["inicio_episodio"] = (
        resultado["alarma_persistente"]
        &
        ~resultado
        .groupby(
            "simulationRun"
        )["alarma_persistente"]
        .shift(fill_value=False)
    )

    # ---------------------------------------------------------
    # 6. Prealarmas
    # ---------------------------------------------------------

    mascara_pre = (
        resultado["inicio_episodio"]
        &
        (
            resultado["sample"]
            < muestra_inicio_falla
        )
    )

    corridas_prealarma = set(
        resultado.loc[
            mascara_pre,
            "simulationRun"
        ].unique()
    )

    # ---------------------------------------------------------
    # 7. Primeras detecciones posteriores a la falla
    # ---------------------------------------------------------

    mascara_post = (
        resultado["inicio_episodio"]
        &
        (
            resultado["sample"]
            >= muestra_inicio_falla
        )
    )

    primeras_detecciones = (
        resultado.loc[
            mascara_post,
            [
                "simulationRun",
                "sample"
            ]
        ]
        .groupby(
            "simulationRun",
            as_index=False
        )
        .agg(
            muestra_deteccion=(
                "sample",
                "min"
            )
        )
    )

    # ---------------------------------------------------------
    # 8. Corridas evaluables
    # ---------------------------------------------------------

    todas_corridas = set(
        resultado[
            "simulationRun"
        ].unique()
    )

    corridas_evaluables = (
        todas_corridas
        - corridas_prealarma
    )

    primeras_detecciones_eval = (
        primeras_detecciones.loc[
            primeras_detecciones[
                "simulationRun"
            ].isin(
                corridas_evaluables
            )
        ]
        .copy()
    )

    # ---------------------------------------------------------
    # 9. Retrasos
    # ---------------------------------------------------------

    primeras_detecciones_eval[
        "retraso_min"
    ] = (
        (
            primeras_detecciones_eval[
                "muestra_deteccion"
            ]
            - muestra_inicio_falla
        )
        * minutos_por_muestra
    )

    total_corridas = len(
        todas_corridas
    )

    n_prealarma = len(
        corridas_prealarma
    )

    n_evaluables = len(
        corridas_evaluables
    )

    n_detectadas = len(
        primeras_detecciones_eval
    )

    tasa_deteccion = (
        n_detectadas
        / n_evaluables
        * 100
        if n_evaluables > 0
        else np.nan
    )

    if n_detectadas > 0:

        retrasos = (
            primeras_detecciones_eval[
                "retraso_min"
            ]
        )

        retraso_medio = retrasos.mean()
        retraso_mediano = retrasos.median()
        retraso_p95 = retrasos.quantile(0.95)

    else:

        retraso_medio = np.nan
        retraso_mediano = np.nan
        retraso_p95 = np.nan

    return {
        "corridas_totales":
            total_corridas,

        "corridas_prealarma":
            n_prealarma,

        "corridas_evaluables":
            n_evaluables,

        "corridas_detectadas":
            n_detectadas,

        "tasa_deteccion_pct":
            tasa_deteccion,

        "retraso_medio_min":
            retraso_medio,

        "retraso_mediano_min":
            retraso_mediano,

        "retraso_p95_min":
            retraso_p95
    }

In [ ]:
resultados_sensibilidad_fallas = []

for numero_falla in range(1, 21):

    print()
    print("=" * 55)
    print(
        f"FALLA {numero_falla} DE 20"
    )
    print("=" * 55)

    # --------------------------------------------------------
    # Cargar únicamente esta falla
    # --------------------------------------------------------

    ruta_falla = (
        PARTITION_TRAIN_DIR
        / f"falla_{numero_falla:02d}"
    )

    df_falla = pd.read_parquet(
        ruta_falla
    )

    print(
        "Datos cargados:",
        df_falla.shape
    )

    # --------------------------------------------------------
    # Evaluar 31, 36 y 41 componentes
    # --------------------------------------------------------

    for n_componentes in [
        31,
        36,
        41
    ]:

        pca_temp = (
            modelos_pca_sensibilidad[
                n_componentes
            ]
        )

        T2_lim = (
            umbrales_componentes[
                n_componentes
            ]["T2_threshold"]
        )

        SPE_lim = (
            umbrales_componentes[
                n_componentes
            ]["SPE_threshold"]
        )

        resultado = (
            evaluar_falla_pca_sensibilidad(
                df_falla=df_falla,
                pca_model=pca_temp,
                scaler=scaler,
                feature_cols=feature_cols,
                T2_threshold=T2_lim,
                SPE_threshold=SPE_lim,
                muestra_inicio_falla=21,
                ventana=5,
                minimo_alarmas=3,
                minutos_por_muestra=3
            )
        )

        resultado[
            "Componentes"
        ] = n_componentes

        resultado[
            "falla"
        ] = numero_falla

        resultados_sensibilidad_fallas.append(
            resultado
        )

        print(
            f"  {n_componentes} PC → "
            f"Detección "
            f"{resultado['tasa_deteccion_pct']:.2f}% | "
            f"Retraso medio "
            f"{resultado['retraso_medio_min']:.1f} min | "
            f"Prealarmas "
            f"{resultado['corridas_prealarma']}"
        )

    # --------------------------------------------------------
    # Liberar esta falla
    # --------------------------------------------------------

    del df_falla
    gc.collect()

print()
print("ANÁLISIS TERMINADO")

In [ ]:
tabla_sensibilidad_fallas = (
    pd.DataFrame(
        resultados_sensibilidad_fallas
    )
    .sort_values(
        [
            "Componentes",
            "falla"
        ]
    )
    .reset_index(drop=True)
)

display(
    tabla_sensibilidad_fallas.round(3)
)

In [ ]:
tabla_deteccion_componentes = (
    tabla_sensibilidad_fallas
    .pivot(
        index="falla",
        columns="Componentes",
        values="tasa_deteccion_pct"
    )
    .reset_index()
)

tabla_deteccion_componentes.columns.name = None

tabla_deteccion_componentes = (
    tabla_deteccion_componentes.rename(
        columns={
            31: "PCA_31",
            36: "PCA_36",
            41: "PCA_41"
        }
    )
)

display(
    tabla_deteccion_componentes.round(2)
)

In [ ]:
resumen_componentes_faulty = []

for n_componentes in [
    31,
    36,
    41
]:

    datos = (
        tabla_sensibilidad_fallas.loc[
            tabla_sensibilidad_fallas[
                "Componentes"
            ]
            == n_componentes
        ]
        .copy()
    )

    total_evaluables = (
        datos[
            "corridas_evaluables"
        ].sum()
    )

    total_detectadas = (
        datos[
            "corridas_detectadas"
        ].sum()
    )

    tasa_global = (
        total_detectadas
        / total_evaluables
        * 100
    )

    # Retraso ponderado por corridas detectadas
    retraso_ponderado = (
        (
            datos[
                "retraso_medio_min"
            ]
            *
            datos[
                "corridas_detectadas"
            ]
        ).sum()
        /
        datos[
            "corridas_detectadas"
        ].sum()
    )

    resumen_componentes_faulty.append({

        "Componentes":
            n_componentes,

        "Deteccion_global_pct":
            tasa_global,

        "Deteccion_media_fallas_pct":
            datos[
                "tasa_deteccion_pct"
            ].mean(),

        "Mediana_deteccion_pct":
            datos[
                "tasa_deteccion_pct"
            ].median(),

        "Fallas_deteccion_99_pct":
            (
                datos[
                    "tasa_deteccion_pct"
                ] >= 99
            ).sum(),

        "Corridas_prealarma_totales":
            datos[
                "corridas_prealarma"
            ].sum(),

        "Retraso_medio_entre_fallas_min":
            datos[
                "retraso_medio_min"
            ].mean(),

        "Retraso_ponderado_min":
            retraso_ponderado,

        "Mediana_retraso_entre_fallas_min":
            datos[
                "retraso_mediano_min"
            ].median()
    })


resumen_componentes_faulty = pd.DataFrame(
    resumen_componentes_faulty
)

display(
    resumen_componentes_faulty.round(3)
)

In [ ]:
tabla_final_sensibilidad_componentes = (
    tabla_sensibilidad_componentes
    .merge(
        resumen_componentes_faulty,
        on="Componentes",
        how="left"
    )
)

columnas_mostrar = [
    "Componentes",
    "Varianza_explicada_pct",
    "FAR_combinada_pct",
    "FAR_persistente_pct",
    "Episodios_por_100h",
    "Deteccion_global_pct",
    "Fallas_deteccion_99_pct",
    "Retraso_ponderado_min",
    "Corridas_prealarma_totales"
]

display(
    tabla_final_sensibilidad_componentes[
        columnas_mostrar
    ].round(3)
)

In [ ]:
display(
    tabla_deteccion_componentes.round(2)
)

In [ ]:
pca_36 = modelos_pca_sensibilidad[36]

T2_train_36, SPE_train_36 = calcular_T2_SPE(
    X_train_pca_scaled,
    pca_36
)

T2_val_36, SPE_val_36 = calcular_T2_SPE(
    X_val_pca_scaled,
    pca_36
)

print("Train:", len(T2_train_36))
print("Validation:", len(T2_val_36))

In [ ]:
percentiles_evaluar = {
    "P98": 0.98,
    "P99": 0.99,
    "P99.5": 0.995
}

umbrales_percentiles = {}

for nombre, q in percentiles_evaluar.items():

    T2_lim = np.quantile(
        T2_train_36,
        q
    )

    SPE_lim = np.quantile(
        SPE_train_36,
        q
    )

    umbrales_percentiles[nombre] = {
        "percentil": q,
        "T2_threshold": T2_lim,
        "SPE_threshold": SPE_lim
    }

    print(
        f"{nombre:5s} | "
        f"T² = {T2_lim:.4f} | "
        f"SPE = {SPE_lim:.4f}"
    )

In [ ]:
resultados_sensibilidad_umbral_normal = []

for nombre, config in umbrales_percentiles.items():

    T2_lim = config["T2_threshold"]
    SPE_lim = config["SPE_threshold"]

    alarma_T2 = (
        T2_val_36 > T2_lim
    )

    alarma_SPE = (
        SPE_val_36 > SPE_lim
    )

    alarma_combinada = (
        alarma_T2
        |
        alarma_SPE
    )

    # Solapamiento: muestras que disparan ambos
    alarma_ambos = (
        alarma_T2
        &
        alarma_SPE
    )

    resultados_sensibilidad_umbral_normal.append({

        "Percentil":
            nombre,

        "T2_threshold":
            T2_lim,

        "SPE_threshold":
            SPE_lim,

        "FAR_T2_pct":
            alarma_T2.mean() * 100,

        "FAR_SPE_pct":
            alarma_SPE.mean() * 100,

        "Solapamiento_pct":
            alarma_ambos.mean() * 100,

        "FAR_combinada_pct":
            alarma_combinada.mean() * 100
    })


tabla_umbral_normal = pd.DataFrame(
    resultados_sensibilidad_umbral_normal
)

display(
    tabla_umbral_normal.round(4)
)

In [ ]:
resultados_umbral_persistencia = []

for nombre, config in umbrales_percentiles.items():

    T2_lim = config["T2_threshold"]
    SPE_lim = config["SPE_threshold"]

    alarma_combinada = (
        (T2_val_36 > T2_lim)
        |
        (SPE_val_36 > SPE_lim)
    )

    resultado_persistencia = aplicar_regla_persistencia(
        metadata=df_val_normal,
        alarma_puntual=alarma_combinada,
        ventana=5,
        minimo_alarmas=3
    )

    FAR_persistente = (
        resultado_persistencia[
            "alarma_persistente"
        ].mean()
        * 100
    )

    resultado_persistencia[
        "inicio_evento"
    ] = (
        resultado_persistencia[
            "alarma_persistente"
        ]
        &
        ~resultado_persistencia
        .groupby("simulationRun")[
            "alarma_persistente"
        ]
        .shift(fill_value=False)
    )

    episodios_falsos = int(
        resultado_persistencia[
            "inicio_evento"
        ].sum()
    )

    horas_normales = (
        len(resultado_persistencia)
        * 3
        / 60
    )

    episodios_100h = (
        episodios_falsos
        / horas_normales
        * 100
    )

    corridas_afectadas = (
        resultado_persistencia.loc[
            resultado_persistencia[
                "inicio_evento"
            ],
            "simulationRun"
        ]
        .nunique()
    )

    resultados_umbral_persistencia.append({

        "Percentil":
            nombre,

        "FAR_persistente_pct":
            FAR_persistente,

        "Episodios_falsos":
            episodios_falsos,

        "Episodios_por_100h":
            episodios_100h,

        "Corridas_afectadas":
            corridas_afectadas,

        "Corridas_afectadas_pct":
            corridas_afectadas / 100 * 100
    })


tabla_umbral_persistencia = pd.DataFrame(
    resultados_umbral_persistencia
)

tabla_sensibilidad_umbral_normal = (
    tabla_umbral_normal
    .merge(
        tabla_umbral_persistencia,
        on="Percentil"
    )
)

display(
    tabla_sensibilidad_umbral_normal.round(4)
)

In [ ]:
resultados_sensibilidad_umbral_fallas = []

for numero_falla in range(1, 21):

    print()
    print("=" * 55)
    print(
        f"FALLA {numero_falla} DE 20"
    )
    print("=" * 55)

    ruta_falla = (
        PARTITION_TRAIN_DIR
        / f"falla_{numero_falla:02d}"
    )

    df_falla = pd.read_parquet(
        ruta_falla
    )

    for nombre, config in umbrales_percentiles.items():

        resultado = evaluar_falla_pca_sensibilidad(
            df_falla=df_falla,
            pca_model=pca_36,
            scaler=scaler,
            feature_cols=feature_cols,
            T2_threshold=config["T2_threshold"],
            SPE_threshold=config["SPE_threshold"],
            muestra_inicio_falla=21,
            ventana=5,
            minimo_alarmas=3,
            minutos_por_muestra=3
        )

        resultado["Percentil"] = nombre
        resultado["falla"] = numero_falla

        resultados_sensibilidad_umbral_fallas.append(
            resultado
        )

        print(
            f"  {nombre:5s} → "
            f"Detección "
            f"{resultado['tasa_deteccion_pct']:.2f}% | "
            f"Retraso "
            f"{resultado['retraso_medio_min']:.1f} min | "
            f"Prealarmas "
            f"{resultado['corridas_prealarma']}"
        )

    del df_falla
    gc.collect()

print("\nANÁLISIS DE UMBRALES TERMINADO")

In [ ]:
tabla_umbral_fallas = (
    pd.DataFrame(
        resultados_sensibilidad_umbral_fallas
    )
)

tabla_deteccion_percentiles = (
    tabla_umbral_fallas
    .pivot(
        index="falla",
        columns="Percentil",
        values="tasa_deteccion_pct"
    )
    .reset_index()
)

tabla_deteccion_percentiles.columns.name = None

display(
    tabla_deteccion_percentiles.round(2)
)

In [ ]:
resumen_umbral_faulty = []

for nombre in [
    "P98",
    "P99",
    "P99.5"
]:

    datos = (
        tabla_umbral_fallas.loc[
            tabla_umbral_fallas[
                "Percentil"
            ] == nombre
        ]
        .copy()
    )

    total_evaluables = (
        datos[
            "corridas_evaluables"
        ].sum()
    )

    total_detectadas = (
        datos[
            "corridas_detectadas"
        ].sum()
    )

    tasa_global = (
        total_detectadas
        / total_evaluables
        * 100
    )

    retraso_ponderado = (
        (
            datos[
                "retraso_medio_min"
            ]
            *
            datos[
                "corridas_detectadas"
            ]
        ).sum()
        /
        total_detectadas
    )

    resumen_umbral_faulty.append({

        "Percentil":
            nombre,

        "Deteccion_global_pct":
            tasa_global,

        "Deteccion_media_fallas_pct":
            datos[
                "tasa_deteccion_pct"
            ].mean(),

        "Fallas_deteccion_99_pct":
            (
                datos[
                    "tasa_deteccion_pct"
                ] >= 99
            ).sum(),

        "Corridas_prealarma_totales":
            datos[
                "corridas_prealarma"
            ].sum(),

        "Retraso_ponderado_min":
            retraso_ponderado,

        "Retraso_medio_entre_fallas_min":
            datos[
                "retraso_medio_min"
            ].mean()
    })


resumen_umbral_faulty = pd.DataFrame(
    resumen_umbral_faulty
)

display(
    resumen_umbral_faulty.round(3)
)

In [ ]:
tabla_final_sensibilidad_umbral = (
    tabla_sensibilidad_umbral_normal
    .merge(
        resumen_umbral_faulty,
        on="Percentil",
        how="left"
    )
)

columnas_mostrar = [
    "Percentil",
    "FAR_T2_pct",
    "FAR_SPE_pct",
    "Solapamiento_pct",
    "FAR_combinada_pct",
    "FAR_persistente_pct",
    "Episodios_por_100h",
    "Deteccion_global_pct",
    "Fallas_deteccion_99_pct",
    "Retraso_ponderado_min",
    "Corridas_prealarma_totales"
]

display(
    tabla_final_sensibilidad_umbral[
        columnas_mostrar
    ].round(3)
)

In [ ]:
# ============================================================
# UMBRALES DE SENSIBILIDAD CALIBRADOS SOBRE VALIDACIÓN NORMAL
# Igual que en el modelo PCA original
# ============================================================

percentiles_evaluar = {
    "P98": 0.98,
    "P99": 0.99,
    "P99.5": 0.995
}

umbrales_percentiles = {}

for nombre, q in percentiles_evaluar.items():

    T2_lim = np.quantile(
        T2_val_36,
        q
    )

    SPE_lim = np.quantile(
        SPE_val_36,
        q
    )

    umbrales_percentiles[nombre] = {
        "percentil": q,
        "T2_threshold": T2_lim,
        "SPE_threshold": SPE_lim
    }

    print(
        f"{nombre:5s} | "
        f"T² = {T2_lim:.6f} | "
        f"SPE = {SPE_lim:.6f}"
    )

In [ ]:
resultados_sensibilidad_umbral_normal = []

for nombre, config in umbrales_percentiles.items():

    T2_lim = config["T2_threshold"]
    SPE_lim = config["SPE_threshold"]

    alarma_T2 = T2_val_36 > T2_lim
    alarma_SPE = SPE_val_36 > SPE_lim

    alarma_combinada = (
        alarma_T2
        |
        alarma_SPE
    )

    alarma_ambos = (
        alarma_T2
        &
        alarma_SPE
    )

    # Regla temporal
    resultado_persistencia = aplicar_regla_persistencia(
        metadata=df_val_normal,
        alarma_puntual=alarma_combinada,
        ventana=5,
        minimo_alarmas=3
    )

    resultado_persistencia["inicio_evento"] = (
        resultado_persistencia["alarma_persistente"]
        &
        ~resultado_persistencia
        .groupby("simulationRun")[
            "alarma_persistente"
        ]
        .shift(fill_value=False)
    )

    episodios_falsos = int(
        resultado_persistencia[
            "inicio_evento"
        ].sum()
    )

    horas_normales = (
        len(resultado_persistencia)
        * 3
        / 60
    )

    corridas_afectadas = (
        resultado_persistencia.loc[
            resultado_persistencia[
                "inicio_evento"
            ],
            "simulationRun"
        ]
        .nunique()
    )

    resultados_sensibilidad_umbral_normal.append({

        "Percentil": nombre,

        "T2_threshold":
            T2_lim,

        "SPE_threshold":
            SPE_lim,

        "FAR_T2_pct":
            alarma_T2.mean() * 100,

        "FAR_SPE_pct":
            alarma_SPE.mean() * 100,

        "Solapamiento_pct":
            alarma_ambos.mean() * 100,

        "FAR_combinada_pct":
            alarma_combinada.mean() * 100,

        "FAR_persistente_pct":
            resultado_persistencia[
                "alarma_persistente"
            ].mean() * 100,

        "Episodios_por_100h":
            episodios_falsos
            / horas_normales
            * 100,

        "Corridas_afectadas":
            corridas_afectadas
    })


tabla_sensibilidad_umbral_normal = (
    pd.DataFrame(
        resultados_sensibilidad_umbral_normal
    )
)

display(
    tabla_sensibilidad_umbral_normal.round(4)
)

In [ ]:
resultados_sensibilidad_umbral_fallas = []

for numero_falla in range(1, 21):

    print(
        f"Evaluando falla {numero_falla}/20..."
    )

    df_falla = pd.read_parquet(
        PARTITION_TRAIN_DIR /
        f"falla_{numero_falla:02d}"
    )

    for nombre, config in umbrales_percentiles.items():

        resultado = evaluar_falla_pca_sensibilidad(
            df_falla=df_falla,
            pca_model=pca_36,
            scaler=scaler,
            feature_cols=feature_cols,
            T2_threshold=config["T2_threshold"],
            SPE_threshold=config["SPE_threshold"],
            muestra_inicio_falla=21,
            ventana=5,
            minimo_alarmas=3,
            minutos_por_muestra=3
        )

        resultado["Percentil"] = nombre
        resultado["falla"] = numero_falla

        resultados_sensibilidad_umbral_fallas.append(
            resultado
        )

    del df_falla
    gc.collect()


tabla_umbral_fallas = pd.DataFrame(
    resultados_sensibilidad_umbral_fallas
)

In [ ]:
resultados_sensibilidad_umbral_fallas = []

for numero_falla in range(1, 21):

    print(
        f"Evaluando falla {numero_falla}/20..."
    )

    df_falla = pd.read_parquet(
        PARTITION_TRAIN_DIR /
        f"falla_{numero_falla:02d}"
    )

    for nombre, config in umbrales_percentiles.items():

        resultado = evaluar_falla_pca_sensibilidad(
            df_falla=df_falla,
            pca_model=pca_36,
            scaler=scaler,
            feature_cols=feature_cols,
            T2_threshold=config["T2_threshold"],
            SPE_threshold=config["SPE_threshold"],
            muestra_inicio_falla=21,
            ventana=5,
            minimo_alarmas=3,
            minutos_por_muestra=3
        )

        resultado["Percentil"] = nombre
        resultado["falla"] = numero_falla

        resultados_sensibilidad_umbral_fallas.append(
            resultado
        )

    del df_falla
    gc.collect()


tabla_umbral_fallas = pd.DataFrame(
    resultados_sensibilidad_umbral_fallas
)

In [ ]:
resumen_umbral_faulty = []

for nombre in [
    "P98",
    "P99",
    "P99.5"
]:

    datos = tabla_umbral_fallas.loc[
        tabla_umbral_fallas["Percentil"]
        == nombre
    ].copy()

    total_evaluables = (
        datos["corridas_evaluables"].sum()
    )

    total_detectadas = (
        datos["corridas_detectadas"].sum()
    )

    tasa_global = (
        total_detectadas
        / total_evaluables
        * 100
    )

    retraso_ponderado = (
        (
            datos["retraso_medio_min"]
            *
            datos["corridas_detectadas"]
        ).sum()
        /
        total_detectadas
    )

    resumen_umbral_faulty.append({

        "Percentil":
            nombre,

        "Deteccion_global_pct":
            tasa_global,

        "Fallas_deteccion_99_pct":
            (
                datos[
                    "tasa_deteccion_pct"
                ] >= 99
            ).sum(),

        "Corridas_prealarma_totales":
            datos[
                "corridas_prealarma"
            ].sum(),

        "Retraso_ponderado_min":
            retraso_ponderado
    })


resumen_umbral_faulty = pd.DataFrame(
    resumen_umbral_faulty
)

tabla_final_sensibilidad_umbral = (
    tabla_sensibilidad_umbral_normal
    .merge(
        resumen_umbral_faulty,
        on="Percentil"
    )
)

display(
    tabla_final_sensibilidad_umbral.round(3)
)

In [ ]:
reglas_persistencia = {
    "1-de-1": {
        "ventana": 1,
        "minimo_alarmas": 1
    },

    "2-de-5": {
        "ventana": 5,
        "minimo_alarmas": 2
    },

    "3-de-5": {
        "ventana": 5,
        "minimo_alarmas": 3
    },

    "4-de-5": {
        "ventana": 5,
        "minimo_alarmas": 4
    }
}

In [ ]:
T2_lim_final = (
    umbrales_percentiles[
        "P99"
    ]["T2_threshold"]
)

SPE_lim_final = (
    umbrales_percentiles[
        "P99"
    ]["SPE_threshold"]
)

print(
    "T² final:",
    T2_lim_final
)

print(
    "SPE final:",
    SPE_lim_final
)

In [ ]:
alarma_puntual_val = (
    (T2_val_36 > T2_lim_final)
    |
    (SPE_val_36 > SPE_lim_final)
)

print(
    "FAR puntual antes de persistencia:",
    alarma_puntual_val.mean() * 100
)

In [ ]:
resultados_reglas_normal = []

for nombre_regla, config in reglas_persistencia.items():

    resultado_regla = aplicar_regla_persistencia(
        metadata=df_val_normal,
        alarma_puntual=alarma_puntual_val,
        ventana=config["ventana"],
        minimo_alarmas=config[
            "minimo_alarmas"
        ]
    )

    # --------------------------------------------------------
    # Inicio de episodios
    # --------------------------------------------------------

    resultado_regla[
        "inicio_evento"
    ] = (
        resultado_regla[
            "alarma_persistente"
        ]
        &
        ~resultado_regla
        .groupby("simulationRun")[
            "alarma_persistente"
        ]
        .shift(fill_value=False)
    )

    # --------------------------------------------------------
    # FAR persistente
    # --------------------------------------------------------

    FAR_persistente = (
        resultado_regla[
            "alarma_persistente"
        ].mean()
        * 100
    )

    # --------------------------------------------------------
    # Número de episodios
    # --------------------------------------------------------

    episodios = int(
        resultado_regla[
            "inicio_evento"
        ].sum()
    )

    horas_normales = (
        len(resultado_regla)
        * 3
        / 60
    )

    episodios_100h = (
        episodios
        / horas_normales
        * 100
    )

    # --------------------------------------------------------
    # Corridas afectadas
    # --------------------------------------------------------

    corridas_afectadas = (
        resultado_regla.loc[
            resultado_regla[
                "inicio_evento"
            ],
            "simulationRun"
        ]
        .nunique()
    )

    resultados_reglas_normal.append({

        "Regla":
            nombre_regla,

        "Ventana":
            config["ventana"],

        "Minimo_alarmas":
            config["minimo_alarmas"],

        "FAR_persistente_pct":
            FAR_persistente,

        "Episodios_falsos":
            episodios,

        "Episodios_por_100h":
            episodios_100h,

        "Corridas_afectadas":
            corridas_afectadas,

        "Corridas_afectadas_pct":
            corridas_afectadas
            / 100
            * 100
    })


tabla_reglas_normal = pd.DataFrame(
    resultados_reglas_normal
)

display(
    tabla_reglas_normal.round(3)
)

In [ ]:
resultados_reglas_fallas = []

for numero_falla in range(1, 21):

    print(
        f"Evaluando falla "
        f"{numero_falla}/20..."
    )

    df_falla = pd.read_parquet(
        PARTITION_TRAIN_DIR /
        f"falla_{numero_falla:02d}"
    )

    for nombre_regla, config in (
        reglas_persistencia.items()
    ):

        resultado = (
            evaluar_falla_pca_sensibilidad(
                df_falla=df_falla,
                pca_model=pca_36,
                scaler=scaler,
                feature_cols=feature_cols,
                T2_threshold=T2_lim_final,
                SPE_threshold=SPE_lim_final,
                muestra_inicio_falla=21,
                ventana=config["ventana"],
                minimo_alarmas=config[
                    "minimo_alarmas"
                ],
                minutos_por_muestra=3
            )
        )

        resultado[
            "Regla"
        ] = nombre_regla

        resultado[
            "falla"
        ] = numero_falla

        resultados_reglas_fallas.append(
            resultado
        )

    del df_falla
    gc.collect()


tabla_reglas_fallas = pd.DataFrame(
    resultados_reglas_fallas
)

In [ ]:
resumen_reglas_faulty = []

for nombre_regla in reglas_persistencia.keys():

    datos = (
        tabla_reglas_fallas.loc[
            tabla_reglas_fallas[
                "Regla"
            ] == nombre_regla
        ]
        .copy()
    )

    total_evaluables = (
        datos[
            "corridas_evaluables"
        ].sum()
    )

    total_detectadas = (
        datos[
            "corridas_detectadas"
        ].sum()
    )

    tasa_global = (
        total_detectadas
        / total_evaluables
        * 100
    )

    retraso_ponderado = (
        (
            datos[
                "retraso_medio_min"
            ]
            *
            datos[
                "corridas_detectadas"
            ]
        ).sum()
        /
        total_detectadas
    )

    resumen_reglas_faulty.append({

        "Regla":
            nombre_regla,

        "Deteccion_global_pct":
            tasa_global,

        "Fallas_deteccion_99_pct":
            (
                datos[
                    "tasa_deteccion_pct"
                ] >= 99
            ).sum(),

        "Corridas_prealarma_totales":
            datos[
                "corridas_prealarma"
            ].sum(),

        "Retraso_ponderado_min":
            retraso_ponderado,

        "Retraso_medio_fallas_min":
            datos[
                "retraso_medio_min"
            ].mean()
    })


resumen_reglas_faulty = pd.DataFrame(
    resumen_reglas_faulty
)

display(
    resumen_reglas_faulty.round(3)
)

In [ ]:
tabla_final_sensibilidad_regla = (
    tabla_reglas_normal
    .merge(
        resumen_reglas_faulty,
        on="Regla",
        how="left"
    )
)

columnas_regla = [
    "Regla",
    "FAR_persistente_pct",
    "Episodios_por_100h",
    "Corridas_afectadas_pct",
    "Deteccion_global_pct",
    "Fallas_deteccion_99_pct",
    "Corridas_prealarma_totales",
    "Retraso_ponderado_min"
]

display(
    tabla_final_sensibilidad_regla[
        columnas_regla
    ].round(3)
)

## 16. Tablas y figuras para la memoria

Construcción de tablas finales por falla y generación de figuras de sensibilidad, curvas PR, scores temporales y contribuciones.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# RUTAS DE RESULTADOS FINALES
# ============================================================

ruta_deteccion_testing = (
    RESULTS_DIR /
    "evaluacion_final_Faulty_Testing_checkpoint.csv"
)

ruta_pr_testing = (
    RESULTS_DIR /
    "PrecisionRecall_20_fallas_Testing.csv"
)

print(
    "Detección existe:",
    ruta_deteccion_testing.exists()
)

print(
    "Precision-Recall existe:",
    ruta_pr_testing.exists()
)

In [ ]:
resultados_testing_tabla = pd.read_csv(
    ruta_deteccion_testing
)

resultados_pr_tabla = pd.read_csv(
    ruta_pr_testing
)

print(
    "Resultados detección:",
    resultados_testing_tabla.shape
)

print(
    "Resultados PR:",
    resultados_pr_tabla.shape
)

print("\nColumnas detección:")
print(
    resultados_testing_tabla.columns.tolist()
)

print("\nColumnas PR:")
print(
    resultados_pr_tabla.columns.tolist()
)

In [ ]:
ap_pca = resultados_pr_tabla[
    [
        "falla",
        "AP_PCA"
    ]
].copy()

ap_pca["Modelo"] = "PCA"

ap_pca = ap_pca.rename(
    columns={
        "AP_PCA": "AP"
    }
)


ap_if = resultados_pr_tabla[
    [
        "falla",
        "AP_IF"
    ]
].copy()

ap_if["Modelo"] = "Isolation Forest"

ap_if = ap_if.rename(
    columns={
        "AP_IF": "AP"
    }
)


ap_ae = resultados_pr_tabla[
    [
        "falla",
        "AP_AE"
    ]
].copy()

ap_ae["Modelo"] = "Autoencoder"

ap_ae = ap_ae.rename(
    columns={
        "AP_AE": "AP"
    }
)


ap_long = pd.concat(
    [
        ap_pca,
        ap_if,
        ap_ae
    ],
    ignore_index=True
)

display(
    ap_long.head()
)

In [ ]:
tabla_resultados_por_falla_long = (
    resultados_testing_tabla
    .merge(
        ap_long,
        on=[
            "falla",
            "Modelo"
        ],
        how="left"
    )
)

In [ ]:
tabla_resultados_por_falla_long = (
    tabla_resultados_por_falla_long[
        [
            "falla",
            "Modelo",
            "corridas_prealarma",
            "corridas_evaluables",
            "corridas_detectadas_evaluables",
            "tasa_deteccion_evaluable_pct",
            "retraso_mediano_min",
            "AP"
        ]
    ]
    .sort_values(
        [
            "falla",
            "Modelo"
        ]
    )
    .reset_index(drop=True)
)

display(
    tabla_resultados_por_falla_long.round(3)
)

In [ ]:
fallas = pd.DataFrame({
    "Falla": range(1, 21)
})

tabla_por_falla = fallas.copy()

In [ ]:
pca_tabla = (
    tabla_resultados_por_falla_long.loc[
        tabla_resultados_por_falla_long[
            "Modelo"
        ] == "PCA"
    ]
    .set_index("falla")
)

tabla_por_falla["PCA_Deteccion_pct"] = (
    tabla_por_falla["Falla"]
    .map(
        pca_tabla[
            "tasa_deteccion_evaluable_pct"
        ]
    )
)

tabla_por_falla["PCA_Retraso_mediano_min"] = (
    tabla_por_falla["Falla"]
    .map(
        pca_tabla[
            "retraso_mediano_min"
        ]
    )
)

tabla_por_falla["PCA_Prealarmas"] = (
    tabla_por_falla["Falla"]
    .map(
        pca_tabla[
            "corridas_prealarma"
        ]
    )
)

tabla_por_falla["PCA_AP"] = (
    tabla_por_falla["Falla"]
    .map(
        pca_tabla[
            "AP"
        ]
    )
)

In [ ]:
if_tabla = (
    tabla_resultados_por_falla_long.loc[
        tabla_resultados_por_falla_long[
            "Modelo"
        ] == "Isolation Forest"
    ]
    .set_index("falla")
)

tabla_por_falla["IF_Deteccion_pct"] = (
    tabla_por_falla["Falla"]
    .map(
        if_tabla[
            "tasa_deteccion_evaluable_pct"
        ]
    )
)

tabla_por_falla["IF_Retraso_mediano_min"] = (
    tabla_por_falla["Falla"]
    .map(
        if_tabla[
            "retraso_mediano_min"
        ]
    )
)

tabla_por_falla["IF_Prealarmas"] = (
    tabla_por_falla["Falla"]
    .map(
        if_tabla[
            "corridas_prealarma"
        ]
    )
)

tabla_por_falla["IF_AP"] = (
    tabla_por_falla["Falla"]
    .map(
        if_tabla[
            "AP"
        ]
    )
)

In [ ]:
ae_tabla = (
    tabla_resultados_por_falla_long.loc[
        tabla_resultados_por_falla_long[
            "Modelo"
        ] == "Autoencoder"
    ]
    .set_index("falla")
)

tabla_por_falla["AE_Deteccion_pct"] = (
    tabla_por_falla["Falla"]
    .map(
        ae_tabla[
            "tasa_deteccion_evaluable_pct"
        ]
    )
)

tabla_por_falla["AE_Retraso_mediano_min"] = (
    tabla_por_falla["Falla"]
    .map(
        ae_tabla[
            "retraso_mediano_min"
        ]
    )
)

tabla_por_falla["AE_Prealarmas"] = (
    tabla_por_falla["Falla"]
    .map(
        ae_tabla[
            "corridas_prealarma"
        ]
    )
)

tabla_por_falla["AE_AP"] = (
    tabla_por_falla["Falla"]
    .map(
        ae_tabla[
            "AP"
        ]
    )
)

In [ ]:
display(
    tabla_por_falla.round(3)
)

In [ ]:
print(
    "Filas:",
    len(tabla_por_falla)
)

print(
    "Valores faltantes:",
    tabla_por_falla
    .isna()
    .sum()
    .sum()
)

print(
    "\nFallas:",
    tabla_por_falla[
        "Falla"
    ].tolist()
)

In [ ]:
ruta_tabla_por_falla = (
    RESULTS_DIR /
    "tabla_final_por_falla_3_modelos.csv"
)

tabla_por_falla.to_csv(
    ruta_tabla_por_falla,
    index=False
)

print(
    "Guardada en:",
    ruta_tabla_por_falla
)

In [ ]:
tabla_resultados_por_falla_long.to_csv(
    RESULTS_DIR /
    "tabla_final_por_falla_formato_largo.csv",
    index=False
)

In [ ]:
tabla_pca_memoria = tabla_por_falla[
    [
        "Falla",
        "PCA_Deteccion_pct",
        "PCA_Retraso_mediano_min",
        "PCA_Prealarmas",
        "PCA_AP"
    ]
].copy()

tabla_pca_memoria.columns = [
    "Falla",
    "Detección (%)",
    "Retraso mediano (min)",
    "Prealarmas",
    "AP"
]

display(
    tabla_pca_memoria.round(3)
)

In [ ]:
tabla_if_memoria = tabla_por_falla[
    [
        "Falla",
        "IF_Deteccion_pct",
        "IF_Retraso_mediano_min",
        "IF_Prealarmas",
        "IF_AP"
    ]
].copy()

tabla_if_memoria.columns = [
    "Falla",
    "Detección (%)",
    "Retraso mediano (min)",
    "Prealarmas",
    "AP"
]

display(
    tabla_if_memoria.round(3)
)

In [ ]:
tabla_ae_memoria = tabla_por_falla[
    [
        "Falla",
        "AE_Deteccion_pct",
        "AE_Retraso_mediano_min",
        "AE_Prealarmas",
        "AE_AP"
    ]
].copy()

tabla_ae_memoria.columns = [
    "Falla",
    "Detección (%)",
    "Retraso mediano (min)",
    "Prealarmas",
    "AP"
]

display(
    tabla_ae_memoria.round(3)
)

In [ ]:
tabla_pca_memoria.to_csv(
    RESULTS_DIR / "tabla_PCA_por_falla_memoria.csv",
    index=False
)

tabla_if_memoria.to_csv(
    RESULTS_DIR / "tabla_IF_por_falla_memoria.csv",
    index=False
)

tabla_ae_memoria.to_csv(
    RESULTS_DIR / "tabla_AE_por_falla_memoria.csv",
    index=False
)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def crear_pipeline_tfm():

    fig, ax = plt.subplots(figsize=(15, 8))
    ax.set_xlim(0, 15)
    ax.set_ylim(0, 10)
    ax.axis("off")

    def caja(x, y, w, h, texto, fontsize=10):
        rect = FancyBboxPatch(
            (x, y),
            w,
            h,
            boxstyle="round,pad=0.03",
            linewidth=1.2,
            facecolor="white",
            edgecolor="black"
        )
        ax.add_patch(rect)
        ax.text(
            x + w/2,
            y + h/2,
            texto,
            ha="center",
            va="center",
            fontsize=fontsize,
            wrap=True
        )

    def flecha(x1, y1, x2, y2):
        arr = FancyArrowPatch(
            (x1, y1),
            (x2, y2),
            arrowstyle="->",
            mutation_scale=15,
            linewidth=1.2
        )
        ax.add_patch(arr)

    # =========================================================
    # TÍTULO DE BLOQUES
    # =========================================================

    ax.text(
        3.3, 9.5,
        "DESARROLLO Y SELECCIÓN",
        ha="center",
        fontsize=13,
        fontweight="bold"
    )

    ax.text(
        11.6, 9.5,
        "EVALUACIÓN FINAL INDEPENDIENTE",
        ha="center",
        fontsize=13,
        fontweight="bold"
    )

    # =========================================================
    # BLOQUE IZQUIERDO: DESARROLLO
    # =========================================================

    caja(
        0.5, 7.8, 3.2, 1.0,
        "FaultFree Training\n500 corridas normales"
    )

    caja(
        4.0, 7.8, 3.2, 1.0,
        "Faulty Training\n20 fallas × 500 corridas"
    )

    caja(
        0.5, 6.0, 3.2, 1.0,
        "Split por corrida\n400 entrenamiento / 100 validación"
    )

    caja(
        0.5, 4.2, 3.2, 1.0,
        "StandardScaler\najustado solo con entrenamiento normal"
    )

    caja(
        4.0, 4.2, 3.2, 1.0,
        "Modelos candidatos\nPCA · Isolation Forest · Autoencoder"
    )

    caja(
        4.0, 2.4, 3.2, 1.0,
        "Calibración\numbrales + regla temporal 3-de-5"
    )

    caja(
        0.5, 2.4, 3.2, 1.0,
        "Validación normal\nFAR y episodios falsos"
    )

    caja(
        2.25, 0.6, 3.2, 1.0,
        "Selección del modelo principal\nusando datos de desarrollo"
    )

    # Flechas desarrollo
    flecha(2.1, 7.8, 2.1, 7.0)
    flecha(2.1, 6.0, 2.1, 5.2)
    flecha(3.7, 4.7, 4.0, 4.7)
    flecha(5.6, 4.2, 5.6, 3.4)
    flecha(4.0, 2.9, 3.7, 2.9)
    flecha(2.1, 2.4, 3.5, 1.6)
    flecha(5.6, 2.4, 4.9, 1.6)

    # Faulty Training hacia modelos/evaluación
    flecha(5.6, 7.8, 5.6, 5.2)

    # =========================================================
    # BLOQUE DERECHO: TESTING
    # =========================================================

    caja(
        8.3, 7.8, 3.0, 1.0,
        "FaultFree Testing\n500 corridas · 24 000 h normales"
    )

    caja(
        11.7, 7.8, 3.0, 1.0,
        "Faulty Testing\n20 fallas × 500 corridas"
    )

    caja(
        8.3, 5.8, 3.0, 1.0,
        "Evaluación normal\nFAR persistente · episodios/100 h"
    )

    caja(
        11.7, 5.8, 3.0, 1.0,
        "Evaluación por falla\nDetección · retraso · prealarmas"
    )

    caja(
        10.0, 3.8, 3.0, 1.0,
        "Precision–Recall / AP\nEvaluación independiente del umbral"
    )

    caja(
        10.0, 2.0, 3.0, 1.0,
        "Interpretabilidad\nContribuciones PCA · SHAP · error AE"
    )

    caja(
        10.0, 0.2, 3.0, 1.0,
        "Prototipo final\nscore → 3-de-5 → alerta → explicación"
    )

    # Flechas testing
    flecha(9.8, 7.8, 9.8, 6.8)
    flecha(13.2, 7.8, 13.2, 6.8)

    flecha(9.8, 5.8, 10.8, 4.8)
    flecha(13.2, 5.8, 12.2, 4.8)

    flecha(11.5, 3.8, 11.5, 3.0)
    flecha(11.5, 2.0, 11.5, 1.2)

    # Separación visual desarrollo/testing
    ax.axvline(
        7.65,
        linestyle="--",
        linewidth=1
    )

    ax.text(
        7.65, 9.0,
        "Modelo congelado",
        ha="center",
        fontsize=10,
        rotation=90
    )

    plt.tight_layout()

    ruta = (
        RESULTS_DIR /
        "Figura_pipeline_metodologico_TFM.png"
    )

    plt.savefig(
        ruta,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print("Figura guardada en:")
    print(ruta)


crear_pipeline_tfm()

In [ ]:
datos_componentes_fig = (
    tabla_final_sensibilidad_componentes[
        [
            "Componentes",
            "Varianza_explicada_pct",
            "Deteccion_global_pct"
        ]
    ]
    .copy()
)

plt.figure(figsize=(8, 5))

plt.plot(
    datos_componentes_fig["Componentes"],
    datos_componentes_fig["Deteccion_global_pct"],
    marker="o",
    linewidth=2
)

for _, fila in datos_componentes_fig.iterrows():

    plt.annotate(
        f"{fila['Varianza_explicada_pct']:.1f}% var.",
        (
            fila["Componentes"],
            fila["Deteccion_global_pct"]
        ),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center"
    )

plt.xlabel(
    "Número de componentes principales"
)

plt.ylabel(
    "Tasa global de detección (%)"
)

plt.title(
    "Sensibilidad del detector PCA al número de componentes"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

ruta_fig_componentes = (
    RESULTS_DIR /
    "Figura_sensibilidad_componentes_PCA.png"
)

plt.savefig(
    ruta_fig_componentes,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
fallas_dificiles_componentes = (
    tabla_deteccion_componentes.loc[
        tabla_deteccion_componentes[
            "falla"
        ].isin(
            [3, 9, 15]
        )
    ]
    .copy()
)

fallas_dificiles_componentes.plot(
    x="falla",
    y=[
        "PCA_31",
        "PCA_36",
        "PCA_41"
    ],
    kind="bar",
    figsize=(9, 5)
)

plt.xlabel(
    "Tipo de falla"
)

plt.ylabel(
    "Tasa de detección (%)"
)

plt.title(
    "Efecto del número de componentes en las fallas difíciles"
)

plt.xticks(
    rotation=0
)

plt.legend(
    [
        "31 componentes",
        "36 componentes",
        "41 componentes"
    ]
)

plt.tight_layout()

ruta_fig_dificiles = (
    RESULTS_DIR /
    "Figura_componentes_fallas_3_9_15.png"
)

plt.savefig(
    ruta_fig_dificiles,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
nombres_historial = [
    "history_ae",
    "history",
    "history_autoencoder"
]

for nombre in nombres_historial:
    if nombre in globals():
        objeto = globals()[nombre]

        print(
            nombre,
            "→ EXISTE"
        )

        if hasattr(objeto, "history"):
            print(
                "  claves:",
                objeto.history.keys()
            )
    else:
        print(
            nombre,
            "→ no existe"
        )

## 17. Reproducción controlada del Autoencoder y configuraciones finales

Recuperación/configuración del Autoencoder, reproducción del entrenamiento final y consolidación de hiperparámetros.


In [ ]:
ruta_config_ae = (
    MODELS_DIR /
    "autoencoder_baseline_v1_config.joblib"
)

print(
    "Existe configuración:",
    ruta_config_ae.exists()
)

if ruta_config_ae.exists():

    config_ae = joblib.load(
        ruta_config_ae
    )

    print("\nClaves guardadas:")
    print(
        config_ae.keys()
        if hasattr(config_ae, "keys")
        else type(config_ae)
    )

In [ ]:
ruta_config_ae = (
    MODELS_DIR /
    "autoencoder_baseline_v1_config.joblib"
)

print(
    "Existe configuración:",
    ruta_config_ae.exists()
)

if ruta_config_ae.exists():

    config_ae = joblib.load(
        ruta_config_ae
    )

    print("\nTipo de objeto:")
    print(type(config_ae))

    print("\nClaves guardadas:")

    if hasattr(config_ae, "keys"):
        for clave in config_ae.keys():
            print("-", clave)

In [ ]:
ruta_config_ae = (
    MODELS_DIR /
    "autoencoder_baseline_v1_config.joblib"
)

print(
    "Existe configuración:",
    ruta_config_ae.exists()
)

if ruta_config_ae.exists():

    config_ae = joblib.load(
        ruta_config_ae
    )

    print("\nTipo de objeto:")
    print(type(config_ae))

    print("\nClaves guardadas:")

    if hasattr(config_ae, "keys"):
        for clave in config_ae.keys():
            print("-", clave)

In [ ]:
if hasattr(config_ae, "items"):

    print("\nResumen del contenido:")

    for clave, valor in config_ae.items():

        if hasattr(valor, "shape"):
            print(
                f"{clave:25s} → "
                f"{type(valor).__name__} | "
                f"shape={valor.shape}"
            )

        elif isinstance(
            valor,
            (list, tuple, dict, str, int, float, bool)
        ):
            texto = str(valor)

            if len(texto) > 200:
                texto = texto[:200] + "..."

            print(
                f"{clave:25s} → {texto}"
            )

        else:
            print(
                f"{clave:25s} → "
                f"{type(valor).__name__}"
            )

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

ruta_modelo_ae = (
    MODELS_DIR /
    "autoencoder_baseline_v1.keras"
)

autoencoder_guardado = keras.models.load_model(
    ruta_modelo_ae
)

print("Modelo recuperado correctamente.\n")

autoencoder_guardado.summary()

print("\nLoss:")
print(autoencoder_guardado.loss)

print("\nOptimizador:")
print(type(autoencoder_guardado.optimizer).__name__)

print("\nConfiguración del optimizador:")
print(autoencoder_guardado.optimizer.get_config())

In [ ]:
corridas_train_ae = set(
    map(int, config_ae["corridas_train"])
)

corridas_val_ae = set(
    map(int, config_ae["corridas_val"])
)

corridas_train_actual = set(
    map(int, df_train_normal["simulationRun"].unique())
)

corridas_val_actual = set(
    map(int, df_val_normal["simulationRun"].unique())
)

print(
    "Train coincide:",
    corridas_train_ae
    == corridas_train_actual
)

print(
    "Validation coincide:",
    corridas_val_ae
    == corridas_val_actual
)

print(
    "N.º corridas train AE:",
    len(corridas_train_ae)
)

print(
    "N.º corridas validación AE:",
    len(corridas_val_ae)
)

In [ ]:
for capa in autoencoder_guardado.layers:

    if hasattr(capa, "activation"):

        print(
            f"{capa.name:20s} → "
            f"{capa.activation.__name__}"
        )

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import random

SEMILLA_REPRODUCCION = 42

random.seed(
    SEMILLA_REPRODUCCION
)

np.random.seed(
    SEMILLA_REPRODUCCION
)

tf.random.set_seed(
    SEMILLA_REPRODUCCION
)

In [ ]:
entrada = keras.Input(
    shape=(52,),
    name="entrada"
)

x = layers.Dense(
    32,
    activation="relu",
    name="encoder_32"
)(entrada)

x = layers.Dense(
    16,
    activation="relu",
    name="encoder_16"
)(x)

latente = layers.Dense(
    8,
    activation="relu",
    name="espacio_latente"
)(x)

x = layers.Dense(
    16,
    activation="relu",
    name="decoder_16"
)(latente)

x = layers.Dense(
    32,
    activation="relu",
    name="decoder_32"
)(x)

salida = layers.Dense(
    52,
    activation="linear",
    name="reconstruccion"
)(x)

autoencoder_repro = keras.Model(
    entrada,
    salida,
    name="autoencoder_TEP_reproduccion"
)

autoencoder_repro.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse"
)

autoencoder_repro.summary()

In [ ]:
early_stopping_repro = (
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        mode="min"
    )
)

In [ ]:
history_ae_repro = autoencoder_repro.fit(
    X_train_pca_scaled,
    X_train_pca_scaled,

    validation_data=(
        X_val_pca_scaled,
        X_val_pca_scaled
    ),

    epochs=50,
    batch_size=512,

    callbacks=[
        early_stopping_repro
    ],

    verbose=1,
    shuffle=True
)

In [ ]:
loss_train_repro = (
    history_ae_repro.history["loss"]
)

loss_val_repro = (
    history_ae_repro.history["val_loss"]
)

mejor_epoca_repro = (
    np.argmin(
        loss_val_repro
    )
    + 1
)

print(
    "Épocas ejecutadas:",
    len(loss_train_repro)
)

print(
    "Mejor época:",
    mejor_epoca_repro
)

print(
    "Loss final entrenamiento:",
    loss_train_repro[-1]
)

print(
    "Loss final validación:",
    loss_val_repro[-1]
)

print(
    "Mejor val_loss:",
    min(loss_val_repro)
)

In [ ]:
epocas_repro = np.arange(
    1,
    len(loss_train_repro) + 1
)

plt.figure(
    figsize=(9, 5)
)

plt.plot(
    epocas_repro,
    loss_train_repro,
    marker="o",
    markersize=3,
    label="Entrenamiento"
)

plt.plot(
    epocas_repro,
    loss_val_repro,
    marker="o",
    markersize=3,
    label="Validación"
)

plt.axvline(
    mejor_epoca_repro,
    linestyle="--",
    linewidth=1.5,
    label=(
        f"Mejor época de la reproducción "
        f"({mejor_epoca_repro})"
    )
)

plt.xlabel(
    "Época"
)

plt.ylabel(
    "Error cuadrático medio (MSE)"
)

plt.title(
    "Convergencia del Autoencoder en entrenamiento y validación"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

ruta_loss_ae = (
    RESULTS_DIR /
    "Figura_perdidas_Autoencoder_reproduccion.png"
)

plt.savefig(
    ruta_loss_ae,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Figura guardada en:",
    ruta_loss_ae
)

In [ ]:
ruta_if = (
    MODELS_DIR /
    "isolation_forest_baseline_v1.joblib"
)

checkpoint_if = joblib.load(
    ruta_if
)

print(
    "Claves IF:",
    checkpoint_if.keys()
)

In [ ]:
if "if_model" in checkpoint_if:

    if_model = checkpoint_if["if_model"]

elif "model" in checkpoint_if:

    if_model = checkpoint_if["model"]

else:

    raise KeyError(
        "No se encontró el modelo IF "
        "dentro del checkpoint."
    )

print(
    "Isolation Forest recuperado."
)PARTITION_TEST_DIR = (
    PROJECT_DIR
    / "Datos_Procesados"
    / "Faulty_Testing_Parquet_v1"
)

print(
    "Particiones testing:",
    PARTITION_TEST_DIR.exists()
)

In [ ]:
PARTITION_TEST_DIR = (
    PROJECT_DIR
    / "Datos_Procesados"
    / "Faulty_Testing_Parquet_v1"
)

print(
    "Particiones testing:",
    PARTITION_TEST_DIR.exists()
)

In [ ]:
archivo_ff_test = (
    DATA_DIR /
    "TEP_FaultFree_Testing.RData"
)

resultado_ff_test = pyreadr.read_r(
    str(archivo_ff_test)
)

nombre_ff_test = next(
    iter(resultado_ff_test.keys())
)

ff_test_pr = (
    resultado_ff_test[
        nombre_ff_test
    ]
    .copy()
)

del resultado_ff_test
gc.collect()

print(
    "FaultFree Testing:",
    ff_test_pr.shape
)

In [ ]:
def calcular_scores_tres_modelos(
    df,
    feature_cols,
    scaler,
    pca_model,
    T2_threshold,
    SPE_threshold,
    if_model,
    autoencoder
):

    X = df[
        feature_cols
    ]

    # ========================================================
    # ESCALADO
    # ========================================================

    X_scaled = scaler.transform(
        X
    )

    # ========================================================
    # PCA
    # ========================================================

    scores_pca = pca_model.transform(
        X_scaled
    )

    eigenvalues = (
        pca_model.explained_variance_
    )

    T2 = np.sum(
        (scores_pca ** 2)
        / eigenvalues,
        axis=1
    )

    X_rec_pca = (
        pca_model.inverse_transform(
            scores_pca
        )
    )

    SPE = np.sum(
        (
            X_scaled
            - X_rec_pca
        ) ** 2,
        axis=1
    )

    score_pca = np.maximum(
        T2 / T2_threshold,
        SPE / SPE_threshold
    )

    # ========================================================
    # ISOLATION FOREST
    # ========================================================

    score_if = (
        -if_model.score_samples(
            X
        )
    )

    # ========================================================
    # AUTOENCODER
    # ========================================================

    X_rec_ae = (
        autoencoder.predict(
            X_scaled,
            batch_size=2048,
            verbose=0
        )
    )

    score_ae = np.mean(
        (
            X_scaled
            - X_rec_ae
        ) ** 2,
        axis=1
    )

    return (
        score_pca,
        score_if,
        score_ae
    )

In [ ]:
score_normal_pca, \
score_normal_if, \
score_normal_ae = (
    calcular_scores_tres_modelos(
        df=ff_test_pr,
        feature_cols=feature_cols,
        scaler=scaler,
        pca_model=pca_model,
        T2_threshold=T2_threshold,
        SPE_threshold=SPE_threshold,
        if_model=if_model,
        autoencoder=autoencoder_guardado
    )
)

print(
    len(score_normal_pca),
    len(score_normal_if),
    len(score_normal_ae)
)

In [ ]:
def cargar_post_falla_testing(
    numero_falla
):

    df = pd.read_parquet(
        PARTITION_TEST_DIR /
        f"falla_{numero_falla:02d}"
    )

    df = (
        df.loc[
            df["sample"] >= 161
        ]
        .sort_values(
            [
                "simulationRun",
                "sample"
            ]
        )
        .reset_index(drop=True)
    )

    return df

In [ ]:
falla1_post = (
    cargar_post_falla_testing(1)
)

falla3_post = (
    cargar_post_falla_testing(3)
)

print(
    "Falla 1:",
    falla1_post.shape
)

print(
    "Falla 3:",
    falla3_post.shape
)

In [ ]:
score_f1_pca, \
score_f1_if, \
score_f1_ae = (
    calcular_scores_tres_modelos(
        df=falla1_post,
        feature_cols=feature_cols,
        scaler=scaler,
        pca_model=pca_model,
        T2_threshold=T2_threshold,
        SPE_threshold=SPE_threshold,
        if_model=if_model,
        autoencoder=autoencoder_guardado
    )
)

In [ ]:
score_f3_pca, \
score_f3_if, \
score_f3_ae = (
    calcular_scores_tres_modelos(
        df=falla3_post,
        feature_cols=feature_cols,
        scaler=scaler,
        pca_model=pca_model,
        T2_threshold=T2_threshold,
        SPE_threshold=SPE_threshold,
        if_model=if_model,
        autoencoder=autoencoder_guardado
    )
)

In [ ]:
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)


def obtener_curva_pr(
    score_normal,
    score_falla
):

    y_true = np.concatenate([
        np.zeros(
            len(score_normal),
            dtype=np.int8
        ),
        np.ones(
            len(score_falla),
            dtype=np.int8
        )
    ])

    y_score = np.concatenate([
        score_normal,
        score_falla
    ])

    precision, recall, _ = (
        precision_recall_curve(
            y_true,
            y_score,
            drop_intermediate=True
        )
    )

    AP = average_precision_score(
        y_true,
        y_score
    )

    prevalencia = (
        y_true.mean()
    )

    return (
        precision,
        recall,
        AP,
        prevalencia
    )

In [ ]:
p_pca_1, r_pca_1, ap_pca_1, prev_1 = (
    obtener_curva_pr(
        score_normal_pca,
        score_f1_pca
    )
)

p_if_1, r_if_1, ap_if_1, _ = (
    obtener_curva_pr(
        score_normal_if,
        score_f1_if
    )
)

p_ae_1, r_ae_1, ap_ae_1, _ = (
    obtener_curva_pr(
        score_normal_ae,
        score_f1_ae
    )
)

In [ ]:
plt.figure(
    figsize=(8, 6)
)

plt.plot(
    r_pca_1,
    p_pca_1,
    label=f"PCA (AP={ap_pca_1:.3f})"
)

plt.plot(
    r_if_1,
    p_if_1,
    label=f"Isolation Forest (AP={ap_if_1:.3f})"
)

plt.plot(
    r_ae_1,
    p_ae_1,
    label=f"Autoencoder (AP={ap_ae_1:.3f})"
)

plt.axhline(
    y=prev_1,
    linestyle="--",
    label=(
        f"Referencia aleatoria "
        f"({prev_1:.3f})"
    )
)

plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.title(
    "Curvas Precision–Recall – Falla 1"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

ruta_pr_f1 = (
    RESULTS_DIR /
    "Figura_PR_Falla_1.png"
)

plt.savefig(
    ruta_pr_f1,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
p_pca_3, r_pca_3, ap_pca_3, prev_3 = (
    obtener_curva_pr(
        score_normal_pca,
        score_f3_pca
    )
)

p_if_3, r_if_3, ap_if_3, _ = (
    obtener_curva_pr(
        score_normal_if,
        score_f3_if
    )
)

p_ae_3, r_ae_3, ap_ae_3, _ = (
    obtener_curva_pr(
        score_normal_ae,
        score_f3_ae
    )
)

In [ ]:
plt.figure(
    figsize=(8, 6)
)

plt.plot(
    r_pca_3,
    p_pca_3,
    label=f"PCA (AP={ap_pca_3:.3f})"
)

plt.plot(
    r_if_3,
    p_if_3,
    label=f"Isolation Forest (AP={ap_if_3:.3f})"
)

plt.plot(
    r_ae_3,
    p_ae_3,
    label=f"Autoencoder (AP={ap_ae_3:.3f})"
)

plt.axhline(
    y=prev_3,
    linestyle="--",
    label=(
        f"Referencia aleatoria "
        f"({prev_3:.3f})"
    )
)

plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.title(
    "Curvas Precision–Recall – Falla 3"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

ruta_pr_f3 = (
    RESULTS_DIR /
    "Figura_PR_Falla_3.png"
)

plt.savefig(
    ruta_pr_f3,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
def procesar_corrida_pca_figura(
    numero_falla,
    numero_corrida,
    muestra_inicio_falla=161,
    ventana=5,
    minimo_alarmas=3,
    minutos_por_muestra=3
):

    df_falla = pd.read_parquet(
        PARTITION_TEST_DIR /
        f"falla_{numero_falla:02d}"
    )

    df = (
        df_falla.loc[
            df_falla["simulationRun"].astype(int)
            == int(numero_corrida)
        ]
        .copy()
        .sort_values("sample")
        .reset_index(drop=True)
    )

    del df_falla
    gc.collect()

    X = df[feature_cols]

    X_scaled = scaler.transform(X)

    scores = pca_model.transform(
        X_scaled
    )

    eigenvalues = (
        pca_model.explained_variance_
    )

    T2 = np.sum(
        (scores ** 2)
        / eigenvalues,
        axis=1
    )

    X_rec = pca_model.inverse_transform(
        scores
    )

    residuals = (
        X_scaled
        - X_rec
    )

    SPE = np.sum(
        residuals ** 2,
        axis=1
    )

    score = np.maximum(
        T2 / T2_threshold,
        SPE / SPE_threshold
    )

    resultados = df[
        [
            "faultNumber",
            "simulationRun",
            "sample"
        ]
    ].copy()

    resultados["tiempo_h"] = (
        (resultados["sample"] - 1)
        * minutos_por_muestra
        / 60
    )

    resultados["T2"] = T2
    resultados["SPE"] = SPE
    resultados["score_pca"] = score

    resultados["alarma_puntual"] = (
        resultados["score_pca"] > 1
    )

    resultados["alarmas_ventana"] = (
        resultados
        .groupby("simulationRun")[
            "alarma_puntual"
        ]
        .transform(
            lambda s:
                s.astype(int)
                .rolling(
                    window=ventana,
                    min_periods=ventana
                )
                .sum()
        )
    )

    resultados["alarma_persistente"] = (
        resultados["alarmas_ventana"]
        >= minimo_alarmas
    )

    resultados["inicio_episodio"] = (
        resultados["alarma_persistente"]
        &
        ~resultados
        .groupby("simulationRun")[
            "alarma_persistente"
        ]
        .shift(fill_value=False)
    )

    detecciones_post = resultados.loc[
        resultados["inicio_episodio"]
        &
        (
            resultados["sample"]
            >= muestra_inicio_falla
        )
    ]

    if len(detecciones_post) > 0:

        muestra_deteccion = int(
            detecciones_post.iloc[0]["sample"]
        )

        retraso_min = (
            muestra_deteccion
            - muestra_inicio_falla
        ) * minutos_por_muestra

    else:
        muestra_deteccion = None
        retraso_min = None

    info = {
        "falla": numero_falla,
        "corrida": numero_corrida,
        "muestra_inicio_falla":
            muestra_inicio_falla,
        "muestra_deteccion":
            muestra_deteccion,
        "retraso_min":
            retraso_min
    }

    datos_interpretacion = {
        "df": df,
        "X_scaled": X_scaled,
        "scores": scores,
        "residuals": residuals
    }

    return (
        resultados,
        info,
        datos_interpretacion
    )

In [ ]:
resultados_f1, info_f1, datos_f1 = (
    procesar_corrida_pca_figura(
        numero_falla=1,
        numero_corrida=1
    )
)

resultados_f3, info_f3, datos_f3 = (
    procesar_corrida_pca_figura(
        numero_falla=3,
        numero_corrida=1
    )
)

print(info_f1)
print(info_f3)

In [ ]:
plt.figure(
    figsize=(11, 5.5)
)

plt.plot(
    resultados_f1["tiempo_h"],
    resultados_f1["score_pca"],
    linewidth=1.4,
    label="Score PCA"
)

plt.axhline(
    y=1,
    linestyle="--",
    linewidth=1.4,
    label="Umbral de anomalía"
)

tiempo_inicio_f1 = (
    (161 - 1)
    * 3
    / 60
)

plt.axvline(
    x=tiempo_inicio_f1,
    linestyle="--",
    linewidth=1.4,
    label="Inicio de la falla"
)

if info_f1["muestra_deteccion"] is not None:

    tiempo_deteccion_f1 = (
        (
            info_f1["muestra_deteccion"]
            - 1
        )
        * 3
        / 60
    )

    plt.axvline(
        x=tiempo_deteccion_f1,
        linestyle="-.",
        linewidth=1.4,
        label=(
            f"Alarma confirmada "
            f"({info_f1['retraso_min']} min)"
        )
    )

plt.xlabel(
    "Tiempo de simulación (h)"
)

plt.ylabel(
    "Score de anomalía normalizado"
)

plt.title(
    "Evolución temporal del score PCA – "
    "Falla 1, corrida 1"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

ruta_score_f1 = (
    RESULTS_DIR /
    "Figura_score_PCA_falla_1_corrida_1.png"
)

plt.savefig(
    ruta_score_f1,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
zona_f1 = resultados_f1.loc[
    (
        resultados_f1["sample"] >= 150
    )
    &
    (
        resultados_f1["sample"] <= 180
    )
].copy()

plt.figure(
    figsize=(9, 5.5)
)

plt.plot(
    zona_f1["tiempo_h"],
    zona_f1["score_pca"],
    marker="o",
    markersize=4,
    linewidth=1.4,
    label="Score PCA"
)

plt.axhline(
    y=1,
    linestyle="--",
    label="Umbral"
)

plt.axvline(
    x=tiempo_inicio_f1,
    linestyle="--",
    label="Inicio de la falla"
)

plt.axvline(
    x=tiempo_deteccion_f1,
    linestyle="-.",
    label=(
        "Alarma confirmada "
        f"({info_f1['retraso_min']} min)"
    )
)

puntos_alarma = zona_f1.loc[
    zona_f1["alarma_puntual"]
]

plt.scatter(
    puntos_alarma["tiempo_h"],
    puntos_alarma["score_pca"],
    s=35,
    label="Alarma puntual"
)

plt.xlabel(
    "Tiempo de simulación (h)"
)

plt.ylabel(
    "Score de anomalía normalizado"
)

plt.title(
    "Detalle de la detección temprana – "
    "Falla 1, corrida 1"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

ruta_zoom_f1 = (
    RESULTS_DIR /
    "Figura_zoom_deteccion_falla_1.png"
)

plt.savefig(
    ruta_zoom_f1,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(
    figsize=(11, 5.5)
)

plt.plot(
    resultados_f3["tiempo_h"],
    resultados_f3["score_pca"],
    linewidth=1.4,
    label="Score PCA"
)

plt.axhline(
    y=1,
    linestyle="--",
    linewidth=1.4,
    label="Umbral de anomalía"
)

tiempo_inicio_f3 = (
    (161 - 1)
    * 3
    / 60
)

plt.axvline(
    x=tiempo_inicio_f3,
    linestyle="--",
    linewidth=1.4,
    label="Inicio de la falla"
)

if info_f3["muestra_deteccion"] is not None:

    tiempo_deteccion_f3 = (
        (
            info_f3["muestra_deteccion"]
            - 1
        )
        * 3
        / 60
    )

    plt.axvline(
        x=tiempo_deteccion_f3,
        linestyle="-.",
        linewidth=1.4,
        label=(
            f"Alarma confirmada "
            f"({info_f3['retraso_min']} min)"
        )
    )

plt.xlabel(
    "Tiempo de simulación (h)"
)

plt.ylabel(
    "Score de anomalía normalizado"
)

plt.title(
    "Evolución temporal del score PCA – "
    "Falla 3, corrida 1"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

ruta_score_f3 = (
    RESULTS_DIR /
    "Figura_score_PCA_falla_3_corrida_1.png"
)

plt.savefig(
    ruta_score_f3,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
muestra_interpretacion = (
    info_f1["muestra_deteccion"]
)

posicion = np.where(
    datos_f1["df"]["sample"]
    .to_numpy()
    == muestra_interpretacion
)[0][0]

residual = (
    datos_f1[
        "residuals"
    ][posicion]
)

contrib_SPE = (
    residual ** 2
)

In [ ]:
tabla_contrib_f1 = pd.DataFrame({
    "variable":
        feature_cols,

    "contribucion":
        contrib_SPE
})

In [ ]:
print(
    "Diccionario existe:",
    "nombres_variables_tep"
    in globals()
)

In [ ]:
nombres_variables_tep = {

    # Variables medidas XMEAS
    "xmeas_1": "Caudal alimentación A - corriente 1",
    "xmeas_2": "Caudal alimentación D - corriente 2",
    "xmeas_3": "Caudal alimentación E - corriente 3",
    "xmeas_4": "Caudal alimentación A/C - corriente 4",
    "xmeas_5": "Caudal de recirculación",
    "xmeas_6": "Caudal de alimentación al reactor",

    "xmeas_7": "Presión del reactor",
    "xmeas_8": "Nivel del reactor",
    "xmeas_9": "Temperatura del reactor",
    "xmeas_10": "Caudal de purga",

    "xmeas_11": "Temperatura del separador",
    "xmeas_12": "Nivel del separador",
    "xmeas_13": "Presión del separador",
    "xmeas_14": "Caudal de salida del separador",

    "xmeas_15": "Nivel del stripper",
    "xmeas_16": "Presión del stripper",
    "xmeas_17": "Caudal de salida del stripper",
    "xmeas_18": "Temperatura del stripper",
    "xmeas_19": "Caudal de vapor al stripper",

    "xmeas_20": "Trabajo del compresor",

    "xmeas_21": "T salida agua enfriamiento reactor",
    "xmeas_22": "T salida agua enfriamiento separador",

    # Composición alimentación al reactor
    "xmeas_23": "Componente A - alimentación reactor",
    "xmeas_24": "Componente B - alimentación reactor",
    "xmeas_25": "Componente C - alimentación reactor",
    "xmeas_26": "Componente D - alimentación reactor",
    "xmeas_27": "Componente E - alimentación reactor",
    "xmeas_28": "Componente F - alimentación reactor",

    # Gas de purga
    "xmeas_29": "Componente A - gas de purga",
    "xmeas_30": "Componente B - gas de purga",
    "xmeas_31": "Componente C - gas de purga",
    "xmeas_32": "Componente D - gas de purga",
    "xmeas_33": "Componente E - gas de purga",
    "xmeas_34": "Componente F - gas de purga",
    "xmeas_35": "Componente G - gas de purga",
    "xmeas_36": "Componente H - gas de purga",

    # Producto
    "xmeas_37": "Componente D - producto",
    "xmeas_38": "Componente E - producto",
    "xmeas_39": "Componente F - producto",
    "xmeas_40": "Componente G - producto",
    "xmeas_41": "Componente H - producto",

    # Variables manipuladas
    "xmv_1": "Válvula alimentación D",
    "xmv_2": "Válvula alimentación E",
    "xmv_3": "Válvula alimentación A",
    "xmv_4": "Válvula alimentación A/C",
    "xmv_5": "Válvula recirculación compresor",
    "xmv_6": "Válvula de purga",
    "xmv_7": "Válvula salida separador",
    "xmv_8": "Válvula producto stripper",
    "xmv_9": "Válvula vapor stripper",
    "xmv_10": "Agua de enfriamiento reactor",
    "xmv_11": "Agua de enfriamiento condensador"
}

print(
    "Variables documentadas:",
    len(nombres_variables_tep)
)

In [ ]:
tabla_contrib_f1["nombre"] = (
    tabla_contrib_f1["variable"]
    .map(nombres_variables_tep)
    .fillna(
        tabla_contrib_f1["variable"]
    )
)

top_contrib_f1 = (
    tabla_contrib_f1
    .sort_values(
        "contribucion",
        ascending=False
    )
    .head(10)
    .copy()
)

display(
    top_contrib_f1.round(4)
)

In [ ]:
datos_plot = (
    top_contrib_f1
    .sort_values(
        "contribucion"
    )
)

plt.figure(
    figsize=(9, 5.5)
)

plt.barh(
    datos_plot["nombre"],
    datos_plot["contribucion"]
)

plt.xlabel(
    "Contribución al SPE/Q"
)

plt.ylabel(
    "Variable de proceso"
)

plt.title(
    "Principales contribuciones a la alarma – "
    "Falla 1, muestra 165"
)

plt.tight_layout()

ruta_contrib_f1 = (
    RESULTS_DIR /
    "Figura_contribuciones_PCA_falla_1.png"
)

plt.savefig(
    ruta_contrib_f1,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Figura guardada en:",
    ruta_contrib_f1
)

In [ ]:
top_contrib_f1.to_csv(
    RESULTS_DIR /
    "tabla_contribuciones_PCA_falla_1.csv",
    index=False
)

In [ ]:
figuras_guardadas = sorted(
    RESULTS_DIR.glob(
        "Figura_*.png"
    )
)

print(
    f"Total de figuras: "
    f"{len(figuras_guardadas)}\n"
)

for figura in figuras_guardadas:

    print(
        figura.name
    )

In [ ]:
print("PCA")
print("Componentes:", pca_model.n_components_)
print(
    "Varianza explicada:",
    pca_model
    .explained_variance_ratio_
    .sum()
)
print("T²:", T2_threshold)
print("SPE:", SPE_threshold)


print("\nISOLATION FOREST")

parametros_if = (
    if_model.get_params()
)

for parametro in [
    "n_estimators",
    "max_samples",
    "contamination",
    "max_features",
    "bootstrap",
    "random_state"
]:
    print(
        parametro,
        "→",
        parametros_if[
            parametro
        ]
    )


print("\nAUTOENCODER")

print(
    "Umbral:",
    config_ae["AE_threshold"]
)

print(
    "FAR objetivo:",
    config_ae["far_objetivo"]
)

print(
    "Ventana:",
    config_ae["ventana"]
)

print(
    "Mínimo alarmas:",
    config_ae["minimo_alarmas"]
)

print(
    "Learning rate:",
    float(
        autoencoder_guardado
        .optimizer
        .learning_rate
        .numpy()
    )
)

print(
    "Loss:",
    autoencoder_guardado.loss
)

In [ ]:
print("Claves del checkpoint IF:")
print(checkpoint_if.keys())

for clave, valor in checkpoint_if.items():
    if "threshold" in clave.lower() or "umbral" in clave.lower():
        print(clave, "→", valor)

In [ ]:
IF_threshold_final = next(
    valor
    for clave, valor in checkpoint_if.items()
    if "threshold" in clave.lower()
    or "umbral" in clave.lower()
)

print("Umbral IF:", IF_threshold_final)

In [ ]:
tabla_hiperparametros_modelos = pd.DataFrame([

    # ========================================================
    # PCA
    # ========================================================

    {
        "Modelo": "PCA",
        "Parámetro": "Variables de entrada",
        "Configuración": "52"
    },

    {
        "Modelo": "PCA",
        "Parámetro": "Número de componentes",
        "Configuración": "36"
    },

    {
        "Modelo": "PCA",
        "Parámetro": "Varianza explicada acumulada",
        "Configuración": "95,84 %"
    },

    {
        "Modelo": "PCA",
        "Parámetro": "Umbral Hotelling T²",
        "Configuración": f"{T2_threshold:.3f}"
    },

    {
        "Modelo": "PCA",
        "Parámetro": "Umbral SPE/Q",
        "Configuración": f"{SPE_threshold:.3f}"
    },

    {
        "Modelo": "PCA",
        "Parámetro": "Criterio de alarma puntual",
        "Configuración": "T² > límite OR SPE > límite"
    },


    # ========================================================
    # ISOLATION FOREST
    # ========================================================

    {
        "Modelo": "Isolation Forest",
        "Parámetro": "Número de árboles",
        "Configuración": str(
            parametros_if["n_estimators"]
        )
    },

    {
        "Modelo": "Isolation Forest",
        "Parámetro": "max_samples",
        "Configuración": str(
            parametros_if["max_samples"]
        )
    },

    {
        "Modelo": "Isolation Forest",
        "Parámetro": "contamination",
        "Configuración": str(
            parametros_if["contamination"]
        )
    },

    {
        "Modelo": "Isolation Forest",
        "Parámetro": "max_features",
        "Configuración": str(
            parametros_if["max_features"]
        )
    },

    {
        "Modelo": "Isolation Forest",
        "Parámetro": "bootstrap",
        "Configuración": str(
            parametros_if["bootstrap"]
        )
    },

    {
        "Modelo": "Isolation Forest",
        "Parámetro": "random_state",
        "Configuración": str(
            parametros_if["random_state"]
        )
    },

    {
        "Modelo": "Isolation Forest",
        "Parámetro": "Umbral de anomalía",
        "Configuración": f"{IF_threshold_final:.6f}"
    },


    # ========================================================
    # AUTOENCODER
    # ========================================================

    {
        "Modelo": "Autoencoder",
        "Parámetro": "Arquitectura",
        "Configuración": "52–32–16–8–16–32–52"
    },

    {
        "Modelo": "Autoencoder",
        "Parámetro": "Dimensión latente",
        "Configuración": "8"
    },

    {
        "Modelo": "Autoencoder",
        "Parámetro": "Activaciones internas",
        "Configuración": "ReLU"
    },

    {
        "Modelo": "Autoencoder",
        "Parámetro": "Activación de salida",
        "Configuración": "Lineal"
    },

    {
        "Modelo": "Autoencoder",
        "Parámetro": "Optimizador",
        "Configuración": "Adam"
    },

    {
        "Modelo": "Autoencoder",
        "Parámetro": "Learning rate",
        "Configuración": "0,001"
    },

    {
        "Modelo": "Autoencoder",
        "Parámetro": "Función de pérdida",
        "Configuración": "MSE"
    },

    {
        "Modelo": "Autoencoder",
        "Parámetro": "Batch size",
        "Configuración": "512"
    },

    {
        "Modelo": "Autoencoder",
        "Parámetro": "Épocas máximas",
        "Configuración": "50"
    },

    {
        "Modelo": "Autoencoder",
        "Parámetro": "Mejor época observada",
        "Configuración": "25"
    },

    {
        "Modelo": "Autoencoder",
        "Parámetro": "Épocas ejecutadas",
        "Configuración": "30"
    },

    {
        "Modelo": "Autoencoder",
        "Parámetro": "Umbral de reconstrucción",
        "Configuración": f"{config_ae['AE_threshold']:.6f}"
    }
])

display(
    tabla_hiperparametros_modelos
)

In [ ]:
tabla_hiperparametros_modelos.to_csv(
    RESULTS_DIR /
    "tabla_hiperparametros_modelos_memoria.csv",
    index=False
)

In [ ]:
tabla_configuracion_comun = pd.DataFrame([

    {
        "Elemento": "Variables de proceso",
        "Configuración": "52"
    },

    {
        "Elemento": "Escalado",
        "Configuración": "StandardScaler"
    },

    {
        "Elemento": "Corridas normales de entrenamiento",
        "Configuración": "400"
    },

    {
        "Elemento": "Corridas normales de validación",
        "Configuración": "100"
    },

    {
        "Elemento": "Separación train/validation",
        "Configuración": "Por simulationRun, sin corridas compartidas"
    },

    {
        "Elemento": "Muestreo temporal",
        "Configuración": "3 min por muestra"
    },

    {
        "Elemento": "Persistencia temporal",
        "Configuración": "3 alarmas en las últimas 5 muestras"
    },

    {
        "Elemento": "FAR puntual de referencia",
        "Configuración": "≈ 1,99 %"
    }
])

display(
    tabla_configuracion_comun
)

In [ ]:
tabla_configuracion_comun.to_csv(
    RESULTS_DIR /
    "tabla_configuracion_comun_memoria.csv",
    index=False
)

### Nota de trazabilidad sobre la comparación con Yin et al. (2012)

El bloque siguiente se conserva únicamente como registro histórico del desarrollo. **No debe utilizarse como comparación cuantitativa directa en la versión final de la memoria**, porque la métrica de este TFM se calcula a nivel de corrida evaluable con persistencia temporal, mientras que Yin et al. reportan tasas a nivel de muestra bajo otro protocolo experimental. La memoria final sustituye esta comparación numérica por una discusión cualitativa de patrones de detectabilidad.


In [ ]:
comparacion_yin = pd.DataFrame({
    "Falla": [1, 2, 3, 4, 5, 6, 7, 8],

    "PCA_este_TFM_pct": [
        100.000,
        100.000,
        50.106,
        100.000,
        100.000,
        100.000,
        100.000,
        100.000
    ],

    "PCA_Yin_2012_pct": [
        99.88,
        98.75,
        33.63,
        100.00,
        100.00,
        98.125,
        99.13,
        95.38
    ]
})

display(
    comparacion_yin.round(2)
)

In [ ]:
comparacion_yin.to_csv(
    RESULTS_DIR /
    "tabla_comparacion_Yin_2012.csv",
    index=False
)